In [2]:
# ============================================================
# CELDA 1 — CONFIGURACIÓN v1.2.0-MULTILINGUAL
# ============================================================

from pathlib import Path
import sys
import json
import platform
import importlib.util

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# LOCALIZAR ROOT
# ------------------------------------------------------------

CURRENT = Path.cwd().resolve()

ROOT = None

for candidato in [
    CURRENT,
    CURRENT.parent,
    CURRENT.parent.parent,
    CURRENT.parent.parent.parent,
]:
    if (
        (candidato / "data").exists()
        and (candidato / "techmind").exists()
    ):
        ROOT = candidato
        break


if ROOT is None:
    raise FileNotFoundError(
        "No se pudo localizar la raíz de TechMind."
    )


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


print("=" * 70)
print("TECHMIND v1.2.0-MULTILINGUAL — EXPERIMENTAL")
print("=" * 70)

print("\nROOT:")
print(ROOT)

print("\nPython:")
print(sys.executable)

print("\nVersión Python:")
print(platform.python_version())

TECHMIND v1.2.0-MULTILINGUAL — EXPERIMENTAL

ROOT:
C:\Users\MAMÁ\Downloads\techmind-v2

Python:
C:\Users\MAMÁ\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe

Versión Python:
3.11.9


In [3]:
# ============================================================
# CELDA 2 — VERIFICAR DEPENDENCIAS
# ============================================================

dependencias = [
    "torch",
    "transformers",
    "sentence_transformers",
    "sklearn",
    "numpy",
    "pandas",
    "joblib"
]

print("=" * 70)
print("DEPENDENCIAS")
print("=" * 70)

for modulo in dependencias:

    encontrado = (
        importlib.util.find_spec(modulo)
        is not None
    )

    estado = "✅" if encontrado else "❌"

    print(
        f"{estado} {modulo:<25} "
        f"{'instalado' if encontrado else 'NO instalado'}"
    )

DEPENDENCIAS
✅ torch                     instalado
✅ transformers              instalado
✅ sentence_transformers     instalado
✅ sklearn                   instalado
✅ numpy                     instalado
✅ pandas                    instalado
✅ joblib                    instalado


In [4]:
# ============================================================
# CELDA 3 — CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================

EXPERIMENT_VERSION = "1.2.0-multilingual"

EMBEDDING_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

RANDOM_STATE = 42
N_SPLITS = 5

TEXT_COLUMN = "texto_combinado_ponderado"
TARGET_COLUMN = "categoria"

MODEL_DATA_PATH = (
    ROOT
    / "data"
    / "processed"
    / "techmind_modelado.csv"
)

EXPERIMENT_ROOT = (
    ROOT
    / "experiments"
    / EXPERIMENT_VERSION
)

REPORT_DIR = (
    EXPERIMENT_ROOT
    / "reports"
)

ARTIFACT_DIR = (
    EXPERIMENT_ROOT
    / "artifacts"
)

CACHE_DIR = (
    EXPERIMENT_ROOT
    / "cache"
)

for path in [
    REPORT_DIR,
    ARTIFACT_DIR,
    CACHE_DIR
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )


print("Modelo embeddings:")
print(EMBEDDING_MODEL)

print("\nDataset:")
print(MODEL_DATA_PATH)

print("\nExperimento:")
print(EXPERIMENT_ROOT)

Modelo embeddings:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Dataset:
C:\Users\MAMÁ\Downloads\techmind-v2\data\processed\techmind_modelado.csv

Experimento:
C:\Users\MAMÁ\Downloads\techmind-v2\experiments\1.2.0-multilingual


In [5]:
# ============================================================
# CELDA 4 — CARGA DEL DATASET
# ============================================================

df = pd.read_csv(
    MODEL_DATA_PATH
)

print("=" * 70)
print("DATASET TECHMIND")
print("=" * 70)

print("\nRegistros:")
print(len(df))

print("\nColumnas:")
print(df.columns.tolist())

print("\nClases:")
print(
    df[TARGET_COLUMN]
    .value_counts()
)

print("\nTextos vacíos:")
print(
    df[TEXT_COLUMN]
    .isna()
    .sum()
)

DATASET TECHMIND

Registros:
4583

Columnas:
['id_documento', 'categoria', 'texto_modelo_base', 'texto_modelo_titulo', 'texto_combinado', 'texto_combinado_ponderado', 'titulo_num_caracteres', 'titulo_num_palabras', 'texto_num_caracteres', 'texto_num_palabras', 'combinado_num_caracteres', 'combinado_num_palabras', 'num_palabras_clave', 'longitud_promedio_palabra', 'proporcion_digitos', 'cantidad_versiones', 'cantidad_terminos_tecnicos', 'densidad_terminos_tecnicos', 'cantidad_urls', 'cantidad_simbolos_codigo', 'tiene_palabras_clave', 'tiene_version', 'tiene_url', 'tiene_codigo', 'tiene_terminos_tecnicos']

Clases:
categoria
frontend       1165
backend        1159
cloud          1157
datascience    1102
Name: count, dtype: int64

Textos vacíos:
0


In [6]:
# ============================================================
# CELDA 5 — SPLIT REPRODUCIBLE
# ============================================================

from sklearn.model_selection import train_test_split

X = (
    df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
)

y = (
    df[TARGET_COLUMN]
    .astype(str)
)


X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


print("=" * 70)
print("SPLIT v1.2.0")
print("=" * 70)

print("\nTrain:")
print(len(X_train))

print("\nTest:")
print(len(X_test))

print("\nDistribución TRAIN:")
print(
    y_train.value_counts()
)

print("\nDistribución TEST:")
print(
    y_test.value_counts()
)

SPLIT v1.2.0

Train:
3666

Test:
917

Distribución TRAIN:
categoria
frontend       932
backend        927
cloud          925
datascience    882
Name: count, dtype: int64

Distribución TEST:
categoria
frontend       233
backend        232
cloud          232
datascience    220
Name: count, dtype: int64


In [7]:
# ============================================================
# CELDA 6 — MULTILINGUAL SENTENCE TRANSFORMER
# ============================================================

from sentence_transformers import SentenceTransformer

print("=" * 70)
print("CARGANDO ENCODER MULTILINGÜE")
print("=" * 70)

encoder = SentenceTransformer(
    EMBEDDING_MODEL
)

print("\nModelo:")
print(EMBEDDING_MODEL)

print("\nDimensión:")
print(
    encoder.get_embedding_dimension()
)

print("\nMáxima longitud:")
print(
    encoder.max_seq_length
)

C:\Users\MAMÁ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CARGANDO ENCODER MULTILINGÜE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1910.55it/s]



Modelo:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Dimensión:
384

Máxima longitud:
128


In [8]:
# ============================================================
# CELDA 7 — SMOKE TEST MULTILINGÜE DEL EMBEDDING
# ============================================================

textos_prueba = [
    "Entrenar un modelo de clasificación de imágenes.",
    "Train an image classification model.",
    "Обучить модель классификации изображений."
]

emb_prueba = encoder.encode(
    textos_prueba,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Shape:")
print(emb_prueba.shape)

print("\nEmbedding sample:")
print(
    emb_prueba[0][:10]
)

Shape:
(3, 384)

Embedding sample:
[-0.00182207  0.01156533 -0.00732725  0.00017226  0.00067924  0.05259613
  0.01542406 -0.03964028 -0.03600878  0.02622338]


In [9]:
# ============================================================
# CELDA 8 — SIMILARIDAD CROSS-LANGUAGE
# ============================================================

similaridades = (
    emb_prueba
    @ emb_prueba.T
)

df_sim = pd.DataFrame(
    similaridades,
    index=[
        "ES",
        "EN",
        "RU"
    ],
    columns=[
        "ES",
        "EN",
        "RU"
    ]
)

print("=" * 70)
print("SIMILARIDAD SEMÁNTICA ENTRE IDIOMAS")
print("=" * 70)

display(
    df_sim.round(4)
)

SIMILARIDAD SEMÁNTICA ENTRE IDIOMAS


,ES,EN,RU
ES,1.0000,0.9275,0.9687
EN,0.9275,1.0000,0.8995
RU,0.9687,0.8995,1.0000


In [10]:
# ============================================================
# CELDA 9 — AUDITORÍA DE LONGITUD DEL CORPUS
# ============================================================

print("=" * 70)
print("AUDITORÍA DE LONGITUD — MULTILINGUAL MINILM")
print("=" * 70)

MAX_SEQ_LENGTH = encoder.max_seq_length

textos_completos = (
    df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)

# Tokenizar SIN truncar para conocer la longitud real
tokens = encoder.tokenizer(
    textos_completos,
    padding=False,
    truncation=False,
    add_special_tokens=True
)

longitudes_tokens = np.array([
    len(ids)
    for ids in tokens["input_ids"]
])

total = len(longitudes_tokens)

superan_limite = int(
    (longitudes_tokens > MAX_SEQ_LENGTH).sum()
)

pct_superan = (
    superan_limite / total * 100
)

print(f"\nDocumentos:            {total}")
print(f"Límite encoder:        {MAX_SEQ_LENGTH} tokens")

print(f"\nLongitud mínima:       {longitudes_tokens.min()}")
print(f"Longitud media:        {longitudes_tokens.mean():.2f}")
print(f"Mediana:               {np.median(longitudes_tokens):.2f}")
print(f"Percentil 90:          {np.percentile(longitudes_tokens, 90):.2f}")
print(f"Percentil 95:          {np.percentile(longitudes_tokens, 95):.2f}")
print(f"Percentil 99:          {np.percentile(longitudes_tokens, 99):.2f}")
print(f"Máxima:                {longitudes_tokens.max()}")

print(f"\nSuperan 128 tokens:    {superan_limite}")
print(f"Porcentaje truncado:   {pct_superan:.2f}%")

AUDITORÍA DE LONGITUD — MULTILINGUAL MINILM


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (129 > 128). Running this sequence through the model will result in indexing errors



Documentos:            4583
Límite encoder:        128 tokens

Longitud mínima:       15
Longitud media:        60.86
Mediana:               60.00
Percentil 90:          81.00
Percentil 95:          89.00
Percentil 99:          109.00
Máxima:                139

Superan 128 tokens:    9
Porcentaje truncado:   0.20%


In [11]:
# ============================================================
# DISTRIBUCIÓN DE LONGITUDES
# ============================================================

rangos = pd.cut(
    longitudes_tokens,
    bins=[
        0,
        32,
        64,
        96,
        128,
        256,
        512,
        np.inf
    ],
    labels=[
        "1-32",
        "33-64",
        "65-96",
        "97-128",
        "129-256",
        "257-512",
        "512+"
    ]
)

distribucion_longitud = (
    pd.Series(rangos)
    .value_counts()
    .sort_index()
    .rename_axis("rango_tokens")
    .reset_index(name="documentos")
)

distribucion_longitud["porcentaje"] = (
    distribucion_longitud["documentos"]
    / total
    * 100
)

display(
    distribucion_longitud.round(2)
)

,rango_tokens,documentos,porcentaje
0,1-32,90,1.96
1,33-64,2684,58.56
2,65-96,1685,36.77
3,97-128,115,2.51
4,129-256,9,0.20
5,257-512,0,0.00
6,512+,0,0.00


In [12]:
# ============================================================
# CELDA 10 — EMBEDDINGS TRAIN
# ============================================================

import time

print("=" * 70)
print("GENERANDO EMBEDDINGS — TRAIN")
print("=" * 70)

inicio = time.perf_counter()

X_train_embeddings = encoder.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

duracion_train = (
    time.perf_counter()
    - inicio
)

print("\n✅ TRAIN completado")

print("\nShape:")
print(X_train_embeddings.shape)

print(
    f"\nDuración: "
    f"{duracion_train:.2f} segundos"
)

print(
    "\nTipo:"
)
print(
    X_train_embeddings.dtype
)

GENERANDO EMBEDDINGS — TRAIN


Batches: 100%|██████████| 115/115 [02:19<00:00,  1.21s/it]


✅ TRAIN completado

Shape:
(3666, 384)

Duración: 139.50 segundos

Tipo:
float32


In [13]:
# ============================================================
# CELDA 11 — EMBEDDINGS TEST
# ============================================================

print("=" * 70)
print("GENERANDO EMBEDDINGS — TEST")
print("=" * 70)

inicio = time.perf_counter()

X_test_embeddings = encoder.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

duracion_test = (
    time.perf_counter()
    - inicio
)

print("\n✅ TEST completado")

print("\nShape:")
print(X_test_embeddings.shape)

print(
    f"\nDuración: "
    f"{duracion_test:.2f} segundos"
)

print("\nTipo:")
print(X_test_embeddings.dtype)

GENERANDO EMBEDDINGS — TEST


Batches: 100%|██████████| 29/29 [00:32<00:00,  1.11s/it]


✅ TEST completado

Shape:
(917, 384)

Duración: 32.32 segundos

Tipo:
float32


In [14]:
# ============================================================
# CELDA 12 — VALIDACIÓN DE EMBEDDINGS
# ============================================================

print("=" * 70)
print("VALIDACIÓN DE EMBEDDINGS")
print("=" * 70)

EXPECTED_DIM = 384

validaciones_embeddings = {

    "train_shape":
        X_train_embeddings.shape
        == (
            len(X_train),
            EXPECTED_DIM
        ),

    "test_shape":
        X_test_embeddings.shape
        == (
            len(X_test),
            EXPECTED_DIM
        ),

    "train_sin_nan":
        not np.isnan(
            X_train_embeddings
        ).any(),

    "test_sin_nan":
        not np.isnan(
            X_test_embeddings
        ).any(),

    "train_sin_inf":
        not np.isinf(
            X_train_embeddings
        ).any(),

    "test_sin_inf":
        not np.isinf(
            X_test_embeddings
        ).any(),

    "train_float":
        np.issubdtype(
            X_train_embeddings.dtype,
            np.floating
        ),

    "test_float":
        np.issubdtype(
            X_test_embeddings.dtype,
            np.floating
        ),
}


for nombre, resultado in (
    validaciones_embeddings.items()
):

    estado = (
        "✅"
        if resultado
        else "❌"
    )

    print(
        f"{estado} "
        f"{nombre}: "
        f"{resultado}"
    )


EMBEDDINGS_VALIDOS = all(
    validaciones_embeddings.values()
)

print("\n" + "-" * 70)

print(
    "EMBEDDINGS LISTOS:",
    EMBEDDINGS_VALIDOS
)

if not EMBEDDINGS_VALIDOS:
    raise RuntimeError(
        "Los embeddings no superaron "
        "las validaciones."
    )

VALIDACIÓN DE EMBEDDINGS
✅ train_shape: True
✅ test_shape: True
✅ train_sin_nan: True
✅ test_sin_nan: True
✅ train_sin_inf: True
✅ test_sin_inf: True
✅ train_float: True
✅ test_float: True

----------------------------------------------------------------------
EMBEDDINGS LISTOS: True


In [15]:
# ============================================================
# CELDA 13 — VALIDAR NORMALIZACIÓN L2
# ============================================================

normas_train = np.linalg.norm(
    X_train_embeddings,
    axis=1
)

normas_test = np.linalg.norm(
    X_test_embeddings,
    axis=1
)

print("=" * 70)
print("NORMALIZACIÓN DE EMBEDDINGS")
print("=" * 70)

print("\nTRAIN")
print(
    f"Media:  {normas_train.mean():.6f}"
)
print(
    f"Mínima: {normas_train.min():.6f}"
)
print(
    f"Máxima: {normas_train.max():.6f}"
)

print("\nTEST")
print(
    f"Media:  {normas_test.mean():.6f}"
)
print(
    f"Mínima: {normas_test.min():.6f}"
)
print(
    f"Máxima: {normas_test.max():.6f}"
)

NORMALIZACION_OK = (
    np.allclose(
        normas_train,
        1.0,
        atol=1e-4
    )
    and
    np.allclose(
        normas_test,
        1.0,
        atol=1e-4
    )
)

print(
    "\nNormalización correcta:",
    NORMALIZACION_OK
)

NORMALIZACIÓN DE EMBEDDINGS

TRAIN
Media:  1.000000
Mínima: 1.000000
Máxima: 1.000000

TEST
Media:  1.000000
Mínima: 1.000000
Máxima: 1.000000

Normalización correcta: True


In [16]:
# ============================================================
# CELDA 14 — CACHE DE EMBEDDINGS
# ============================================================

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    CACHE_DIR
    / "X_train_embeddings.npy",
    X_train_embeddings
)

np.save(
    CACHE_DIR
    / "X_test_embeddings.npy",
    X_test_embeddings
)

np.save(
    CACHE_DIR
    / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    CACHE_DIR
    / "y_test.npy",
    y_test.to_numpy()
)


print("=" * 70)
print("CACHE DE EMBEDDINGS")
print("=" * 70)

archivos_cache = [
    "X_train_embeddings.npy",
    "X_test_embeddings.npy",
    "y_train.npy",
    "y_test.npy"
]

for archivo in archivos_cache:

    path = (
        CACHE_DIR
        / archivo
    )

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"✅ {archivo:<30} "
        f"{size_mb:.2f} MB"
    )

CACHE DE EMBEDDINGS
✅ X_train_embeddings.npy         5.37 MB
✅ X_test_embeddings.npy          1.34 MB
✅ y_train.npy                    0.01 MB
✅ y_test.npy                     0.00 MB


In [17]:
# ============================================================
# CELDA 15 — DOCUMENTOS POTENCIALMENTE TRUNCADOS
# ============================================================

indices_truncados = np.where(
    longitudes_tokens > MAX_SEQ_LENGTH
)[0]

df_truncados = df.iloc[
    indices_truncados
].copy()

df_truncados[
    "tokens_encoder"
] = (
    longitudes_tokens[
        indices_truncados
    ]
)

print("=" * 70)
print("DOCUMENTOS > 128 TOKENS")
print("=" * 70)

print(
    "\nTotal:",
    len(df_truncados)
)

display(
    df_truncados[
        [
            TARGET_COLUMN,
            TEXT_COLUMN,
            "tokens_encoder"
        ]
    ].sort_values(
        "tokens_encoder",
        ascending=False
    )
)

DOCUMENTOS > 128 TOKENS

Total: 9


,categoria,texto_combinado_ponderado,tokens_encoder
2667,cloud,unlocking claude fable 5 and mythos 5: a new e...,139
3924,backend,building toykv: a from-scratch persistent kv i...,139
4126,backend,the native-first revolution: how node.js 24 is...,136
451,cloud,the ai coding price war is a bloodbath: gpt-5....,134
1967,frontend,one cut my blazor wasm lcp by 500ms the invisi...,134
4328,backend,ai for api design testing in 2026: speakeasy v...,134
1790,datascience,samkhya v1.0: plug claude gpt-4o-mini or local...,130
3450,frontend,this post shares a real accessibility case stu...,130
126,cloud,auto-fix is not the problem. the signal is the...,129


In [18]:
# ============================================================
# CELDA 16 — CONFIGURACIÓN DE SELECCIÓN
# ============================================================

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.linear_model import (
    LogisticRegression,
    SGDClassifier
)

from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

from sklearn.base import clone

import time


RANDOM_STATE = 42
N_SPLITS = 5

# En Windows prefiero mantener un solo proceso
# para evitar problemas de multiprocessing.
N_JOBS = 1


cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


print("=" * 70)
print("SELECCIÓN DE CLASIFICADORES — v1.2.0-MULTILINGUAL")
print("=" * 70)

print("\nFolds:", N_SPLITS)
print("Métrica principal: F1 Macro")
print("Test reservado: NO utilizado")

SELECCIÓN DE CLASIFICADORES — v1.2.0-MULTILINGUAL

Folds: 5
Métrica principal: F1 Macro
Test reservado: NO utilizado


In [19]:
# ============================================================
# CELDA 17 — MODELOS CANDIDATOS
# ============================================================

modelos = {

    "LogisticRegression": LogisticRegression(
        C=1.0,
        max_iter=3000,
        class_weight=None,
        random_state=RANDOM_STATE
    ),

    "LinearSVC": LinearSVC(
        C=1.0,
        random_state=RANDOM_STATE
    ),

    "SGDClassifier": SGDClassifier(
        loss="hinge",
        alpha=1e-4,
        max_iter=3000,
        tol=1e-4,
        random_state=RANDOM_STATE
    )
}


print("=" * 70)
print("CANDIDATOS")
print("=" * 70)

for nombre, modelo in modelos.items():
    print(f"✅ {nombre}")
    print(f"   {modelo}")

CANDIDATOS
✅ LogisticRegression
   LogisticRegression(max_iter=3000, random_state=42)
✅ LinearSVC
   LinearSVC(random_state=42)
✅ SGDClassifier
   SGDClassifier(max_iter=3000, random_state=42, tol=0.0001)


In [20]:
# ============================================================
# CELDA 18 — CROSS-VALIDATION
# ============================================================

resultados_cv = []

for nombre, modelo in modelos.items():

    print("\n" + "=" * 70)
    print(nombre)
    print("=" * 70)

    inicio = time.perf_counter()

    scores = cross_validate(
        modelo,
        X_train_embeddings,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro"
        },
        return_train_score=True,
        n_jobs=N_JOBS
    )

    duracion = (
        time.perf_counter()
        - inicio
    )

    accuracy_cv = (
        scores["test_accuracy"].mean()
    )

    accuracy_std = (
        scores["test_accuracy"].std()
    )

    f1_cv = (
        scores["test_f1_macro"].mean()
    )

    f1_std = (
        scores["test_f1_macro"].std()
    )

    f1_train = (
        scores["train_f1_macro"].mean()
    )

    brecha = (
        f1_train - f1_cv
    )

    resultados_cv.append({
        "modelo": nombre,

        "accuracy_cv":
            accuracy_cv,

        "accuracy_std":
            accuracy_std,

        "f1_macro_cv":
            f1_cv,

        "f1_macro_std":
            f1_std,

        "f1_macro_train":
            f1_train,

        "brecha_train_cv":
            brecha,

        "duracion_segundos":
            duracion
    })

    print(
        f"Accuracy CV: "
        f"{accuracy_cv:.4f} "
        f"± {accuracy_std:.4f}"
    )

    print(
        f"F1 Macro CV: "
        f"{f1_cv:.4f} "
        f"± {f1_std:.4f}"
    )

    print(
        f"F1 Train:    "
        f"{f1_train:.4f}"
    )

    print(
        f"Brecha:      "
        f"{brecha:.4f}"
    )

    print(
        f"Duración:    "
        f"{duracion:.2f} s"
    )


LogisticRegression
Accuracy CV: 0.7586 ± 0.0133
F1 Macro CV: 0.7604 ± 0.0131
F1 Train:    0.8087
Brecha:      0.0483
Duración:    2.45 s

LinearSVC
Accuracy CV: 0.7717 ± 0.0106
F1 Macro CV: 0.7729 ± 0.0103
F1 Train:    0.8476
Brecha:      0.0747
Duración:    6.81 s

SGDClassifier
Accuracy CV: 0.7709 ± 0.0142
F1 Macro CV: 0.7724 ± 0.0141
F1 Train:    0.8458
Brecha:      0.0734
Duración:    3.06 s


In [21]:
# ============================================================
# CELDA 19 — COMPARACIÓN DE MODELOS
# ============================================================

df_comparacion_modelos = (
    pd.DataFrame(
        resultados_cv
    )
    .sort_values(
        "f1_macro_cv",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("COMPARACIÓN — EMBEDDINGS MULTILINGÜES")
print("=" * 70)

display(
    df_comparacion_modelos.round(4)
)

COMPARACIÓN — EMBEDDINGS MULTILINGÜES


,modelo,accuracy_cv,accuracy_std,f1_macro_cv,f1_macro_std,f1_macro_train,brecha_train_cv,duracion_segundos
0,LinearSVC,0.7717,0.0106,0.7729,0.0103,0.8476,0.0747,6.8124
1,SGDClassifier,0.7709,0.0142,0.7724,0.0141,0.8458,0.0734,3.0562
2,LogisticRegression,0.7586,0.0133,0.7604,0.0131,0.8087,0.0483,2.4545


In [22]:
# ============================================================
# CELDA 20 — COMPARACIÓN CONTRA BASELINE OFICIAL v1.1.0
# ============================================================

# Métricas oficiales de v1.1.0 documentadas en Notebook 05 / CHANGELOG.
# Se centralizan aquí para evitar confundirlas con las métricas de v1.0.
V11_F1_MACRO_CV = 0.8493
V11_ACCURACY_TEST = 0.8430
V11_PRECISION_MACRO_TEST = 0.8455
V11_RECALL_MACRO_TEST = 0.8434
V11_F1_MACRO_TEST = 0.8441
V11_F1_WEIGHTED_TEST = 0.8435

df_comparacion_modelos[
    "diferencia_vs_v1_1"
] = (
    df_comparacion_modelos[
        "f1_macro_cv"
    ]
    - V11_F1_MACRO_CV
)


print("=" * 70)
print("COMPARACIÓN CONTRA TECHMIND v1.1.0")
print("=" * 70)

display(
    df_comparacion_modelos[
        [
            "modelo",
            "f1_macro_cv",
            "f1_macro_std",
            "f1_macro_train",
            "brecha_train_cv",
            "diferencia_vs_v1_1",
            "duracion_segundos"
        ]
    ].round(4)
)


COMPARACIÓN CONTRA TECHMIND v1.1.0


,modelo,f1_macro_cv,f1_macro_std,f1_macro_train,brecha_train_cv,diferencia_vs_v1_1,duracion_segundos
0,LinearSVC,0.7729,0.0103,0.8476,0.0747,-0.0764,6.8124
1,SGDClassifier,0.7724,0.0141,0.8458,0.0734,-0.0769,3.0562
2,LogisticRegression,0.7604,0.0131,0.8087,0.0483,-0.0889,2.4545


In [23]:
# ============================================================
# CELDA 21 — SELECCIÓN DEL MEJOR MODELO
# ============================================================

mejor_fila = (
    df_comparacion_modelos
    .iloc[0]
)

MEJOR_MODELO_NOMBRE = (
    mejor_fila["modelo"]
)

MEJOR_F1_CV = float(
    mejor_fila["f1_macro_cv"]
)

MEJOR_STD = float(
    mejor_fila["f1_macro_std"]
)

MEJOR_BRECHA = float(
    mejor_fila["brecha_train_cv"]
)


print("=" * 70)
print("CANDIDATO SELECCIONADO")
print("=" * 70)

print(
    "\nModelo:",
    MEJOR_MODELO_NOMBRE
)

print(
    f"F1 Macro CV: "
    f"{MEJOR_F1_CV:.4f}"
)

print(
    f"Std:         "
    f"{MEJOR_STD:.4f}"
)

print(
    f"Brecha:      "
    f"{MEJOR_BRECHA:.4f}"
)

print(
    f"\nDiferencia vs v1.1.0: "
    f"{MEJOR_F1_CV - V11_F1_MACRO_CV:+.4f}"
)

CANDIDATO SELECCIONADO

Modelo: LinearSVC
F1 Macro CV: 0.7729
Std:         0.0103
Brecha:      0.0747

Diferencia vs v1.1.0: -0.0703


In [24]:
# ============================================================
# CELDA 22 — OPTIMIZACIÓN LinearSVC
# ============================================================

from sklearn.svm import LinearSVC

valores_C = [
    0.03,
    0.1,
    0.3,
    0.5,
    1.0,
    2.0,
    3.0,
    5.0,
    10.0
]

resultados_svc = []

print("=" * 70)
print("OPTIMIZACIÓN LinearSVC — EMBEDDINGS MULTILINGÜES")
print("=" * 70)

for C in valores_C:

    modelo = LinearSVC(
        C=C,
        random_state=RANDOM_STATE
    )

    scores = cross_validate(
        modelo,
        X_train_embeddings,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro"
        },
        return_train_score=True,
        n_jobs=1
    )

    f1_cv = scores[
        "test_f1_macro"
    ].mean()

    f1_std = scores[
        "test_f1_macro"
    ].std()

    f1_train = scores[
        "train_f1_macro"
    ].mean()

    brecha = (
        f1_train -
        f1_cv
    )

    resultados_svc.append({
        "C": C,
        "f1_macro_cv": f1_cv,
        "f1_macro_std": f1_std,
        "f1_macro_train": f1_train,
        "brecha_train_cv": brecha
    })

    print(
        f"C={C:<5} "
        f"F1={f1_cv:.4f} "
        f"std={f1_std:.4f} "
        f"train={f1_train:.4f} "
        f"gap={brecha:.4f}"
    )

OPTIMIZACIÓN LinearSVC — EMBEDDINGS MULTILINGÜES
C=0.03  F1=0.7514 std=0.0089 train=0.7768 gap=0.0254
C=0.1   F1=0.7544 std=0.0125 train=0.8035 gap=0.0490
C=0.3   F1=0.7626 std=0.0140 train=0.8242 gap=0.0616
C=0.5   F1=0.7663 std=0.0127 train=0.8343 gap=0.0679
C=1.0   F1=0.7729 std=0.0103 train=0.8476 gap=0.0747
C=2.0   F1=0.7749 std=0.0078 train=0.8604 gap=0.0854
C=3.0   F1=0.7755 std=0.0078 train=0.8660 gap=0.0905
C=5.0   F1=0.7759 std=0.0068 train=0.8738 gap=0.0979
C=10.0  F1=0.7764 std=0.0072 train=0.8824 gap=0.1060


In [25]:
# ============================================================
# CELDA 23 — RESULTADOS OPTIMIZACIÓN
# ============================================================

df_svc_tuning = (
    pd.DataFrame(resultados_svc)
    .sort_values(
        "f1_macro_cv",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    df_svc_tuning.round(4)
)

mejor_C = float(
    df_svc_tuning.iloc[0]["C"]
)

mejor_f1_embeddings = float(
    df_svc_tuning.iloc[0][
        "f1_macro_cv"
    ]
)

print("=" * 70)

print(
    "Mejor C:",
    mejor_C
)

print(
    f"Mejor F1 CV: "
    f"{mejor_f1_embeddings:.4f}"
)

print(
    f"Diferencia vs v1.1.0: "
    f"{mejor_f1_embeddings - V11_F1_MACRO_CV:+.4f}"
)

,C,f1_macro_cv,f1_macro_std,f1_macro_train,brecha_train_cv
0,10.00,0.7764,0.0072,0.8824,0.1060
1,5.00,0.7759,0.0068,0.8738,0.0979
2,3.00,0.7755,0.0078,0.8660,0.0905
3,2.00,0.7749,0.0078,0.8604,0.0854
4,1.00,0.7729,0.0103,0.8476,0.0747
5,0.50,0.7663,0.0127,0.8343,0.0679
6,0.30,0.7626,0.0140,0.8242,0.0616
7,0.10,0.7544,0.0125,0.8035,0.0490
8,0.03,0.7514,0.0089,0.7768,0.0254


Mejor C: 10.0
Mejor F1 CV: 0.7764
Diferencia vs v1.1.0: -0.0729


In [26]:
# ============================================================
# CELDA 24 — CARGAR PIPELINE v1.1.0 COMO PLANTILLA
# ============================================================

import joblib

candidatos_modelo = [
    ROOT
    / "models"
    / "v1.1.0"
    / "techmind_modelo_final_v1_1_0.joblib",

    ROOT
    / "artifacts"
    / "v1.1.0"
    / "techmind_modelo_final.joblib",

    ROOT
    / "artifacts"
    / "techmind_modelo_final.joblib",
]

MODEL_V11_PATH = None

for candidato in candidatos_modelo:

    if candidato.exists():
        MODEL_V11_PATH = candidato
        break


if MODEL_V11_PATH is None:

    raise FileNotFoundError(
        "No se encontró el artefacto "
        "TechMind v1.1.0."
    )


modelo_v11 = joblib.load(
    MODEL_V11_PATH
)

print("=" * 70)
print("PIPELINE v1.1.0")
print("=" * 70)

print("\nArtefacto:")
print(MODEL_V11_PATH)

print("\nSteps:")
print(
    modelo_v11.named_steps.keys()
)

PIPELINE v1.1.0

Artefacto:
C:\Users\MAMÁ\Downloads\techmind-v2\artifacts\v1.1.0\techmind_modelo_final.joblib

Steps:
dict_keys(['features', 'clasificador'])


In [27]:
# ============================================================
# CELDA 25 — PLANTILLA TF-IDF v1.1.0
# ============================================================

from sklearn.base import clone

features_template = clone(
    modelo_v11.named_steps[
        "features"
    ]
)

print("=" * 70)
print("FEATURE EXTRACTOR v1.1.0")
print("=" * 70)

print(features_template)

FEATURE EXTRACTOR v1.1.0
FeatureUnion(transformer_list=[('word',
                                TfidfVectorizer(lowercase=False, max_df=0.95,
                                                max_features=30000,
                                                ngram_range=(1, 2),
                                                stop_words=['a', 'al', 'algo',
                                                            'alguna', 'algunas',
                                                            'alguno', 'algunos',
                                                            'ante', 'antes',
                                                            'cada', 'como',
                                                            'con', 'cual',
                                                            'cuando', 'de',
                                                            'del', 'desde',
                                                            'donde', 'durante',
                    

In [28]:
# ============================================================
# CELDA 26 — CROSS-VALIDATION HÍBRIDA
# ============================================================

from scipy.sparse import (
    csr_matrix,
    hstack
)

from sklearn.metrics import (
    f1_score,
    accuracy_score
)

indices = np.arange(
    len(X_train)
)

resultados_hybrid = []

C_HYBRID = [
    0.1,
    0.3,
    1.0,
    3.0
]


for C in C_HYBRID:

    f1_folds = []
    acc_folds = []
    f1_train_folds = []

    print("\n" + "=" * 70)
    print(f"HYBRID LinearSVC — C={C}")
    print("=" * 70)

    for fold, (
        idx_train,
        idx_val
    ) in enumerate(
        cv.split(
            indices,
            y_train
        ),
        start=1
    ):

        # --------------------------------
        # Textos
        # --------------------------------

        textos_fold_train = (
            X_train.iloc[
                idx_train
            ]
            .tolist()
        )

        textos_fold_val = (
            X_train.iloc[
                idx_val
            ]
            .tolist()
        )

        # --------------------------------
        # Labels
        # --------------------------------

        y_fold_train = (
            y_train.iloc[
                idx_train
            ]
        )

        y_fold_val = (
            y_train.iloc[
                idx_val
            ]
        )

        # --------------------------------
        # TF-IDF nuevo en cada fold
        # --------------------------------

        features_fold = clone(
            features_template
        )

        X_tfidf_train = (
            features_fold
            .fit_transform(
                textos_fold_train
            )
        )

        X_tfidf_val = (
            features_fold
            .transform(
                textos_fold_val
            )
        )

        # --------------------------------
        # Embeddings correspondientes
        # --------------------------------

        X_emb_train = csr_matrix(
            X_train_embeddings[
                idx_train
            ]
        )

        X_emb_val = csr_matrix(
            X_train_embeddings[
                idx_val
            ]
        )

        # --------------------------------
        # FUSIÓN
        # --------------------------------

        X_hybrid_train = hstack(
            [
                X_tfidf_train,
                X_emb_train
            ],
            format="csr"
        )

        X_hybrid_val = hstack(
            [
                X_tfidf_val,
                X_emb_val
            ],
            format="csr"
        )

        # --------------------------------
        # Clasificador
        # --------------------------------

        clf = LinearSVC(
            C=C,
            random_state=RANDOM_STATE
        )

        clf.fit(
            X_hybrid_train,
            y_fold_train
        )

        pred_train = clf.predict(
            X_hybrid_train
        )

        pred_val = clf.predict(
            X_hybrid_val
        )

        f1_train_fold = f1_score(
            y_fold_train,
            pred_train,
            average="macro"
        )

        f1_val_fold = f1_score(
            y_fold_val,
            pred_val,
            average="macro"
        )

        acc_val_fold = accuracy_score(
            y_fold_val,
            pred_val
        )

        f1_train_folds.append(
            f1_train_fold
        )

        f1_folds.append(
            f1_val_fold
        )

        acc_folds.append(
            acc_val_fold
        )

        print(
            f"Fold {fold}: "
            f"F1={f1_val_fold:.4f}"
        )


    resultados_hybrid.append({

        "C":
            C,

        "f1_macro_cv":
            np.mean(
                f1_folds
            ),

        "f1_macro_std":
            np.std(
                f1_folds
            ),

        "accuracy_cv":
            np.mean(
                acc_folds
            ),

        "f1_train":
            np.mean(
                f1_train_folds
            ),

        "brecha":
            (
                np.mean(
                    f1_train_folds
                )
                -
                np.mean(
                    f1_folds
                )
            )
    })


HYBRID LinearSVC — C=0.1
Fold 1: F1=0.8498
Fold 2: F1=0.8672
Fold 3: F1=0.8314
Fold 4: F1=0.8562
Fold 5: F1=0.8257

HYBRID LinearSVC — C=0.3
Fold 1: F1=0.8674
Fold 2: F1=0.8697
Fold 3: F1=0.8448
Fold 4: F1=0.8708
Fold 5: F1=0.8341

HYBRID LinearSVC — C=1.0
Fold 1: F1=0.8646
Fold 2: F1=0.8627
Fold 3: F1=0.8501
Fold 4: F1=0.8735
Fold 5: F1=0.8336

HYBRID LinearSVC — C=3.0
Fold 1: F1=0.8618
Fold 2: F1=0.8640
Fold 3: F1=0.8461
Fold 4: F1=0.8706
Fold 5: F1=0.8349


In [29]:
# ============================================================
# CELDA 27 — COMPARACIÓN HÍBRIDA
# ============================================================

df_hybrid = (
    pd.DataFrame(
        resultados_hybrid
    )
    .sort_values(
        "f1_macro_cv",
        ascending=False
    )
    .reset_index(drop=True)
)

df_hybrid[
    "diferencia_vs_v11"
] = (
    df_hybrid[
        "f1_macro_cv"
    ]
    - V11_F1_MACRO_CV
)

print("=" * 70)
print("TECHMIND v1.2 HYBRID — CV")
print("=" * 70)

display(
    df_hybrid.round(4)
)

TECHMIND v1.2 HYBRID — CV


,C,f1_macro_cv,f1_macro_std,accuracy_cv,f1_train,brecha,diferencia_vs_v11
0,0.3,0.8574,0.0150,0.8565,0.9991,0.1417,0.0081
1,1.0,0.8569,0.0138,0.8562,1.0000,0.1431,0.0076
2,3.0,0.8555,0.0131,0.8549,1.0000,0.1445,0.0062
3,0.1,0.8461,0.0155,0.8451,0.9786,0.1325,-0.0032


In [30]:
# ============================================================
# CELDA 28 — CONFIGURACIÓN CONGELADA v1.2 HYBRID
# ============================================================

HYBRID_VERSION = "1.2.0-multilingual"

HYBRID_C = 0.3

HYBRID_EMBEDDING_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

HYBRID_EMBEDDING_DIM = 384

HYBRID_CV_F1 = 0.8574
HYBRID_CV_STD = 0.0150


print("=" * 70)
print("TECHMIND v1.2.0-MULTILINGUAL — CONFIGURACIÓN CONGELADA")
print("=" * 70)

print(f"\nVersión:          {HYBRID_VERSION}")
print(f"Encoder:          {HYBRID_EMBEDDING_MODEL}")
print(f"Embedding dim:    {HYBRID_EMBEDDING_DIM}")
print(f"Clasificador:     LinearSVC")
print(f"C:                {HYBRID_C}")
print(f"F1 Macro CV:      {HYBRID_CV_F1:.4f}")
print(f"Std CV:           {HYBRID_CV_STD:.4f}")

print("\n🔒 Arquitectura congelada antes de evaluar TEST.")

TECHMIND v1.2.0-MULTILINGUAL — CONFIGURACIÓN CONGELADA

Versión:          1.2.0-multilingual
Encoder:          sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embedding dim:    384
Clasificador:     LinearSVC
C:                0.3
F1 Macro CV:      0.8574
Std CV:           0.0150

🔒 Arquitectura congelada antes de evaluar TEST.


In [31]:
# ============================================================
# CELDA 29 — FEATURE EXTRACTOR FINAL
# ============================================================

from sklearn.base import clone

print("=" * 70)
print("ENTRENANDO TF-IDF FINAL — TRAIN COMPLETO")
print("=" * 70)

features_hybrid_final = clone(
    features_template
)

X_train_tfidf_final = (
    features_hybrid_final
    .fit_transform(
        X_train.tolist()
    )
)

X_test_tfidf_final = (
    features_hybrid_final
    .transform(
        X_test.tolist()
    )
)


print("\nTF-IDF TRAIN:")
print(X_train_tfidf_final.shape)

print("\nTF-IDF TEST:")
print(X_test_tfidf_final.shape)

ENTRENANDO TF-IDF FINAL — TRAIN COMPLETO

TF-IDF TRAIN:
(3666, 60000)

TF-IDF TEST:
(917, 60000)


In [32]:
# ============================================================
# CELDA 30 — FEATURE FUSION FINAL
# ============================================================

from scipy.sparse import (
    csr_matrix,
    hstack
)

X_train_emb_sparse = csr_matrix(
    X_train_embeddings
)

X_test_emb_sparse = csr_matrix(
    X_test_embeddings
)


X_train_hybrid = hstack(
    [
        X_train_tfidf_final,
        X_train_emb_sparse
    ],
    format="csr"
)

X_test_hybrid = hstack(
    [
        X_test_tfidf_final,
        X_test_emb_sparse
    ],
    format="csr"
)


print("=" * 70)
print("FEATURE FUSION — v1.2 HYBRID")
print("=" * 70)

print("\nTRAIN:")
print(X_train_hybrid.shape)

print("\nTEST:")
print(X_test_hybrid.shape)

print("\nComponentes:")

print(
    "TF-IDF:",
    X_train_tfidf_final.shape[1]
)

print(
    "Embeddings:",
    X_train_embeddings.shape[1]
)

print(
    "Total:",
    X_train_hybrid.shape[1]
)

FEATURE FUSION — v1.2 HYBRID

TRAIN:
(3666, 60384)

TEST:
(917, 60384)

Componentes:
TF-IDF: 60000
Embeddings: 384
Total: 60384


In [33]:
# ============================================================
# CELDA 31 — ENTRENAMIENTO FINAL
# ============================================================

from sklearn.svm import LinearSVC

import time


print("=" * 70)
print("ENTRENAMIENTO FINAL — v1.2 HYBRID")
print("=" * 70)

inicio = time.perf_counter()


modelo_hybrid_final = LinearSVC(
    C=HYBRID_C,
    random_state=RANDOM_STATE
)

modelo_hybrid_final.fit(
    X_train_hybrid,
    y_train
)


duracion_fit = (
    time.perf_counter()
    - inicio
)


print("\n✅ Entrenamiento completado")

print(
    f"Duración: "
    f"{duracion_fit:.2f} s"
)

print(
    "Clases:",
    modelo_hybrid_final.classes_
)

print(
    "Features:",
    modelo_hybrid_final.n_features_in_
)

ENTRENAMIENTO FINAL — v1.2 HYBRID

✅ Entrenamiento completado
Duración: 0.45 s
Clases: ['backend' 'cloud' 'datascience' 'frontend']
Features: 60384


In [34]:
# ============================================================
# CELDA 32 — EVALUACIÓN FINAL TEST
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


y_pred_test_hybrid = (
    modelo_hybrid_final.predict(
        X_test_hybrid
    )
)


accuracy_test_hybrid = accuracy_score(
    y_test,
    y_pred_test_hybrid
)

precision_macro_hybrid = precision_score(
    y_test,
    y_pred_test_hybrid,
    average="macro"
)

recall_macro_hybrid = recall_score(
    y_test,
    y_pred_test_hybrid,
    average="macro"
)

f1_macro_test_hybrid = f1_score(
    y_test,
    y_pred_test_hybrid,
    average="macro"
)

f1_weighted_hybrid = f1_score(
    y_test,
    y_pred_test_hybrid,
    average="weighted"
)

gap_test_cv_hybrid = (
    f1_macro_test_hybrid
    - HYBRID_CV_F1
)


print("=" * 70)
print("TECHMIND v1.2 HYBRID — TEST FINAL")
print("=" * 70)

print(
    f"\nAccuracy:         "
    f"{accuracy_test_hybrid:.4f}"
)

print(
    f"Precision Macro:  "
    f"{precision_macro_hybrid:.4f}"
)

print(
    f"Recall Macro:     "
    f"{recall_macro_hybrid:.4f}"
)

print(
    f"F1 Macro:         "
    f"{f1_macro_test_hybrid:.4f}"
)

print(
    f"F1 Weighted:      "
    f"{f1_weighted_hybrid:.4f}"
)

print(
    f"\nF1 CV:            "
    f"{HYBRID_CV_F1:.4f}"
)

print(
    f"TEST - CV:        "
    f"{gap_test_cv_hybrid:+.4f}"
)

TECHMIND v1.2 HYBRID — TEST FINAL

Accuracy:         0.8746
Precision Macro:  0.8763
Recall Macro:     0.8749
F1 Macro:         0.8753
F1 Weighted:      0.8749

F1 CV:            0.8574
TEST - CV:        +0.0179


In [35]:
# ============================================================
# CELDA 33 — CLASSIFICATION REPORT
# ============================================================

reporte_test_hybrid = classification_report(
    y_test,
    y_pred_test_hybrid,
    output_dict=True,
    zero_division=0
)

df_reporte_test_hybrid = (
    pd.DataFrame(
        reporte_test_hybrid
    )
    .T
)

print("=" * 70)
print("RENDIMIENTO POR CATEGORÍA")
print("=" * 70)

display(
    df_reporte_test_hybrid.round(4)
)

RENDIMIENTO POR CATEGORÍA


,precision,recall,f1-score,support
backend,0.8138,0.8664,0.8392,232.0000
cloud,0.9022,0.8750,0.8884,232.0000
datascience,0.9041,0.9000,0.9021,220.0000
frontend,0.8850,0.8584,0.8715,233.0000
accuracy,0.8746,0.8746,0.8746,0.8746
macro avg,0.8763,0.8749,0.8753,917.0000
weighted avg,0.8759,0.8746,0.8749,917.0000


In [36]:
# ============================================================
# CELDA 34 — v1.1.0 OFICIAL VS v1.2.0 HYBRID
# ============================================================

comparacion_versiones = pd.DataFrame({

    "metrica": [
        "F1 Macro CV",
        "Accuracy Test",
        "Precision Macro Test",
        "Recall Macro Test",
        "F1 Macro Test",
        "F1 Weighted Test"
    ],

    "v1.1.0": [
        V11_F1_MACRO_CV,
        V11_ACCURACY_TEST,
        V11_PRECISION_MACRO_TEST,
        V11_RECALL_MACRO_TEST,
        V11_F1_MACRO_TEST,
        V11_F1_WEIGHTED_TEST
    ],

    "v1.2_hybrid": [
        HYBRID_CV_F1,
        accuracy_test_hybrid,
        precision_macro_hybrid,
        recall_macro_hybrid,
        f1_macro_test_hybrid,
        f1_weighted_hybrid
    ]
})


comparacion_versiones[
    "diferencia"
] = (
    comparacion_versiones[
        "v1.2_hybrid"
    ]
    -
    comparacion_versiones[
        "v1.1.0"
    ]
)


print("=" * 70)
print("COMPARACIÓN v1.1.0 OFICIAL VS v1.2 HYBRID")
print("=" * 70)

display(
    comparacion_versiones.round(4)
)


COMPARACIÓN v1.1.0 OFICIAL VS v1.2 HYBRID


metrica,v1.1.0,v1.2_hybrid,diferencia
F1 Macro CV,0.8493,0.8574,0.0081
Accuracy Test,0.8430,0.8746,0.0316
Precision Macro Test,0.8455,0.8763,0.0308
Recall Macro Test,0.8434,0.8749,0.0315
F1 Macro Test,0.8441,0.8753,0.0312
F1 Weighted Test,0.8435,0.8749,0.0314


In [37]:
# ============================================================
# CELDA 35 — CONGELAR CANDIDATO EXPERIMENTAL
# ============================================================

CONFIG_V12 = {
    "version": "1.2.0-multilingual",
    "status": "experimental_frozen_candidate",

    "architecture": {
        "text_features": "TechMind v1.1.0 Word+Char TF-IDF",
        "embedding_model":
            "sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2",
        "embedding_dimension": 384,
        "normalize_embeddings": True,
        "classifier": "LinearSVC",
        "C": 0.3,
        "random_state": 42
    },

    "validation": {
        "f1_macro_cv": 0.8574,
        "f1_macro_cv_std": 0.0150,
        "accuracy_test": float(
            accuracy_test_hybrid
        ),
        "precision_macro_test": float(
            precision_macro_hybrid
        ),
        "recall_macro_test": float(
            recall_macro_hybrid
        ),
        "f1_macro_test": float(
            f1_macro_test_hybrid
        ),
        "f1_weighted_test": float(
            f1_weighted_hybrid
        )
    }
}

print(
    json.dumps(
        CONFIG_V12,
        indent=2,
        ensure_ascii=False
    )
)

{
  "version": "1.2.0-multilingual",
  "status": "experimental_frozen_candidate",
  "architecture": {
    "text_features": "TechMind v1.1.0 Word+Char TF-IDF",
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_dimension": 384,
    "normalize_embeddings": true,
    "classifier": "LinearSVC",
    "C": 0.3,
    "random_state": 42
  },
  "validation": {
    "f1_macro_cv": 0.8574,
    "f1_macro_cv_std": 0.015,
    "accuracy_test": 0.8745910577971646,
    "precision_macro_test": 0.8762631864154856,
    "recall_macro_test": 0.8749371022643185,
    "f1_macro_test": 0.875290217235722,
    "f1_weighted_test": 0.8749358559683713
  }
}


In [38]:
# ============================================================
# CELDA 36 — GUARDAR ARTEFACTO EXPERIMENTAL
# ============================================================

import joblib
import hashlib


MODEL_DIR_V12 = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
)

MODEL_DIR_V12.mkdir(
    parents=True,
    exist_ok=True
)

PATH_MODEL_V12 = (
    MODEL_DIR_V12
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)


artefacto_v12 = {

    "version":
        "1.2.0-multilingual",

    "status":
        "experimental",

    "features":
        features_hybrid_final,

    "classifier":
        modelo_hybrid_final,

    "embedding_model":
        HYBRID_EMBEDDING_MODEL,

    "embedding_dimension":
        HYBRID_EMBEDDING_DIM,

    "normalize_embeddings":
        True,

    "configuration":
        CONFIG_V12
}


joblib.dump(
    artefacto_v12,
    PATH_MODEL_V12
)


# SHA256
sha256 = hashlib.sha256()

with open(
    PATH_MODEL_V12,
    "rb"
) as f:

    for bloque in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha256.update(bloque)


SHA_V12 = sha256.hexdigest()


print("=" * 70)
print("ARTEFACTO EXPERIMENTAL CONGELADO")
print("=" * 70)

print("\nPath:")
print(PATH_MODEL_V12)

print("\nSHA256:")
print(SHA_V12)

print(
    "\nTamaño:",
    f"{PATH_MODEL_V12.stat().st_size / 1024**2:.2f} MB"
)

ARTEFACTO EXPERIMENTAL CONGELADO

Path:
C:\Users\MAMÁ\Downloads\techmind-v2\models\experimental\v1.2.0-multilingual\techmind_hybrid_v1_2_0_multilingual.joblib

SHA256:
521e791eb9c63bb4634dfce714c5050690b08a7bb51d367cc14014f6b884e3b8

Tamaño: 3.78 MB


In [39]:
# ============================================================
# CELDA 37 — MULTILINGUAL DEVELOPMENT BENCHMARK
# ============================================================

PATH_MULTILINGUAL = (
    ROOT
    / "data"
    / "evaluation"
    / "multilingual_benchmark.csv"
)

df_multi = pd.read_csv(
    PATH_MULTILINGUAL
)


print("=" * 70)
print("MULTILINGUAL DEVELOPMENT BENCHMARK")
print("=" * 70)

print("\nDocumentos:")
print(len(df_multi))

print("\nIdiomas:")
print(
    df_multi["idioma"]
    .value_counts()
)

print("\nCategorías:")
print(
    df_multi["categoria_real"]
    .value_counts()
)

MULTILINGUAL DEVELOPMENT BENCHMARK

Documentos:
80

Idiomas:
idioma
es       20
en       20
ru       20
es_en    20
Name: count, dtype: int64

Categorías:
categoria_real
backend        20
cloud          20
datascience    20
frontend       20
Name: count, dtype: int64


In [40]:
# ============================================================
# CELDA 38 — EMBEDDINGS MULTILINGÜES
# ============================================================

textos_multi = (
    df_multi["texto"]
    .fillna("")
    .astype(str)
    .tolist()
)


X_multi_embeddings = encoder.encode(
    textos_multi,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)


print("=" * 70)
print("EMBEDDINGS MULTILINGÜES")
print("=" * 70)

print(
    "\nShape:",
    X_multi_embeddings.shape
)

print(
    "NaN:",
    np.isnan(
        X_multi_embeddings
    ).sum()
)

print(
    "Inf:",
    np.isinf(
        X_multi_embeddings
    ).sum()
)

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.92it/s]

EMBEDDINGS MULTILINGÜES

Shape: (80, 384)
NaN: 0
Inf: 0


In [41]:
# ============================================================
# CELDA 39 — TF-IDF CONGELADO SOBRE BENCHMARK
# ============================================================

X_multi_tfidf = (
    features_hybrid_final
    .transform(
        textos_multi
    )
)


print("=" * 70)
print("TF-IDF MULTILINGÜE")
print("=" * 70)

print(
    "\nShape:",
    X_multi_tfidf.shape
)

TF-IDF MULTILINGÜE

Shape: (80, 60000)


In [42]:
# ============================================================
# CELDA 40 — FUSIÓN MULTILINGÜE
# ============================================================

X_multi_hybrid = hstack(
    [
        X_multi_tfidf,
        csr_matrix(
            X_multi_embeddings
        )
    ],
    format="csr"
)


print("=" * 70)
print("FEATURES MULTILINGÜES HÍBRIDAS")
print("=" * 70)

print(
    "\nShape:",
    X_multi_hybrid.shape
)

print(
    "Dimensión esperada:",
    modelo_hybrid_final.n_features_in_
)

assert (
    X_multi_hybrid.shape[1]
    ==
    modelo_hybrid_final.n_features_in_
)

FEATURES MULTILINGÜES HÍBRIDAS

Shape: (80, 60384)
Dimensión esperada: 60384


In [43]:
# ============================================================
# CELDA 41 — PREDICCIONES v1.2 MULTILINGUAL
# ============================================================

pred_multi_v12 = (
    modelo_hybrid_final.predict(
        X_multi_hybrid
    )
)


df_multi_v12 = (
    df_multi.copy()
)

df_multi_v12[
    "categoria_predicha_v12"
] = pred_multi_v12

df_multi_v12[
    "correcta_v12"
] = (
    df_multi_v12[
        "categoria_real"
    ]
    ==
    df_multi_v12[
        "categoria_predicha_v12"
    ]
)


correctas_v12 = int(
    df_multi_v12[
        "correcta_v12"
    ].sum()
)

accuracy_multi_v12 = (
    df_multi_v12[
        "correcta_v12"
    ].mean()
)


print("=" * 70)
print("TECHMIND v1.2 — MULTILINGUAL RAW RESULTS")
print("=" * 70)

print(
    f"\nCorrectas: "
    f"{correctas_v12}/"
    f"{len(df_multi_v12)}"
)

print(
    f"Accuracy: "
    f"{accuracy_multi_v12:.2%}"
)

TECHMIND v1.2 — MULTILINGUAL RAW RESULTS

Correctas: 79/80
Accuracy: 98.75%


In [44]:
# ============================================================
# CELDA 42 — ACCURACY POR IDIOMA
# ============================================================

accuracy_idioma_v12 = (
    df_multi_v12
    .groupby(
        "idioma"
    )
    .agg(
        documentos=(
            "correcta_v12",
            "size"
        ),
        correctas=(
            "correcta_v12",
            "sum"
        ),
        accuracy=(
            "correcta_v12",
            "mean"
        )
    )
    .reset_index()
)


print("=" * 70)
print("v1.2 — ACCURACY POR IDIOMA")
print("=" * 70)

display(
    accuracy_idioma_v12.round(4)
)

v1.2 — ACCURACY POR IDIOMA


,idioma,documentos,correctas,accuracy
0,en,20,20,1.00
1,es,20,20,1.00
2,es_en,20,20,1.00
3,ru,20,19,0.95


In [45]:
# ============================================================
# CELDA 43 — IDIOMA × CATEGORÍA
# ============================================================

accuracy_lang_cat_v12 = (
    df_multi_v12
    .groupby([
        "idioma",
        "categoria_real"
    ])[
        "correcta_v12"
    ]
    .mean()
    .unstack()
)


print("=" * 70)
print("v1.2 — ACCURACY IDIOMA × CATEGORÍA")
print("=" * 70)

display(
    accuracy_lang_cat_v12.round(4)
)

v1.2 — ACCURACY IDIOMA × CATEGORÍA


categoria_real,backend,cloud,datascience,frontend
idioma,,,,
en,1.0,1.0,1.0,1.0
es,1.0,1.0,1.0,1.0
es_en,1.0,1.0,1.0,1.0
ru,1.0,1.0,1.0,0.8


In [46]:
# ============================================================
# CELDA 44 — CONSISTENCIA CROSS-LANGUAGE v1.2
# ============================================================

pivot_v12 = (
    df_multi_v12
    .pivot(
        index="case_id",
        columns="idioma",
        values="categoria_predicha_v12"
    )
)


pivot_v12[
    "n_predicciones"
] = (
    pivot_v12[
        [
            "es",
            "en",
            "ru",
            "es_en"
        ]
    ]
    .nunique(
        axis=1
    )
)


pivot_v12[
    "consistente"
] = (
    pivot_v12[
        "n_predicciones"
    ]
    == 1
)


consistencia_v12 = (
    pivot_v12[
        "consistente"
    ]
    .mean()
)


print("=" * 70)
print("CONSISTENCIA CROSS-LANGUAGE — v1.2")
print("=" * 70)

print(
    f"\nCasos consistentes: "
    f"{pivot_v12['consistente'].sum()}"
    f"/{len(pivot_v12)}"
)

print(
    f"Consistencia: "
    f"{consistencia_v12:.2%}"
)

display(
    pivot_v12
)

CONSISTENCIA CROSS-LANGUAGE — v1.2

Casos consistentes: 19/20
Consistencia: 95.00%


idioma,en,es,es_en,ru,n_predicciones,consistente
case_id,,,,,,
backend_001,backend,backend,backend,backend,1,True
backend_002,backend,backend,backend,backend,1,True
backend_003,backend,backend,backend,backend,1,True
backend_004,backend,backend,backend,backend,1,True
backend_005,backend,backend,backend,backend,1,True
cloud_001,cloud,cloud,cloud,cloud,1,True
cloud_002,cloud,cloud,cloud,cloud,1,True
cloud_003,cloud,cloud,cloud,cloud,1,True
cloud_004,cloud,cloud,cloud,cloud,1,True


In [47]:
# ============================================================
# CELDA 45 — COMPARACIÓN MULTILINGÜE
# ============================================================

baseline_idioma = {
    "en": 0.95,
    "es": 0.75,
    "es_en": 0.95,
    "ru": 0.80
}


filas = []

for idioma in [
    "es",
    "en",
    "es_en",
    "ru"
]:

    accuracy_v12 = float(
        accuracy_idioma_v12.loc[
            accuracy_idioma_v12[
                "idioma"
            ] == idioma,
            "accuracy"
        ].iloc[0]
    )

    filas.append({
        "metrica":
            f"Accuracy {idioma}",

        "v1.1":
            baseline_idioma[
                idioma
            ],

        "v1.2_hybrid":
            accuracy_v12
    })


filas.extend([
    {
        "metrica":
            "Accuracy global",

        "v1.1":
            0.8625,

        "v1.2_hybrid":
            accuracy_multi_v12
    },

    {
        "metrica":
            "Cross-language consistency",

        "v1.1":
            0.65,

        "v1.2_hybrid":
            consistencia_v12
    }
])


df_comparacion_multi = (
    pd.DataFrame(
        filas
    )
)


df_comparacion_multi[
    "diferencia"
] = (
    df_comparacion_multi[
        "v1.2_hybrid"
    ]
    -
    df_comparacion_multi[
        "v1.1"
    ]
)


print("=" * 70)
print("v1.1 VS v1.2 — MULTILINGUAL DEVELOPMENT BENCHMARK")
print("=" * 70)

display(
    df_comparacion_multi.round(4)
)

v1.1 VS v1.2 — MULTILINGUAL DEVELOPMENT BENCHMARK


,metrica,v1.1,v1.2_hybrid,diferencia
0,Accuracy es,0.7500,1.0000,0.250
1,Accuracy en,0.9500,1.0000,0.050
2,Accuracy es_en,0.9500,1.0000,0.050
3,Accuracy ru,0.8000,0.9500,0.150
4,Accuracy global,0.8625,0.9875,0.125
5,Cross-language consistency,0.6500,0.9500,0.300


In [48]:
# ============================================================
# CELDA 46 — ÚNICO ERROR MULTILINGÜE v1.2
# ============================================================

errores_v12 = (
    df_multi_v12[
        ~df_multi_v12["correcta_v12"]
    ]
    .copy()
)

print("=" * 70)
print("ERRORES MULTILINGÜES — v1.2")
print("=" * 70)

print(
    "\nTotal:",
    len(errores_v12)
)

display(
    errores_v12[
        [
            "case_id",
            "idioma",
            "categoria_real",
            "categoria_predicha_v12",
            "dificultad",
            "texto"
        ]
    ]
)

ERRORES MULTILINGÜES — v1.2

Total: 1


,case_id,idioma,categoria_real,categoria_predicha_v12,dificultad,texto
78,frontend_005,ru,frontend,backend,dificil,"Создать SPA на TypeScript, которая обновляет D..."


In [49]:
# ============================================================
# CELDA 47 — SCORES Y MÁRGENES MULTILINGÜES
# ============================================================

scores_multi_v12 = (
    modelo_hybrid_final
    .decision_function(
        X_multi_hybrid
    )
)

clases_v12 = (
    modelo_hybrid_final.classes_
)

orden_scores = np.argsort(
    scores_multi_v12,
    axis=1
)

idx_top1 = orden_scores[:, -1]
idx_top2 = orden_scores[:, -2]


score_top1 = scores_multi_v12[
    np.arange(len(scores_multi_v12)),
    idx_top1
]

score_top2 = scores_multi_v12[
    np.arange(len(scores_multi_v12)),
    idx_top2
]


df_multi_v12[
    "segunda_categoria_v12"
] = clases_v12[
    idx_top2
]

df_multi_v12[
    "score_top1_v12"
] = score_top1

df_multi_v12[
    "score_top2_v12"
] = score_top2

df_multi_v12[
    "margen_v12"
] = (
    score_top1
    -
    score_top2
)


print("=" * 70)
print("ERROR RESTANTE — DETALLE")
print("=" * 70)

display(
    df_multi_v12[
        ~df_multi_v12["correcta_v12"]
    ][
        [
            "case_id",
            "idioma",
            "categoria_real",
            "categoria_predicha_v12",
            "segunda_categoria_v12",
            "score_top1_v12",
            "score_top2_v12",
            "margen_v12",
            "texto"
        ]
    ].round(4)
)

ERROR RESTANTE — DETALLE


,case_id,idioma,categoria_real,categoria_predicha_v12,segunda_categoria_v12,score_top1_v12,score_top2_v12,margen_v12,texto
78,frontend_005,ru,frontend,backend,frontend,0.261,0.2569,0.0041,"Создать SPA на TypeScript, которая обновляет D..."


In [50]:
# ============================================================
# CELDA 48 — OOF PREDICTIONS v1.2 HYBRID
# ============================================================

from sklearn.base import clone
from sklearn.svm import LinearSVC
from scipy.sparse import csr_matrix, hstack


n_train = len(X_train)

clases_globales = np.array(
    sorted(
        y_train.unique()
    )
)

n_clases = len(
    clases_globales
)


oof_scores = np.full(
    (
        n_train,
        n_clases
    ),
    np.nan
)

oof_pred = np.empty(
    n_train,
    dtype=object
)

oof_tfidf_activas = np.zeros(
    n_train,
    dtype=int
)


indices = np.arange(
    n_train
)


print("=" * 70)
print("OOF v1.2 HYBRID")
print("=" * 70)


for fold, (
    idx_train,
    idx_val
) in enumerate(
    cv.split(
        indices,
        y_train
    ),
    start=1
):

    print(
        f"\nFold {fold}/{N_SPLITS}"
    )

    # ------------------------------------------
    # TF-IDF entrenado exclusivamente en fold
    # ------------------------------------------

    features_fold = clone(
        features_template
    )

    X_tfidf_fold_train = (
        features_fold
        .fit_transform(
            X_train.iloc[
                idx_train
            ].tolist()
        )
    )

    X_tfidf_fold_val = (
        features_fold
        .transform(
            X_train.iloc[
                idx_val
            ].tolist()
        )
    )

    # ------------------------------------------
    # Embeddings
    # ------------------------------------------

    X_emb_fold_train = csr_matrix(
        X_train_embeddings[
            idx_train
        ]
    )

    X_emb_fold_val = csr_matrix(
        X_train_embeddings[
            idx_val
        ]
    )

    # ------------------------------------------
    # Hybrid
    # ------------------------------------------

    X_hybrid_fold_train = hstack(
        [
            X_tfidf_fold_train,
            X_emb_fold_train
        ],
        format="csr"
    )

    X_hybrid_fold_val = hstack(
        [
            X_tfidf_fold_val,
            X_emb_fold_val
        ],
        format="csr"
    )

    # ------------------------------------------
    # Modelo congelado
    # ------------------------------------------

    clf_fold = LinearSVC(
        C=0.3,
        random_state=RANDOM_STATE
    )

    clf_fold.fit(
        X_hybrid_fold_train,
        y_train.iloc[
            idx_train
        ]
    )

    scores_fold = (
        clf_fold
        .decision_function(
            X_hybrid_fold_val
        )
    )

    pred_fold = (
        clf_fold.predict(
            X_hybrid_fold_val
        )
    )

    # Validación del orden de clases
    assert np.array_equal(
        clf_fold.classes_,
        clases_globales
    )

    oof_scores[
        idx_val
    ] = scores_fold

    oof_pred[
        idx_val
    ] = pred_fold

    # Número de features TF-IDF activas.
    # Solo diagnóstico por ahora.
    oof_tfidf_activas[
        idx_val
    ] = np.asarray(
        (
            X_tfidf_fold_val
            != 0
        ).sum(
            axis=1
        )
    ).ravel()


print("\n✅ OOF completado")

print(
    "Scores faltantes:",
    np.isnan(
        oof_scores
    ).sum()
)

OOF v1.2 HYBRID

Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5

✅ OOF completado
Scores faltantes: 0


In [51]:
# ============================================================
# CELDA 49 — DATAFRAME OOF
# ============================================================

from sklearn.metrics import f1_score

# ------------------------------------------------------------
# 1. Ordenar scores por predicción
# ------------------------------------------------------------

orden_oof = np.argsort(
    oof_scores,
    axis=1
)

idx_oof_top1 = orden_oof[:, -1]
idx_oof_top2 = orden_oof[:, -2]


# ------------------------------------------------------------
# 2. Obtener scores top-1 y top-2
# ------------------------------------------------------------

oof_score_top1 = oof_scores[
    np.arange(n_train),
    idx_oof_top1
]

oof_score_top2 = oof_scores[
    np.arange(n_train),
    idx_oof_top2
]


# ------------------------------------------------------------
# 3. Segunda categoría
# ------------------------------------------------------------

oof_segunda = clases_globales[
    idx_oof_top2
]


# ------------------------------------------------------------
# 4. Margen de decisión
# ------------------------------------------------------------

oof_margen = (
    oof_score_top1
    -
    oof_score_top2
)


# ------------------------------------------------------------
# 5. Construir DataFrame OOF
# ------------------------------------------------------------

df_oof_v12 = pd.DataFrame({

    "categoria_real":
        y_train.to_numpy(),

    "categoria_predicha":
        oof_pred,

    "segunda_categoria":
        oof_segunda,

    "score_top1":
        oof_score_top1,

    "score_top2":
        oof_score_top2,

    "margen":
        oof_margen,

    "tfidf_features_activas":
        oof_tfidf_activas
})


# ------------------------------------------------------------
# 6. Identificar predicciones correctas
# ------------------------------------------------------------

df_oof_v12["correcta"] = (
    df_oof_v12["categoria_real"]
    ==
    df_oof_v12["categoria_predicha"]
)


# ------------------------------------------------------------
# 7. Métricas OOF
# ------------------------------------------------------------

accuracy_oof_v12 = (
    df_oof_v12["correcta"].mean()
)

f1_macro_oof_v12 = f1_score(
    df_oof_v12["categoria_real"],
    df_oof_v12["categoria_predicha"],
    average="macro"
)


# ------------------------------------------------------------
# 8. Resumen
# ------------------------------------------------------------

print("=" * 70)
print("DIAGNÓSTICO OOF — v1.2")
print("=" * 70)

print(
    f"\nAccuracy OOF: "
    f"{accuracy_oof_v12:.4f}"
)

print(
    f"F1 Macro OOF: "
    f"{f1_macro_oof_v12:.4f}"
)


# ------------------------------------------------------------
# 9. Cantidad de aciertos y errores
# ------------------------------------------------------------

n_correctas_oof = int(
    df_oof_v12["correcta"].sum()
)

n_errores_oof = int(
    (~df_oof_v12["correcta"]).sum()
)

print(
    f"\nCorrectas: "
    f"{n_correctas_oof}/{len(df_oof_v12)}"
)

print(
    f"Errores:   "
    f"{n_errores_oof}/{len(df_oof_v12)}"
)


# ------------------------------------------------------------
# 10. Distribución del margen — correctas
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MARGEN — PREDICCIONES CORRECTAS")
print("=" * 70)

display(
    df_oof_v12.loc[
        df_oof_v12["correcta"],
        "margen"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)


# ------------------------------------------------------------
# 11. Distribución del margen — incorrectas
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MARGEN — PREDICCIONES INCORRECTAS")
print("=" * 70)

display(
    df_oof_v12.loc[
        ~df_oof_v12["correcta"],
        "margen"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)


# ------------------------------------------------------------
# 12. Validación
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDACIÓN OOF")
print("=" * 70)

print(
    "Scores NaN:",
    np.isnan(oof_scores).sum()
)

print(
    "Margen NaN:",
    df_oof_v12["margen"].isna().sum()
)

print(
    "Predicciones vacías:",
    df_oof_v12[
        "categoria_predicha"
    ].isna().sum()
)

DIAGNÓSTICO OOF — v1.2

Accuracy OOF: 0.8565
F1 Macro OOF: 0.8573

Correctas: 3140/3666
Errores:   526/3666

MARGEN — PREDICCIONES CORRECTAS


count    3140.000000
mean        1.427452
std         0.817049
min         0.003385
10%         0.351507
25%         0.782498
50%         1.375643
75%         2.040976
90%         2.486217
max         5.492442
Name: margen, dtype: float64


MARGEN — PREDICCIONES INCORRECTAS


count    526.000000
mean       0.401866
std        0.385337
min        0.000677
10%        0.051204
25%        0.120833
50%        0.296762
75%        0.567436
90%        0.910920
max        3.072878
Name: margen, dtype: float64


VALIDACIÓN OOF
Scores NaN: 0
Margen NaN: 0
Predicciones vacías: 0


In [52]:
# ============================================================
# CELDA 50 — CALIBRACIÓN DE MARGEN
# ============================================================

thresholds = np.unique(
    np.quantile(
        df_oof_v12["margen"],
        np.linspace(
            0,
            0.95,
            300
        )
    )
)


filas_calibracion = []

total_errores_oof = int(
    (
        ~df_oof_v12[
            "correcta"
        ]
    ).sum()
)


for threshold in thresholds:

    aceptada = (
        df_oof_v12[
            "margen"
        ]
        >= threshold
    )

    n_aceptadas = int(
        aceptada.sum()
    )

    if n_aceptadas == 0:
        continue

    accuracy_aceptadas = (
        df_oof_v12.loc[
            aceptada,
            "correcta"
        ]
        .mean()
    )

    cobertura = (
        n_aceptadas
        /
        len(df_oof_v12)
    )

    errores_aceptados = int(
        (
            ~df_oof_v12.loc[
                aceptada,
                "correcta"
            ]
        ).sum()
    )

    errores_capturados = (
        total_errores_oof
        -
        errores_aceptados
    )

    error_capture = (
        errores_capturados
        /
        total_errores_oof
        if total_errores_oof
        else 1.0
    )

    filas_calibracion.append({

        "threshold":
            threshold,

        "coverage":
            cobertura,

        "accepted":
            n_aceptadas,

        "accepted_accuracy":
            accuracy_aceptadas,

        "error_capture":
            error_capture,

        "errors_accepted":
            errores_aceptados
    })


df_calibracion_v12 = (
    pd.DataFrame(
        filas_calibracion
    )
)

In [53]:
# ============================================================
# CELDA 51 — UMBRALES CANDIDATOS
# ============================================================

objetivos = [
    0.95,
    0.97,
    0.98
]

candidatos_threshold = []


for objetivo in objetivos:

    candidatos = (
        df_calibracion_v12[
            df_calibracion_v12[
                "accepted_accuracy"
            ]
            >= objetivo
        ]
        .sort_values(
            [
                "coverage",
                "threshold"
            ],
            ascending=[
                False,
                True
            ]
        )
    )

    if len(candidatos) == 0:
        continue

    mejor = (
        candidatos.iloc[0]
    )

    candidatos_threshold.append({

        "target_accuracy":
            objetivo,

        "threshold":
            mejor[
                "threshold"
            ],

        "coverage":
            mejor[
                "coverage"
            ],

        "accepted_accuracy":
            mejor[
                "accepted_accuracy"
            ],

        "error_capture":
            mejor[
                "error_capture"
            ],

        "errors_accepted":
            int(
                mejor[
                    "errors_accepted"
                ]
            )
    })


df_threshold_candidates = (
    pd.DataFrame(
        candidatos_threshold
    )
)


print("=" * 70)
print("CANDIDATOS DE UMBRAL OPERACIONAL")
print("=" * 70)

display(
    df_threshold_candidates.round(4)
)

CANDIDATOS DE UMBRAL OPERACIONAL


,target_accuracy,threshold,coverage,accepted_accuracy,error_capture,errors_accepted
0,0.95,0.5327,0.7616,0.9506,0.7376,138
1,0.97,0.8132,0.6506,0.9715,0.8707,68
2,0.98,0.9837,0.5805,0.9807,0.9221,41


In [54]:
# ============================================================
# CELDA 52 — UMBRAL OPERACIONAL CANDIDATO
# ============================================================

UMBRAL_MARGEN_REVISION_V12 = 0.8132

TARGET_ACCEPTED_ACCURACY_V12 = 0.97

print("=" * 70)
print("CONTROL OPERACIONAL v1.2 — CANDIDATO CONGELADO")
print("=" * 70)

print(
    f"\nUmbral margen:         "
    f"{UMBRAL_MARGEN_REVISION_V12:.4f}"
)

print(
    f"Objetivo accuracy:     "
    f"{TARGET_ACCEPTED_ACCURACY_V12:.0%}"
)

print(
    "\nFuente de calibración:"
    "\n5-fold OOF sobre TRAIN"
)

print(
    "\nTEST y benchmark multilingüe:"
    "\nNO utilizados para seleccionar el umbral."
)

CONTROL OPERACIONAL v1.2 — CANDIDATO CONGELADO

Umbral margen:         0.8132
Objetivo accuracy:     97%

Fuente de calibración:
5-fold OOF sobre TRAIN

TEST y benchmark multilingüe:
NO utilizados para seleccionar el umbral.


In [55]:
# ============================================================
# CELDA 53 — MÁRGENES SOBRE TEST
# ============================================================

scores_test_v12 = (
    modelo_hybrid_final
    .decision_function(
        X_test_hybrid
    )
)

orden_test = np.argsort(
    scores_test_v12,
    axis=1
)

idx_test_top1 = orden_test[:, -1]
idx_test_top2 = orden_test[:, -2]

score_test_top1 = scores_test_v12[
    np.arange(len(scores_test_v12)),
    idx_test_top1
]

score_test_top2 = scores_test_v12[
    np.arange(len(scores_test_v12)),
    idx_test_top2
]

segunda_test = (
    modelo_hybrid_final.classes_[
        idx_test_top2
    ]
)

margen_test = (
    score_test_top1
    -
    score_test_top2
)


df_test_operacional_v12 = pd.DataFrame({

    "categoria_real":
        y_test.to_numpy(),

    "categoria_predicha":
        y_pred_test_hybrid,

    "segunda_categoria":
        segunda_test,

    "margen":
        margen_test
})


df_test_operacional_v12[
    "correcta"
] = (
    df_test_operacional_v12[
        "categoria_real"
    ]
    ==
    df_test_operacional_v12[
        "categoria_predicha"
    ]
)


df_test_operacional_v12[
    "estado"
] = np.where(
    df_test_operacional_v12[
        "margen"
    ]
    >= UMBRAL_MARGEN_REVISION_V12,
    "aceptada",
    "revision"
)

In [56]:
# ============================================================
# CELDA 54 — CONTROL OPERACIONAL EN TEST
# ============================================================

mask_aceptadas_test = (
    df_test_operacional_v12[
        "estado"
    ]
    == "aceptada"
)

n_test = len(
    df_test_operacional_v12
)

n_aceptadas_test = int(
    mask_aceptadas_test.sum()
)

cobertura_test = (
    n_aceptadas_test
    /
    n_test
)


accuracy_aceptadas_test = (
    df_test_operacional_v12.loc[
        mask_aceptadas_test,
        "correcta"
    ]
    .mean()
)


errores_test = int(
    (
        ~df_test_operacional_v12[
            "correcta"
        ]
    ).sum()
)


errores_aceptados_test = int(
    (
        mask_aceptadas_test
        &
        ~df_test_operacional_v12[
            "correcta"
        ]
    ).sum()
)


errores_capturados_test = (
    errores_test
    -
    errores_aceptados_test
)


error_capture_test = (
    errores_capturados_test
    /
    errores_test
    if errores_test
    else 1.0
)


print("=" * 70)
print("OPERACIÓN v1.2 — TEST")
print("=" * 70)

print(
    f"\nDocumentos:             "
    f"{n_test}"
)

print(
    f"Aceptadas:              "
    f"{n_aceptadas_test}"
)

print(
    f"Revisión:               "
    f"{n_test - n_aceptadas_test}"
)

print(
    f"\nCobertura automática:   "
    f"{cobertura_test:.2%}"
)

print(
    f"Accuracy aceptadas:     "
    f"{accuracy_aceptadas_test:.2%}"
)

print(
    f"Error capture:          "
    f"{error_capture_test:.2%}"
)

print(
    f"Errores totales:        "
    f"{errores_test}"
)

print(
    f"Errores aceptados:      "
    f"{errores_aceptados_test}"
)

OPERACIÓN v1.2 — TEST

Documentos:             917
Aceptadas:              627
Revisión:               290

Cobertura automática:   68.38%
Accuracy aceptadas:     96.33%
Error capture:          80.00%
Errores totales:        115
Errores aceptados:      23


In [57]:
# ============================================================
# CELDA 55 — CONTROL OPERACIONAL MULTILINGÜE
# ============================================================

df_multi_v12[
    "estado_v12"
] = np.where(
    df_multi_v12[
        "margen_v12"
    ]
    >= UMBRAL_MARGEN_REVISION_V12,
    "aceptada",
    "revision"
)


mask_multi_aceptadas = (
    df_multi_v12[
        "estado_v12"
    ]
    == "aceptada"
)


n_multi_aceptadas = int(
    mask_multi_aceptadas.sum()
)


accuracy_multi_aceptadas = (
    df_multi_v12.loc[
        mask_multi_aceptadas,
        "correcta_v12"
    ]
    .mean()
)


errores_multi = int(
    (
        ~df_multi_v12[
            "correcta_v12"
        ]
    ).sum()
)


errores_multi_aceptados = int(
    (
        mask_multi_aceptadas
        &
        ~df_multi_v12[
            "correcta_v12"
        ]
    ).sum()
)


error_capture_multi = (
    (
        errores_multi
        -
        errores_multi_aceptados
    )
    /
    errores_multi
    if errores_multi
    else 1.0
)


print("=" * 70)
print("OPERACIÓN MULTILINGÜE — v1.2")
print("=" * 70)

print(
    f"\nDocumentos:           "
    f"{len(df_multi_v12)}"
)

print(
    f"Aceptadas:            "
    f"{n_multi_aceptadas}"
)

print(
    f"Revisión:             "
    f"{len(df_multi_v12) - n_multi_aceptadas}"
)

print(
    f"\nCobertura:            "
    f"{n_multi_aceptadas / len(df_multi_v12):.2%}"
)

print(
    f"Accuracy aceptadas:   "
    f"{accuracy_multi_aceptadas:.2%}"
)

print(
    f"Error capture:        "
    f"{error_capture_multi:.2%}"
)

OPERACIÓN MULTILINGÜE — v1.2

Documentos:           80
Aceptadas:            57
Revisión:             23

Cobertura:            71.25%
Accuracy aceptadas:   100.00%
Error capture:        100.00%


In [58]:
# ============================================================
# CELDA 56 — OPERACIÓN POR IDIOMA
# ============================================================

resumen_operacional_idioma = []

for idioma, grupo in (
    df_multi_v12.groupby(
        "idioma"
    )
):

    aceptadas = (
        grupo[
            "estado_v12"
        ]
        == "aceptada"
    )

    n = len(grupo)

    n_aceptadas = int(
        aceptadas.sum()
    )

    accuracy_raw = (
        grupo[
            "correcta_v12"
        ]
        .mean()
    )

    accuracy_accepted = (
        grupo.loc[
            aceptadas,
            "correcta_v12"
        ]
        .mean()
        if n_aceptadas > 0
        else np.nan
    )

    errores = int(
        (
            ~grupo[
                "correcta_v12"
            ]
        ).sum()
    )

    errores_aceptados = int(
        (
            aceptadas
            &
            ~grupo[
                "correcta_v12"
            ]
        ).sum()
    )

    capture = (
        (
            errores
            -
            errores_aceptados
        )
        /
        errores
        if errores > 0
        else 1.0
    )

    resumen_operacional_idioma.append({

        "idioma":
            idioma,

        "documentos":
            n,

        "aceptadas":
            n_aceptadas,

        "coverage":
            n_aceptadas / n,

        "accuracy_raw":
            accuracy_raw,

        "accepted_accuracy":
            accuracy_accepted,

        "error_capture":
            capture
    })


df_operacional_idioma_v12 = (
    pd.DataFrame(
        resumen_operacional_idioma
    )
)


print("=" * 70)
print("OPERACIÓN MULTILINGÜE POR IDIOMA")
print("=" * 70)

display(
    df_operacional_idioma_v12.round(4)
)

OPERACIÓN MULTILINGÜE POR IDIOMA


,idioma,documentos,aceptadas,coverage,accuracy_raw,accepted_accuracy,error_capture
0,en,20,16,0.80,1.00,1.0,1.0
1,es,20,12,0.60,1.00,1.0,1.0
2,es_en,20,16,0.80,1.00,1.0,1.0
3,ru,20,13,0.65,0.95,1.0,1.0


In [59]:
# ============================================================
# CELDA 57 — ÍNDICE SEMÁNTICO DEL DOMINIO
# ============================================================

from sklearn.neighbors import NearestNeighbors

N_NEIGHBORS_DOMAIN = 5

domain_index = NearestNeighbors(
    n_neighbors=N_NEIGHBORS_DOMAIN,
    metric="cosine",
    algorithm="brute",
    n_jobs=1
)

domain_index.fit(
    X_train_embeddings
)

print("=" * 70)
print("ÍNDICE SEMÁNTICO TECHMIND")
print("=" * 70)

print(
    "\nDocumentos indexados:",
    len(X_train_embeddings)
)

print(
    "Dimensión:",
    X_train_embeddings.shape[1]
)

print(
    "Vecinos:",
    N_NEIGHBORS_DOMAIN
)

ÍNDICE SEMÁNTICO TECHMIND

Documentos indexados: 3666
Dimensión: 384
Vecinos: 5


In [60]:
# ============================================================
# CELDA 58 — SIMILITUD TEST ↔ TRAIN
# ============================================================

distancias_test, indices_nn_test = (
    domain_index.kneighbors(
        X_test_embeddings
    )
)

similitudes_test = (
    1.0
    -
    distancias_test
)

similitud_max_test = (
    similitudes_test[:, 0]
)

similitud_media5_test = (
    similitudes_test.mean(
        axis=1
    )
)


df_test_operacional_v12[
    "similitud_max_train"
] = similitud_max_test

df_test_operacional_v12[
    "similitud_media_5nn"
] = similitud_media5_test


print("=" * 70)
print("SIMILITUD SEMÁNTICA — TEST")
print("=" * 70)

display(
    df_test_operacional_v12[
        [
            "similitud_max_train",
            "similitud_media_5nn"
        ]
    ].describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

SIMILITUD SEMÁNTICA — TEST


,similitud_max_train,similitud_media_5nn
count,917.000000,917.000000
mean,0.662880,0.607253
std,0.156673,0.152925
min,0.352077,0.348859
1%,0.422022,0.395021
5%,0.473217,0.438116
10%,0.496659,0.465558
25%,0.548690,0.505412
50%,0.618172,0.562382
75%,0.748810,0.654494


In [61]:
# ============================================================
# CELDA 59 — LATENCIA END-TO-END v1.2
# ============================================================

import time

textos_latency = (
    X_test.iloc[:100]
    .tolist()
)

# Warm-up
_ = encoder.encode(
    textos_latency[:5],
    normalize_embeddings=True,
    show_progress_bar=False
)


inicio = time.perf_counter()

emb_latency = encoder.encode(
    textos_latency,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

tfidf_latency = (
    features_hybrid_final
    .transform(
        textos_latency
    )
)

hybrid_latency = hstack(
    [
        tfidf_latency,
        csr_matrix(
            emb_latency
        )
    ],
    format="csr"
)

pred_latency = (
    modelo_hybrid_final
    .predict(
        hybrid_latency
    )
)

duracion_total = (
    time.perf_counter()
    -
    inicio
)

latencia_media_ms = (
    duracion_total
    /
    len(textos_latency)
    * 1000
)


print("=" * 70)
print("LATENCIA v1.2 HYBRID")
print("=" * 70)

print(
    f"\nDocumentos:      "
    f"{len(textos_latency)}"
)

print(
    f"Tiempo total:    "
    f"{duracion_total:.3f} s"
)

print(
    f"Media/documento: "
    f"{latencia_media_ms:.2f} ms"
)

print(
    f"Throughput:      "
    f"{len(textos_latency) / duracion_total:.2f} docs/s"
)

LATENCIA v1.2 HYBRID

Documentos:      100
Tiempo total:    2.583 s
Media/documento: 25.83 ms
Throughput:      38.72 docs/s


In [62]:
# ============================================================
# CELDA 60 — LATENCIA INDIVIDUAL
# ============================================================

textos_individuales = (
    X_test.iloc[:50]
    .tolist()
)

latencias_ms = []


for texto in textos_individuales:

    inicio = time.perf_counter()

    emb = encoder.encode(
        [texto],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    tfidf = (
        features_hybrid_final
        .transform(
            [texto]
        )
    )

    hybrid = hstack(
        [
            tfidf,
            csr_matrix(emb)
        ],
        format="csr"
    )

    _ = modelo_hybrid_final.predict(
        hybrid
    )

    duracion = (
        time.perf_counter()
        -
        inicio
    )

    latencias_ms.append(
        duracion * 1000
    )


latencias_ms = np.array(
    latencias_ms
)


print("=" * 70)
print("LATENCIA INDIVIDUAL — v1.2")
print("=" * 70)

print(
    f"\nMedia:     "
    f"{latencias_ms.mean():.2f} ms"
)

print(
    f"Mediana:   "
    f"{np.median(latencias_ms):.2f} ms"
)

print(
    f"P90:       "
    f"{np.percentile(latencias_ms, 90):.2f} ms"
)

print(
    f"P95:       "
    f"{np.percentile(latencias_ms, 95):.2f} ms"
)

print(
    f"Máxima:    "
    f"{latencias_ms.max():.2f} ms"
)

LATENCIA INDIVIDUAL — v1.2

Media:     43.28 ms
Mediana:   43.32 ms
P90:       50.95 ms
P95:       51.90 ms
Máxima:    55.93 ms


In [63]:
# ============================================================
# CELDA 61 — OOD CHALLENGE v1
# ============================================================

ood_examples = {

    # ========================================================
    # COCINA
    # ========================================================

    "cocina": {

        "es": [
            "Preparar una tortilla de patatas con cebolla, huevos y aceite de oliva.",
            "Hornear un pastel de chocolate y decorar la superficie con crema.",
            "Cocinar arroz con verduras y servirlo caliente."
        ],

        "en": [
            "Prepare a potato omelette with onions, eggs and olive oil.",
            "Bake a chocolate cake and decorate the top with cream.",
            "Cook rice with vegetables and serve it hot."
        ],

        "ru": [
            "Приготовить картофельный омлет с луком, яйцами и оливковым маслом.",
            "Испечь шоколадный торт и украсить его кремом.",
            "Приготовить рис с овощами и подать горячим."
        ],

        "es_en": [
            "Preparar una potato omelette con cebolla, huevos y olive oil.",
            "Hornear un chocolate cake y decorar la superficie con crema.",
            "Cocinar rice con verduras y servirlo hot."
        ]
    },

    # ========================================================
    # DEPORTES
    # ========================================================

    "deportes": {

        "es": [
            "El equipo marcó dos goles durante la segunda mitad del partido.",
            "La corredora entrenó durante meses para completar el maratón.",
            "El jugador lanzó la pelota desde fuera del área."
        ],

        "en": [
            "The team scored two goals during the second half of the match.",
            "The runner trained for months to complete the marathon.",
            "The player kicked the ball from outside the penalty area."
        ],

        "ru": [
            "Команда забила два гола во втором тайме матча.",
            "Бегунья тренировалась несколько месяцев перед марафоном.",
            "Игрок ударил по мячу из-за пределов штрафной площади."
        ],

        "es_en": [
            "El team marcó dos goals durante la segunda mitad del match.",
            "La runner entrenó durante meses para completar el marathon.",
            "El player lanzó la ball desde fuera del área."
        ]
    },

    # ========================================================
    # VIAJES
    # ========================================================

    "viajes": {

        "es": [
            "Reservar un hotel cerca del centro histórico para tres noches.",
            "El vuelo sale por la mañana y llega a la ciudad por la tarde.",
            "Visitaremos varios museos y monumentos durante las vacaciones."
        ],

        "en": [
            "Book a hotel near the historic center for three nights.",
            "The flight leaves in the morning and arrives in the city in the afternoon.",
            "We will visit several museums and monuments during the vacation."
        ],

        "ru": [
            "Забронировать отель рядом с историческим центром на три ночи.",
            "Самолёт вылетает утром и прибывает в город днём.",
            "Во время отпуска мы посетим несколько музеев и памятников."
        ],

        "es_en": [
            "Reservar un hotel near the historic center por tres noches.",
            "El flight sale por la mañana y arrives por la tarde.",
            "Visitaremos varios museums y monuments durante las vacaciones."
        ]
    },

    # ========================================================
    # HISTORIA
    # ========================================================

    "historia": {

        "es": [
            "El Imperio Romano se extendió por gran parte de Europa y el Mediterráneo.",
            "La Revolución Industrial transformó la producción durante los siglos XVIII y XIX.",
            "La antigua civilización egipcia se desarrolló alrededor del río Nilo."
        ],

        "en": [
            "The Roman Empire expanded across much of Europe and the Mediterranean.",
            "The Industrial Revolution transformed production during the eighteenth and nineteenth centuries.",
            "Ancient Egyptian civilization developed around the Nile River."
        ],

        "ru": [
            "Римская империя занимала значительную часть Европы и Средиземноморья.",
            "Промышленная революция изменила производство в восемнадцатом и девятнадцатом веках.",
            "Древнеегипетская цивилизация развивалась вокруг реки Нил."
        ],

        "es_en": [
            "El Roman Empire se extendió por gran parte de Europe.",
            "La Industrial Revolution transformó la producción durante varios siglos.",
            "La ancient Egyptian civilization se desarrolló alrededor del Nile."
        ]
    },

    # ========================================================
    # MÚSICA
    # ========================================================

    "musica": {

        "es": [
            "La banda interpretó varias canciones durante el concierto.",
            "El pianista practicó la pieza antes de la presentación.",
            "La melodía comienza lentamente y aumenta de intensidad."
        ],

        "en": [
            "The band performed several songs during the concert.",
            "The pianist practiced the piece before the performance.",
            "The melody begins slowly and gradually becomes more intense."
        ],

        "ru": [
            "Группа исполнила несколько песен во время концерта.",
            "Пианист репетировал произведение перед выступлением.",
            "Мелодия начинается медленно и постепенно становится интенсивнее."
        ],

        "es_en": [
            "La band interpretó varias songs durante el concert.",
            "El pianist practicó la piece antes de la presentación.",
            "La melody comienza lentamente y aumenta de intensidad."
        ]
    },

    # ========================================================
    # LITERATURA
    # ========================================================

    "literatura": {

        "es": [
            "La novela cuenta la historia de una familia durante varias generaciones.",
            "El personaje principal abandona su ciudad para comenzar una nueva vida.",
            "El poema describe un paisaje durante una noche de invierno."
        ],

        "en": [
            "The novel tells the story of a family across several generations.",
            "The main character leaves the city to begin a new life.",
            "The poem describes a landscape during a winter night."
        ],

        "ru": [
            "Роман рассказывает историю семьи на протяжении нескольких поколений.",
            "Главный герой покидает город, чтобы начать новую жизнь.",
            "Стихотворение описывает зимний ночной пейзаж."
        ],

        "es_en": [
            "La novel cuenta la historia de una family durante varias generaciones.",
            "El main character abandona su ciudad para comenzar una nueva vida.",
            "El poem describe un paisaje durante una winter night."
        ]
    },

    # ========================================================
    # SALUD GENERAL
    # ========================================================

    "salud_general": {

        "es": [
            "Dormir suficientes horas ayuda a mantener una rutina saludable.",
            "Beber agua regularmente es importante durante los días calurosos.",
            "Caminar diariamente puede formar parte de un estilo de vida activo."
        ],

        "en": [
            "Getting enough sleep helps maintain a healthy daily routine.",
            "Drinking water regularly is important during hot days.",
            "Walking every day can be part of an active lifestyle."
        ],

        "ru": [
            "Достаточный сон помогает поддерживать здоровый распорядок дня.",
            "В жаркую погоду важно регулярно пить воду.",
            "Ежедневные прогулки могут быть частью активного образа жизни."
        ],

        "es_en": [
            "Dormir enough hours ayuda a mantener una healthy routine.",
            "Beber water regularmente es importante durante los hot days.",
            "Caminar every day puede formar parte de un active lifestyle."
        ]
    },

    # ========================================================
    # VIDA COTIDIANA
    # ========================================================

    "vida_cotidiana": {

        "es": [
            "Comprar alimentos para preparar la cena de esta noche.",
            "Organizar la habitación y guardar la ropa en el armario.",
            "Lavar el automóvil durante el fin de semana."
        ],

        "en": [
            "Buy groceries to prepare dinner tonight.",
            "Organize the room and put the clothes in the closet.",
            "Wash the car during the weekend."
        ],

        "ru": [
            "Купить продукты для приготовления сегодняшнего ужина.",
            "Убрать комнату и сложить одежду в шкаф.",
            "Помыть автомобиль в выходные."
        ],

        "es_en": [
            "Comprar groceries para preparar dinner esta noche.",
            "Organizar la room y guardar la clothes en el armario.",
            "Lavar el car durante el weekend."
        ]
    },

    # ========================================================
    # JARDINERÍA
    # ========================================================

    "jardineria": {

        "es": [
            "Regar las plantas del jardín durante las primeras horas de la mañana.",
            "Plantar flores nuevas alrededor de la entrada de la casa.",
            "Podar las ramas secas para mantener el árbol en buenas condiciones."
        ],

        "en": [
            "Water the garden plants during the early morning.",
            "Plant new flowers around the entrance of the house.",
            "Prune dry branches to keep the tree in good condition."
        ],

        "ru": [
            "Поливать растения в саду ранним утром.",
            "Посадить новые цветы возле входа в дом.",
            "Обрезать сухие ветви, чтобы поддерживать дерево в хорошем состоянии."
        ],

        "es_en": [
            "Regar las garden plants durante las primeras horas de la morning.",
            "Plantar new flowers alrededor de la entrada de la house.",
            "Podar dry branches para mantener el tree en buenas condiciones."
        ]
    },

    # ========================================================
    # CINE
    # ========================================================

    "cine": {

        "es": [
            "La película comienza con una escena ambientada en una pequeña ciudad.",
            "El actor interpreta a un detective que investiga un misterio.",
            "El director utilizó diferentes locaciones para filmar la historia."
        ],

        "en": [
            "The movie begins with a scene set in a small town.",
            "The actor plays a detective investigating a mystery.",
            "The director used several locations to film the story."
        ],

        "ru": [
            "Фильм начинается со сцены в небольшом городе.",
            "Актёр играет детектива, расследующего загадочное дело.",
            "Режиссёр использовал несколько мест для съёмок истории."
        ],

        "es_en": [
            "La movie comienza con una scene ambientada en una pequeña ciudad.",
            "El actor interpreta a un detective investigating a mystery.",
            "El director utilizó diferentes locations para filmar la story."
        ]
    }
}

In [64]:
import gc
import psutil

# Liberar memoria no utilizada
gc.collect()

# Mostrar uso de memoria actual
proceso = psutil.Process()
info_memoria = proceso.memory_info()
memoria_mb = info_memoria.rss / 1024 / 1024
print(f"Memoria utilizada: {memoria_mb:.2f} MB")
print(f"Disponible (estimado): {psutil.virtual_memory().available / 1024 / 1024:.2f} MB")

Memoria utilizada: 817.47 MB
Disponible (estimado): 510.38 MB


In [65]:
# ============================================================
# CELDA 62 — DATAFRAME OOD
# ============================================================

filas_ood = []

contador = 1

for dominio, idiomas in ood_examples.items():

    for idioma, textos in idiomas.items():

        for texto in textos:

            filas_ood.append({
                "ood_id":
                    f"ood_{contador:03d}",

                "dominio":
                    dominio,

                "idioma":
                    idioma,

                "texto":
                    texto,

                "domain":
                    "OOD"
            })

            contador += 1


df_ood = pd.DataFrame(
    filas_ood
)


print("=" * 70)
print("OOD CHALLENGE v1")
print("=" * 70)

print("\nDocumentos:")
print(len(df_ood))

print("\nPor idioma:")
print(
    df_ood["idioma"]
    .value_counts()
)

print("\nPor dominio:")
print(
    df_ood["dominio"]
    .value_counts()
)

OOD CHALLENGE v1

Documentos:
120

Por idioma:
idioma
es       30
en       30
ru       30
es_en    30
Name: count, dtype: int64

Por dominio:
dominio
cocina            12
deportes          12
viajes            12
historia          12
musica            12
literatura        12
salud_general     12
vida_cotidiana    12
jardineria        12
cine              12
Name: count, dtype: int64


In [66]:
# ============================================================
# CELDA 63 — EXPORTAR OOD CHALLENGE
# ============================================================

PATH_OOD = (
    ROOT
    / "data"
    / "evaluation"
    / "ood_challenge_v1.csv"
)

PATH_OOD.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_ood.to_csv(
    PATH_OOD,
    index=False,
    encoding="utf-8-sig"
)

print(
    "✅ Guardado:",
    PATH_OOD
)

✅ Guardado: C:\Users\MAMÁ\Downloads\techmind-v2\data\evaluation\ood_challenge_v1.csv


In [67]:
# ============================================================
# CELDA 64 — EMBEDDINGS OOD
# ============================================================

X_ood_embeddings = encoder.encode(
    df_ood["texto"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)


print("=" * 70)
print("EMBEDDINGS OOD")
print("=" * 70)

print(
    "\nShape:",
    X_ood_embeddings.shape
)

Batches: 100%|██████████| 4/4 [00:01<00:00,  3.92it/s]

EMBEDDINGS OOD

Shape: (120, 384)


In [68]:
# ============================================================
# CELDA 65 — DISTANCIA AL DOMINIO
# ============================================================

distancias_ood, indices_nn_ood = (
    domain_index.kneighbors(
        X_ood_embeddings
    )
)

similitudes_ood = (
    1.0
    -
    distancias_ood
)

df_ood[
    "similitud_max_train"
] = (
    similitudes_ood[:, 0]
)

df_ood[
    "similitud_media_5nn"
] = (
    similitudes_ood.mean(
        axis=1
    )
)


print("=" * 70)
print("SIMILITUD — OOD")
print("=" * 70)

display(
    df_ood[
        [
            "similitud_max_train",
            "similitud_media_5nn"
        ]
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

SIMILITUD — OOD


,similitud_max_train,similitud_media_5nn
count,120.000000,120.000000
mean,0.345390,0.299419
std,0.063031,0.050079
min,0.232824,0.225218
1%,0.241498,0.227851
5%,0.251407,0.234621
10%,0.271659,0.244005
25%,0.298393,0.255944
50%,0.342189,0.292141
75%,0.377835,0.328721


In [69]:
# ============================================================
# CELDA 66 — IN-DOMAIN VS OOD
# ============================================================

comparacion_domain = pd.DataFrame({

    "grupo": (
        ["IN_DOMAIN"]
        * len(similitud_max_test)
        +
        ["OOD"]
        * len(df_ood)
    ),

    "similitud_max": np.concatenate([
        similitud_max_test,
        df_ood[
            "similitud_max_train"
        ].to_numpy()
    ]),

    "similitud_5nn": np.concatenate([
        similitud_media5_test,
        df_ood[
            "similitud_media_5nn"
        ].to_numpy()
    ])
})


resumen_domain = (
    comparacion_domain
    .groupby("grupo")
    .agg(
        max_mean=(
            "similitud_max",
            "mean"
        ),
        max_median=(
            "similitud_max",
            "median"
        ),
        nn5_mean=(
            "similitud_5nn",
            "mean"
        ),
        nn5_median=(
            "similitud_5nn",
            "median"
        )
    )
)


print("=" * 70)
print("IN-DOMAIN VS OOD")
print("=" * 70)

display(
    resumen_domain.round(4)
)

IN-DOMAIN VS OOD


,max_mean,max_median,nn5_mean,nn5_median
grupo,,,,
IN_DOMAIN,0.6629,0.6182,0.6073,0.5624
OOD,0.3454,0.3422,0.2994,0.2921


In [70]:
# ============================================================
# CELDA 68 — IN-DOMAIN TRAIN LEAVE-ONE-OUT (PROCESAMIENTO EN CHUNKS)
# ============================================================

from sklearn.neighbors import NearestNeighbors
import numpy as np

N_NEIGHBORS_LOO = 6

# Usar brute pero procesar en chunks para menor uso de memoria
domain_index_loo = NearestNeighbors(
    n_neighbors=N_NEIGHBORS_LOO,
    metric="cosine",
    algorithm="brute",
    n_jobs=1
)

domain_index_loo.fit(
    X_train_embeddings
)

# Procesar en chunks para evitar MemoryError
CHUNK_SIZE = 500
n_samples = X_train_embeddings.shape[0]
n_chunks = (n_samples + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Procesando {n_samples} muestras en {n_chunks} chunks de {CHUNK_SIZE}...")

distancias_train_loo_list = []
indices_train_loo_list = []

for i in range(n_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, n_samples)
    
    chunk = X_train_embeddings[start_idx:end_idx]
    
    dist_chunk, idx_chunk = domain_index_loo.kneighbors(chunk)
    distancias_train_loo_list.append(dist_chunk)
    indices_train_loo_list.append(idx_chunk)
    
    print(f"  Chunk {i+1}/{n_chunks} completado ({end_idx}/{n_samples})")

# Combinar resultados
distancias_train_loo = np.vstack(distancias_train_loo_list)
indices_train_loo = np.vstack(indices_train_loo_list)

similitudes_train_loo = (
    1.0
    -
    distancias_train_loo
)

# Primer vecino = propio documento.
# Lo eliminamos.
similitudes_train_vecinos = (
    similitudes_train_loo[:, 1:]
)

sim_max_train_loo = (
    similitudes_train_vecinos[:, 0]
)

sim_5nn_train_loo = (
    similitudes_train_vecinos.mean(
        axis=1
    )
)

print("=" * 70)
print("IN-DOMAIN TRAIN — LEAVE-ONE-OUT")
print("=" * 70)

df_train_domain_stats = pd.DataFrame({
    "similitud_max": sim_max_train_loo,
    "similitud_5nn": sim_5nn_train_loo
})

display(
    df_train_domain_stats.describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    ).round(4)
)

Procesando 3666 muestras en 8 chunks de 500...
  Chunk 1/8 completado (500/3666)
  Chunk 2/8 completado (1000/3666)
  Chunk 3/8 completado (1500/3666)
  Chunk 4/8 completado (2000/3666)
  Chunk 5/8 completado (2500/3666)
  Chunk 6/8 completado (3000/3666)
  Chunk 7/8 completado (3500/3666)
  Chunk 8/8 completado (3666/3666)
IN-DOMAIN TRAIN — LEAVE-ONE-OUT


,similitud_max,similitud_5nn
count,3666.0000,3666.0000
mean,0.6625,0.6077
std,0.1570,0.1513
min,0.3415,0.3288
1%,0.4203,0.3950
5%,0.4687,0.4388
10%,0.4959,0.4642
25%,0.5464,0.5060
50%,0.6188,0.5635
75%,0.7488,0.6554


In [71]:
# ============================================================
# CELDA 68 — DATASET DE CALIBRACIÓN OOD
# ============================================================

df_domain_calibration = pd.concat(
    [
        pd.DataFrame({
            "grupo": "IN_DOMAIN",
            "similitud_max": sim_max_train_loo,
            "similitud_5nn": sim_5nn_train_loo
        }),

        pd.DataFrame({
            "grupo": "OOD",
            "similitud_max":
                df_ood["similitud_max_train"].to_numpy(),

            "similitud_5nn":
                df_ood["similitud_media_5nn"].to_numpy()
        })
    ],
    ignore_index=True
)

print("=" * 70)
print("CALIBRACIÓN OOD")
print("=" * 70)

display(
    df_domain_calibration
    .groupby("grupo")
    [
        [
            "similitud_max",
            "similitud_5nn"
        ]
    ]
    .describe()
    .round(4)
)

CALIBRACIÓN OOD


similitud_max                                                 \
                  count    mean    std     min     25%     50%     75%   
grupo                                                                    
IN_DOMAIN        3666.0  0.6625  0.157  0.3415  0.5464  0.6188  0.7488   
OOD               120.0  0.3454  0.063  0.2328  0.2984  0.3422  0.3778   

                  similitud_5nn                                          \
              max         count    mean     std     min     25%     50%   
grupo                                                                     
IN_DOMAIN  0.9981        3666.0  0.6077  0.1513  0.3288  0.5060  0.5635   
OOD        0.5304         120.0  0.2994  0.0501  0.2252  0.2559  0.2921   

                           
              75%     max  
grupo                      
IN_DOMAIN  0.6554  0.9961  
OOD        0.3287  0.4263

In [72]:
# ============================================================
# CELDA 69 — BÚSQUEDA DE UMBRAL OOD 5NN
# ============================================================

thresholds_ood = np.linspace(
    0.20,
    0.65,
    1000
)

resultados_ood_threshold = []


for threshold in thresholds_ood:

    # IN_DOMAIN si similitud >= threshold
    in_domain_retained = (
        sim_5nn_train_loo
        >= threshold
    )

    # OOD detectado si similitud < threshold
    ood_detected = (
        df_ood[
            "similitud_media_5nn"
        ].to_numpy()
        < threshold
    )

    recall_in_domain = (
        in_domain_retained.mean()
    )

    recall_ood = (
        ood_detected.mean()
    )

    balanced_accuracy = (
        recall_in_domain
        +
        recall_ood
    ) / 2

    resultados_ood_threshold.append({
        "threshold":
            threshold,

        "in_domain_retention":
            recall_in_domain,

        "ood_detection":
            recall_ood,

        "balanced_accuracy":
            balanced_accuracy
    })


df_ood_thresholds = pd.DataFrame(
    resultados_ood_threshold
)

In [73]:
# ============================================================
# CELDA 70 — CANDIDATOS OOD
# ============================================================

top_ood = (
    df_ood_thresholds
    .sort_values(
        [
            "balanced_accuracy",
            "in_domain_retention"
        ],
        ascending=False
    )
    .head(20)
)

print("=" * 70)
print("MEJORES UMBRALES OOD")
print("=" * 70)

display(
    top_ood.round(4)
)

MEJORES UMBRALES OOD


,threshold,in_domain_retention,ood_detection,balanced_accuracy
503,0.4266,0.9681,1.0000,0.9840
504,0.4270,0.9675,1.0000,0.9838
505,0.4275,0.9667,1.0000,0.9834
506,0.4279,0.9664,1.0000,0.9832
507,0.4284,0.9651,1.0000,0.9825
508,0.4288,0.9648,1.0000,0.9824
509,0.4293,0.9643,1.0000,0.9821
494,0.4225,0.9724,0.9917,0.9821
495,0.4230,0.9724,0.9917,0.9821
510,0.4297,0.9640,1.0000,0.9820


In [74]:
objetivos_ood = [
    0.95,
    0.97,
    0.99,
    1.00
]

filas_ood_operacional = []

for objetivo in objetivos_ood:

    candidatos = (
        df_ood_thresholds[
            df_ood_thresholds[
                "ood_detection"
            ]
            >= objetivo
        ]
        .sort_values(
            [
                "in_domain_retention",
                "threshold"
            ],
            ascending=[
                False,
                True
            ]
        )
    )

    if len(candidatos) == 0:
        continue

    mejor = candidatos.iloc[0]

    filas_ood_operacional.append({
        "target_ood_detection":
            objetivo,

        "threshold":
            mejor["threshold"],

        "in_domain_retention":
            mejor["in_domain_retention"],

        "ood_detection":
            mejor["ood_detection"],

        "balanced_accuracy":
            mejor["balanced_accuracy"]
    })


df_ood_operacional = pd.DataFrame(
    filas_ood_operacional
)

print("\n" + "=" * 70)
print("UMBRALES SEGÚN OBJETIVO OOD")
print("=" * 70)

display(
    df_ood_operacional.round(4)
)


UMBRALES SEGÚN OBJETIVO OOD


,target_ood_detection,threshold,in_domain_retention,ood_detection,balanced_accuracy
0,0.95,0.4027,0.9861,0.9500,0.9680
1,0.97,0.4171,0.9776,0.9833,0.9805
2,0.99,0.4225,0.9724,0.9917,0.9821
3,1.00,0.4266,0.9681,1.0000,0.9840


In [75]:
# ============================================================
# CELDA 71 — CONGELAR DETECTOR OOD v1.2
# ============================================================

UMBRAL_OOD_5NN_V12 = 0.4266

OOD_CALIBRATION = {
    "threshold_5nn":
        UMBRAL_OOD_5NN_V12,

    "metric":
        "mean_cosine_similarity_5nn",

    "reference":
        "TRAIN embeddings",

    "in_domain_calibration":
        "TRAIN leave-one-out",

    "ood_calibration":
        "ood_challenge_v1",

    "in_domain_retention":
        0.9681,

    "ood_detection":
        1.0000,

    "balanced_accuracy":
        0.9840
}


print("=" * 70)
print("DETECTOR OOD — CONFIGURACIÓN CONGELADA")
print("=" * 70)

print(
    f"\nUmbral 5NN:            "
    f"{UMBRAL_OOD_5NN_V12:.4f}"
)

print(
    f"Retención IN-DOMAIN:  "
    f"{OOD_CALIBRATION['in_domain_retention']:.2%}"
)

print(
    f"Detección OOD dev:    "
    f"{OOD_CALIBRATION['ood_detection']:.2%}"
)

print(
    f"Balanced accuracy:    "
    f"{OOD_CALIBRATION['balanced_accuracy']:.2%}"
)

print(
    "\n🔒 No reajustar usando TEST "
    "o benchmark final."
)

DETECTOR OOD — CONFIGURACIÓN CONGELADA

Umbral 5NN:            0.4266
Retención IN-DOMAIN:  96.81%
Detección OOD dev:    100.00%
Balanced accuracy:    98.40%

🔒 No reajustar usando TEST o benchmark final.


In [76]:
# ============================================================
# CELDA 72 — OOD DETECTOR SOBRE TEST
# ============================================================

df_test_operacional_v12[
    "ood_similarity_5nn"
] = similitud_media5_test

df_test_operacional_v12[
    "es_ood"
] = (
    df_test_operacional_v12[
        "ood_similarity_5nn"
    ]
    <
    UMBRAL_OOD_5NN_V12
)


n_ood_test = int(
    df_test_operacional_v12[
        "es_ood"
    ].sum()
)

retencion_test = (
    1
    -
    n_ood_test
    / len(df_test_operacional_v12)
)


print("=" * 70)
print("OOD DETECTOR — TEST IN-DOMAIN")
print("=" * 70)

print(
    f"\nDocumentos TEST:       "
    f"{len(df_test_operacional_v12)}"
)

print(
    f"Marcados como OOD:     "
    f"{n_ood_test}"
)

print(
    f"Conservados:           "
    f"{len(df_test_operacional_v12) - n_ood_test}"
)

print(
    f"\nRetención IN-DOMAIN:   "
    f"{retencion_test:.2%}"
)

OOD DETECTOR — TEST IN-DOMAIN

Documentos TEST:       917
Marcados como OOD:     30
Conservados:           887

Retención IN-DOMAIN:   96.73%


In [77]:
# ============================================================
# CELDA 73 — FALSE OOD SOBRE TEST
# ============================================================

df_false_ood_test = (
    df_test_operacional_v12[
        df_test_operacional_v12[
            "es_ood"
        ]
    ]
    .copy()
)


print("=" * 70)
print("TEST VÁLIDO MARCADO COMO OOD")
print("=" * 70)

print(
    "\nTotal:",
    len(df_false_ood_test)
)

display(
    df_false_ood_test[
        [
            "categoria_real",
            "categoria_predicha",
            "correcta",
            "margen",
            "ood_similarity_5nn"
        ]
    ]
    .sort_values(
        "ood_similarity_5nn"
    )
    .head(30)
)

TEST VÁLIDO MARCADO COMO OOD

Total: 30


,categoria_real,categoria_predicha,correcta,margen,ood_similarity_5nn
426,cloud,datascience,False,0.515189,0.348859
190,datascience,datascience,True,0.024373,0.350619
681,frontend,frontend,True,0.095928,0.359268
417,datascience,datascience,True,1.110783,0.366234
583,cloud,datascience,False,0.350395,0.370192
551,backend,backend,True,1.159247,0.376261
651,cloud,backend,False,0.122101,0.388623
657,datascience,cloud,False,0.167588,0.389993
708,datascience,datascience,True,0.182905,0.394339
368,frontend,frontend,True,0.873465,0.394487


In [78]:
# ============================================================
# CELDA 74 — OOD SOBRE BENCHMARK MULTILINGÜE
# ============================================================

dist_multi_domain, idx_multi_domain = (
    domain_index.kneighbors(
        X_multi_embeddings
    )
)

sim_multi_domain = (
    1.0
    -
    dist_multi_domain
)

df_multi_v12[
    "ood_similarity_5nn"
] = (
    sim_multi_domain.mean(
        axis=1
    )
)

df_multi_v12[
    "es_ood_v12"
] = (
    df_multi_v12[
        "ood_similarity_5nn"
    ]
    <
    UMBRAL_OOD_5NN_V12
)


print("=" * 70)
print("OOD — MULTILINGUAL DEVELOPMENT")
print("=" * 70)

resumen_ood_multi = (
    df_multi_v12
    .groupby("idioma")
    .agg(
        documentos=(
            "es_ood_v12",
            "size"
        ),
        ood_detectados=(
            "es_ood_v12",
            "sum"
        ),
        similarity_mean=(
            "ood_similarity_5nn",
            "mean"
        ),
        similarity_min=(
            "ood_similarity_5nn",
            "min"
        )
    )
    .reset_index()
)

resumen_ood_multi[
    "retencion"
] = (
    1
    -
    resumen_ood_multi[
        "ood_detectados"
    ]
    /
    resumen_ood_multi[
        "documentos"
    ]
)

display(
    resumen_ood_multi.round(4)
)

OOD — MULTILINGUAL DEVELOPMENT


,idioma,documentos,ood_detectados,similarity_mean,similarity_min,retencion
0,en,20,0,0.5252,0.4493,1.0
1,es,20,0,0.5245,0.4424,1.0
2,es_en,20,0,0.5269,0.4508,1.0
3,ru,20,0,0.5236,0.4571,1.0


In [79]:
# ============================================================
# CELDA 75 — REGLA OPERACIONAL v1.2
# ============================================================

def asignar_estado_v12(
    similarity_5nn,
    margen,
    valid_input=True
):

    if not valid_input:
        return "rechazada"

    if (
        similarity_5nn
        <
        UMBRAL_OOD_5NN_V12
    ):
        return "rechazada_ood"

    if (
        margen
        <
        UMBRAL_MARGEN_REVISION_V12
    ):
        return "revision"

    return "aceptada"

In [80]:
df_test_operacional_v12[
    "estado_final_v12"
] = [

    asignar_estado_v12(
        similarity_5nn=sim,
        margen=margen
    )

    for sim, margen in zip(
        df_test_operacional_v12[
            "ood_similarity_5nn"
        ],
        df_test_operacional_v12[
            "margen"
        ]
    )
]

In [81]:
# ============================================================
# CELDA 76 — OPERACIÓN COMPLETA EN TEST
# ============================================================

aceptadas_final = (
    df_test_operacional_v12[
        "estado_final_v12"
    ]
    ==
    "aceptada"
)

revision_final = (
    df_test_operacional_v12[
        "estado_final_v12"
    ]
    ==
    "revision"
)

rechazada_ood_final = (
    df_test_operacional_v12[
        "estado_final_v12"
    ]
    ==
    "rechazada_ood"
)


n_aceptadas_final = int(
    aceptadas_final.sum()
)

accuracy_aceptadas_final = (
    df_test_operacional_v12.loc[
        aceptadas_final,
        "correcta"
    ]
    .mean()
)


errores_totales = int(
    (
        ~df_test_operacional_v12[
            "correcta"
        ]
    ).sum()
)


errores_aceptados_final = int(
    (
        aceptadas_final
        &
        ~df_test_operacional_v12[
            "correcta"
        ]
    ).sum()
)


error_capture_final = (
    (
        errores_totales
        -
        errores_aceptados_final
    )
    /
    errores_totales
)


print("=" * 70)
print("OPERACIÓN COMPLETA — TEST")
print("=" * 70)

print(
    f"\nAceptadas:       "
    f"{aceptadas_final.sum()}"
)

print(
    f"Revisión:        "
    f"{revision_final.sum()}"
)

print(
    f"Rechazadas OOD:  "
    f"{rechazada_ood_final.sum()}"
)

print(
    f"\nCobertura automática: "
    f"{n_aceptadas_final / len(df_test_operacional_v12):.2%}"
)

print(
    f"Accuracy aceptadas:   "
    f"{accuracy_aceptadas_final:.2%}"
)

print(
    f"Error capture:        "
    f"{error_capture_final:.2%}"
)

print(
    f"Errores aceptados:    "
    f"{errores_aceptados_final}"
)

OPERACIÓN COMPLETA — TEST

Aceptadas:       618
Revisión:        269
Rechazadas OOD:  30

Cobertura automática: 67.39%
Accuracy aceptadas:   96.60%
Error capture:        81.74%
Errores aceptados:    21


In [82]:
# ============================================================
# CELDA 77 — CONFIGURACIÓN OPERACIONAL FINAL CANDIDATA
# ============================================================

TECHMIND_V12_OPERATIONAL_CONFIG = {

    "version":
        "1.2.0-multilingual",

    "status":
        "frozen_candidate",

    "architecture": {
        "tfidf":
            "TechMind v1.1.0 Word+Char",

        "embedding_model":
            "sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2",

        "embedding_dimension":
            384,

        "embedding_normalization":
            True,

        "classifier":
            "LinearSVC",

        "classifier_C":
            0.3
    },

    "ood_control": {
        "metric":
            "mean_cosine_similarity_5nn",

        "threshold":
            0.4266,

        "reference":
            "train_embeddings",

        "calibration":
            "TRAIN leave-one-out + OOD challenge v1"
    },

    "confidence_control": {
        "metric":
            "top1_minus_top2_decision_margin",

        "threshold":
            0.8132,

        "calibration":
            "5-fold OOF TRAIN"
    },

    "decision_order": [
        "input_validation",
        "ood_detection",
        "classification",
        "decision_margin"
    ]
}


print("=" * 70)
print("TECHMIND v1.2 — FROZEN CANDIDATE")
print("=" * 70)

print(
    json.dumps(
        TECHMIND_V12_OPERATIONAL_CONFIG,
        indent=2,
        ensure_ascii=False
    )
)

TECHMIND v1.2 — FROZEN CANDIDATE
{
  "version": "1.2.0-multilingual",
  "status": "frozen_candidate",
  "architecture": {
    "tfidf": "TechMind v1.1.0 Word+Char",
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_dimension": 384,
    "embedding_normalization": true,
    "classifier": "LinearSVC",
    "classifier_C": 0.3
  },
  "ood_control": {
    "metric": "mean_cosine_similarity_5nn",
    "threshold": 0.4266,
    "reference": "train_embeddings",
    "calibration": "TRAIN leave-one-out + OOD challenge v1"
  },
  "confidence_control": {
    "metric": "top1_minus_top2_decision_margin",
    "threshold": 0.8132,
    "calibration": "5-fold OOF TRAIN"
  },
  "decision_order": [
    "input_validation",
    "ood_detection",
    "classification",
    "decision_margin"
  ]
}


In [83]:
# ============================================================
# CELDA 78 — APORTE INCREMENTAL DEL CONTROL OOD
# ============================================================

margin_accept = (
    df_test_operacional_v12["margen"]
    >=
    UMBRAL_MARGEN_REVISION_V12
)

ood_reject = (
    df_test_operacional_v12[
        "ood_similarity_5nn"
    ]
    <
    UMBRAL_OOD_5NN_V12
)


ood_total = int(
    ood_reject.sum()
)

ood_correctas = int(
    (
        ood_reject
        &
        df_test_operacional_v12["correcta"]
    ).sum()
)

ood_errores = int(
    (
        ood_reject
        &
        ~df_test_operacional_v12["correcta"]
    ).sum()
)


# Eran aceptadas por margen pero OOD las bloquea
ood_bloquea_aceptadas = (
    ood_reject
    &
    margin_accept
)

bloqueadas_total = int(
    ood_bloquea_aceptadas.sum()
)

bloqueadas_correctas = int(
    (
        ood_bloquea_aceptadas
        &
        df_test_operacional_v12["correcta"]
    ).sum()
)

bloqueadas_errores = int(
    (
        ood_bloquea_aceptadas
        &
        ~df_test_operacional_v12["correcta"]
    ).sum()
)


print("=" * 70)
print("APORTE INCREMENTAL DEL DETECTOR OOD")
print("=" * 70)

print(f"\nTEST marcados OOD:             {ood_total}")
print(f"  Correctos:                   {ood_correctas}")
print(f"  Incorrectos:                 {ood_errores}")

print(
    "\nDe esos, margen habría aceptado:"
)

print(
    f"  Total:                       "
    f"{bloqueadas_total}"
)

print(
    f"  Predicciones correctas:      "
    f"{bloqueadas_correctas}"
)

print(
    f"  Errores evitados:            "
    f"{bloqueadas_errores}"
)

APORTE INCREMENTAL DEL DETECTOR OOD

TEST marcados OOD:             30
  Correctos:                   16
  Incorrectos:                 14

De esos, margen habría aceptado:
  Total:                       9
  Predicciones correctas:      7
  Errores evitados:            2


In [84]:
# ============================================================
# CELDA 79 — RESUMEN MAESTRO v1.2
# ============================================================

resumen_v12 = pd.DataFrame({

    "metrica": [

        "F1 Macro CV",
        "Accuracy Test",
        "F1 Macro Test",

        "Multilingual Accuracy",
        "Cross-language consistency",

        "Margin-only coverage Test",
        "Margin-only accepted accuracy",
        "Margin-only error capture",

        "Final coverage Test",
        "Final accepted accuracy",
        "Final error capture",

        "OOD IN-domain retention Test",
        "OOD dev detection",

        "Latency individual median ms",
        "Latency individual P95 ms"
    ],

    "valor": [

        0.8574,
        0.8746,
        0.8753,

        0.9875,
        0.95,

        0.6838,
        0.9633,
        0.80,

        0.6739,
        0.9660,
        0.8174,

        0.9673,
        1.00,

        46.61,
        152.38
    ]
})


display(resumen_v12)

,metrica,valor
0,F1 Macro CV,0.8574
1,Accuracy Test,0.8746
2,F1 Macro Test,0.8753
3,Multilingual Accuracy,0.9875
4,Cross-language consistency,0.9500
5,Margin-only coverage Test,0.6838
6,Margin-only accepted accuracy,0.9633
7,Margin-only error capture,0.8000
8,Final coverage Test,0.6739
9,Final accepted accuracy,0.9660


In [85]:
# ============================================================
# CELDA 80 — CARGAR BENCHMARK FINAL INDEPENDIENTE
# ============================================================

PATH_FINAL_BENCHMARK = (
    ROOT
    / "data"
    / "evaluation"
    / "multilingual_final_benchmark_v1.csv"
)

df_final_benchmark = pd.read_csv(
    PATH_FINAL_BENCHMARK
)

print("=" * 70)
print("FINAL MULTILINGUAL BENCHMARK — v1")
print("=" * 70)

print("\nArchivo:")
print(PATH_FINAL_BENCHMARK)

print("\nFilas:")
print(len(df_final_benchmark))

print("\nCasos semánticos:")
print(
    df_final_benchmark[
        "case_id"
    ].nunique()
)

print("\nIdiomas:")
print(
    df_final_benchmark[
        "idioma"
    ].value_counts()
)

print("\nCategorías:")
print(
    df_final_benchmark[
        "categoria_real"
    ].value_counts()
)

print("\nDificultad por casos:")

display(
    df_final_benchmark[
        [
            "case_id",
            "dificultad"
        ]
    ]
    .drop_duplicates()
    ["dificultad"]
    .value_counts()
)

FINAL MULTILINGUAL BENCHMARK — v1

Archivo:
C:\Users\MAMÁ\Downloads\techmind-v2\data\evaluation\multilingual_final_benchmark_v1.csv

Filas:
320

Casos semánticos:
80

Idiomas:
idioma
es       80
en       80
ru       80
es_en    80
Name: count, dtype: int64

Categorías:
categoria_real
backend        80
cloud          80
datascience    80
frontend       80
Name: count, dtype: int64

Dificultad por casos:


dificultad
medium       40
easy         20
difficult    20
Name: count, dtype: int64

In [86]:
# ============================================================
# CELDA 81 — VALIDACIÓN BENCHMARK FINAL
# ============================================================

validaciones_final = {

    "320_filas":
        len(df_final_benchmark) == 320,

    "80_casos":
        df_final_benchmark[
            "case_id"
        ].nunique() == 80,

    "4_idiomas":
        df_final_benchmark[
            "idioma"
        ].nunique() == 4,

    "4_categorias":
        df_final_benchmark[
            "categoria_real"
        ].nunique() == 4,

    "sin_textos_vacios":
        (
            df_final_benchmark[
                "texto"
            ]
            .fillna("")
            .str.strip()
            .ne("")
            .all()
        ),

    "sin_case_language_duplicados":
        not df_final_benchmark.duplicated(
            subset=[
                "case_id",
                "idioma"
            ]
        ).any(),

    "20_casos_por_categoria":
        (
            df_final_benchmark[
                [
                    "case_id",
                    "categoria_real"
                ]
            ]
            .drop_duplicates()
            ["categoria_real"]
            .value_counts()
            .eq(20)
            .all()
        ),

    "80_por_idioma":
        (
            df_final_benchmark[
                "idioma"
            ]
            .value_counts()
            .eq(80)
            .all()
        )
}


print("=" * 70)
print("VALIDACIÓN BENCHMARK FINAL")
print("=" * 70)

for nombre, valor in validaciones_final.items():

    print(
        f"{'✅' if valor else '❌'} "
        f"{nombre}: {valor}"
    )


FINAL_BENCHMARK_VALID = all(
    validaciones_final.values()
)

print("\n" + "-" * 70)

print(
    "BENCHMARK LISTO:",
    FINAL_BENCHMARK_VALID
)

if not FINAL_BENCHMARK_VALID:

    raise RuntimeError(
        "El benchmark final "
        "no pasó las validaciones."
    )

VALIDACIÓN BENCHMARK FINAL
✅ 320_filas: True
✅ 80_casos: True
✅ 4_idiomas: True
✅ 4_categorias: True
✅ sin_textos_vacios: True
✅ sin_case_language_duplicados: True
✅ 20_casos_por_categoria: True
✅ 80_por_idioma: True

----------------------------------------------------------------------
BENCHMARK LISTO: True


In [87]:
# ============================================================
# CELDA 82 — INTEGRIDAD DEL BENCHMARK FINAL
# ============================================================

import hashlib

EXPECTED_FINAL_BENCHMARK_SHA256 = (
    "5d994e36a49b362fa4277656e7c4254bfb7fa9bab00db3b9f194228c115e13f8"
)

sha256 = hashlib.sha256()

with open(
    PATH_FINAL_BENCHMARK,
    "rb"
) as f:

    for bloque in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha256.update(bloque)

FINAL_BENCHMARK_SHA256 = sha256.hexdigest()

SHA_OK = (
    FINAL_BENCHMARK_SHA256
    ==
    EXPECTED_FINAL_BENCHMARK_SHA256
)

print("=" * 70)
print("INTEGRIDAD — BENCHMARK FINAL")
print("=" * 70)

print("\nEsperado:")
print(EXPECTED_FINAL_BENCHMARK_SHA256)

print("\nCalculado:")
print(FINAL_BENCHMARK_SHA256)

print(
    "\nSHA256 correcto:",
    SHA_OK
)

if not SHA_OK:
    raise RuntimeError(
        "El benchmark final no coincide "
        "con la versión congelada."
    )

INTEGRIDAD — BENCHMARK FINAL

Esperado:
5d994e36a49b362fa4277656e7c4254bfb7fa9bab00db3b9f194228c115e13f8

Calculado:
5d994e36a49b362fa4277656e7c4254bfb7fa9bab00db3b9f194228c115e13f8

SHA256 correcto: True


In [88]:
# ============================================================
# CELDA 83 — EMBEDDINGS BENCHMARK FINAL
# ============================================================

textos_final = (
    df_final_benchmark["texto"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_final_embeddings = encoder.encode(
    textos_final,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("=" * 70)
print("EMBEDDINGS — BENCHMARK FINAL")
print("=" * 70)

print(
    "\nShape:",
    X_final_embeddings.shape
)

print(
    "NaN:",
    np.isnan(
        X_final_embeddings
    ).sum()
)

print(
    "Inf:",
    np.isinf(
        X_final_embeddings
    ).sum()
)

Batches: 100%|██████████| 10/10 [00:03<00:00,  2.62it/s]

EMBEDDINGS — BENCHMARK FINAL

Shape: (320, 384)
NaN: 0
Inf: 0


In [89]:
# ============================================================
# CELDA 84 — TF-IDF CONGELADO
# ============================================================

X_final_tfidf = (
    features_hybrid_final
    .transform(
        textos_final
    )
)

print("=" * 70)
print("TF-IDF — BENCHMARK FINAL")
print("=" * 70)

print(
    "\nShape:",
    X_final_tfidf.shape
)

TF-IDF — BENCHMARK FINAL

Shape: (320, 60000)


In [90]:
# ============================================================
# CELDA 85 — REPRESENTACIÓN HÍBRIDA FINAL
# ============================================================

from scipy.sparse import (
    csr_matrix,
    hstack
)

X_final_hybrid = hstack(
    [
        X_final_tfidf,
        csr_matrix(
            X_final_embeddings
        )
    ],
    format="csr"
)

print("=" * 70)
print("HYBRID FEATURES — BENCHMARK FINAL")
print("=" * 70)

print(
    "\nShape:",
    X_final_hybrid.shape
)

print(
    "Esperadas:",
    modelo_hybrid_final.n_features_in_
)

assert (
    X_final_hybrid.shape[1]
    ==
    modelo_hybrid_final.n_features_in_
)

HYBRID FEATURES — BENCHMARK FINAL

Shape: (320, 60384)
Esperadas: 60384


In [91]:
# ============================================================
# CELDA 86 — PREDICCIONES RAW
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

pred_final = (
    modelo_hybrid_final
    .predict(
        X_final_hybrid
    )
)

df_final_resultados = (
    df_final_benchmark.copy()
)

df_final_resultados[
    "categoria_predicha"
] = pred_final

df_final_resultados[
    "correcta"
] = (
    df_final_resultados[
        "categoria_real"
    ]
    ==
    df_final_resultados[
        "categoria_predicha"
    ]
)


accuracy_final = accuracy_score(
    df_final_resultados[
        "categoria_real"
    ],
    df_final_resultados[
        "categoria_predicha"
    ]
)

precision_macro_final = precision_score(
    df_final_resultados[
        "categoria_real"
    ],
    df_final_resultados[
        "categoria_predicha"
    ],
    average="macro",
    zero_division=0
)

recall_macro_final = recall_score(
    df_final_resultados[
        "categoria_real"
    ],
    df_final_resultados[
        "categoria_predicha"
    ],
    average="macro",
    zero_division=0
)

f1_macro_final = f1_score(
    df_final_resultados[
        "categoria_real"
    ],
    df_final_resultados[
        "categoria_predicha"
    ],
    average="macro",
    zero_division=0
)

f1_weighted_final = f1_score(
    df_final_resultados[
        "categoria_real"
    ],
    df_final_resultados[
        "categoria_predicha"
    ],
    average="weighted",
    zero_division=0
)


print("=" * 70)
print("TECHMIND v1.2 — FINAL INDEPENDENT BENCHMARK")
print("=" * 70)

print(
    f"\nCorrectas:        "
    f"{df_final_resultados['correcta'].sum()}"
    f"/{len(df_final_resultados)}"
)

print(
    f"Accuracy:         "
    f"{accuracy_final:.4f}"
)

print(
    f"Precision Macro:  "
    f"{precision_macro_final:.4f}"
)

print(
    f"Recall Macro:     "
    f"{recall_macro_final:.4f}"
)

print(
    f"F1 Macro:         "
    f"{f1_macro_final:.4f}"
)

print(
    f"F1 Weighted:      "
    f"{f1_weighted_final:.4f}"
)

TECHMIND v1.2 — FINAL INDEPENDENT BENCHMARK

Correctas:        244/320
Accuracy:         0.7625
Precision Macro:  0.8167
Recall Macro:     0.7625
F1 Macro:         0.7570
F1 Weighted:      0.7570


In [92]:
# ============================================================
# CELDA 87 — RESULTADOS POR IDIOMA
# ============================================================

metricas_idioma_final = []

for idioma, grupo in (
    df_final_resultados
    .groupby("idioma")
):

    y_true = grupo[
        "categoria_real"
    ]

    y_pred = grupo[
        "categoria_predicha"
    ]

    metricas_idioma_final.append({

        "idioma":
            idioma,

        "documentos":
            len(grupo),

        "correctas":
            int(
                grupo[
                    "correcta"
                ].sum()
            ),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision_macro":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "recall_macro":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "f1_macro":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
    })


df_metricas_idioma_final = (
    pd.DataFrame(
        metricas_idioma_final
    )
)


print("=" * 70)
print("FINAL BENCHMARK — MÉTRICAS POR IDIOMA")
print("=" * 70)

display(
    df_metricas_idioma_final
    .round(4)
)

FINAL BENCHMARK — MÉTRICAS POR IDIOMA


,idioma,documentos,correctas,accuracy,precision_macro,recall_macro,f1_macro
0,en,80,62,0.7750,0.8262,0.7750,0.7718
1,es,80,60,0.7500,0.7933,0.7500,0.7398
2,es_en,80,63,0.7875,0.8416,0.7875,0.7892
3,ru,80,59,0.7375,0.8034,0.7375,0.7235


In [93]:
# ============================================================
# CELDA 88 — IDIOMA × CATEGORÍA
# ============================================================

accuracy_final_lang_cat = (
    df_final_resultados
    .groupby([
        "idioma",
        "categoria_real"
    ])[
        "correcta"
    ]
    .mean()
    .unstack()
)

print("=" * 70)
print("FINAL BENCHMARK — ACCURACY IDIOMA × CATEGORÍA")
print("=" * 70)

display(
    accuracy_final_lang_cat
    .round(4)
)

FINAL BENCHMARK — ACCURACY IDIOMA × CATEGORÍA


categoria_real,backend,cloud,datascience,frontend
idioma,,,,
en,0.9,0.45,0.85,0.90
es,0.9,0.35,0.85,0.90
es_en,0.9,0.50,0.85,0.90
ru,0.9,0.30,0.80,0.95


In [94]:
# ============================================================
# CELDA 89 — RESULTADOS POR CATEGORÍA
# ============================================================

from sklearn.metrics import (
    classification_report
)

reporte_final_dict = (
    classification_report(
        df_final_resultados[
            "categoria_real"
        ],
        df_final_resultados[
            "categoria_predicha"
        ],
        output_dict=True,
        zero_division=0
    )
)

df_reporte_final = (
    pd.DataFrame(
        reporte_final_dict
    )
    .T
)

print("=" * 70)
print("FINAL BENCHMARK — POR CATEGORÍA")
print("=" * 70)

display(
    df_reporte_final
    .round(4)
)

FINAL BENCHMARK — POR CATEGORÍA


,precision,recall,f1-score,support
backend,0.5455,0.9000,0.6792,80.0000
cloud,0.8649,0.4000,0.5470,80.0000
datascience,0.8701,0.8375,0.8535,80.0000
frontend,0.9865,0.9125,0.9481,80.0000
accuracy,0.7625,0.7625,0.7625,0.7625
macro avg,0.8167,0.7625,0.7570,320.0000
weighted avg,0.8167,0.7625,0.7570,320.0000


In [95]:
# ============================================================
# CELDA 90 — CROSS-LANGUAGE CONSISTENCY
# ============================================================

pivot_final = (
    df_final_resultados
    .pivot(
        index="case_id",
        columns="idioma",
        values="categoria_predicha"
    )
)

IDIOMAS_FINAL = [
    "es",
    "en",
    "ru",
    "es_en"
]

pivot_final[
    "n_predicciones"
] = (
    pivot_final[
        IDIOMAS_FINAL
    ]
    .nunique(
        axis=1
    )
)

pivot_final[
    "consistente"
] = (
    pivot_final[
        "n_predicciones"
    ]
    == 1
)

casos_consistentes_final = int(
    pivot_final[
        "consistente"
    ]
    .sum()
)

consistencia_final = (
    pivot_final[
        "consistente"
    ]
    .mean()
)


print("=" * 70)
print("FINAL BENCHMARK — CROSS-LANGUAGE CONSISTENCY")
print("=" * 70)

print(
    f"\nCasos consistentes: "
    f"{casos_consistentes_final}/"
    f"{len(pivot_final)}"
)

print(
    f"Consistencia:       "
    f"{consistencia_final:.2%}"
)

FINAL BENCHMARK — CROSS-LANGUAGE CONSISTENCY

Casos consistentes: 64/80
Consistencia:       80.00%


In [96]:
# ============================================================
# CELDA 91 — ERRORES DEL BENCHMARK FINAL
# ============================================================

df_errores_final = (
    df_final_resultados[
        ~df_final_resultados[
            "correcta"
        ]
    ]
    .copy()
)


print("=" * 70)
print("ERRORES — FINAL BENCHMARK")
print("=" * 70)

print(
    "\nTotal:",
    len(df_errores_final)
)

display(
    df_errores_final[
        [
            "case_id",
            "semantic_family",
            "idioma",
            "categoria_real",
            "categoria_predicha",
            "dificultad",
            "texto"
        ]
    ]
)

ERRORES — FINAL BENCHMARK

Total: 76


,case_id,semantic_family,idioma,categoria_real,categoria_predicha,dificultad,texto
46,backend_final_012,cache_invalidation,ru,backend,cloud,difficult,Инвалидировать распределённый кэш при изменени...
52,backend_final_014,rbac_authorization,es,backend,cloud,easy,Comprobar roles y permisos antes de permitir q...
53,backend_final_014,rbac_authorization,en,backend,cloud,easy,Check roles and permissions before allowing a ...
55,backend_final_014,rbac_authorization,es_en,backend,cloud,easy,Comprobar roles y permissions antes de permiti...
56,backend_final_015,audit_logging,es,backend,datascience,medium,"Registrar quién modificó cada recurso, qué cam..."
...,...,...,...,...,...,...,...
288,frontend_final_013,accessible_drag_drop,es,frontend,cloud,difficult,Permitir reordenar tarjetas con arrastrar y so...
296,frontend_final_015,debounced_search,es,frontend,backend,easy,Esperar unos milisegundos después de que el us...
297,frontend_final_015,debounced_search,en,frontend,backend,easy,Wait a few milliseconds after the user stops t...
298,frontend_final_015,debounced_search,ru,frontend,backend,easy,Подождать несколько миллисекунд после окончани...


In [97]:
# ============================================================
# CELDA 92 — OOD FINAL
# ============================================================

dist_final_domain, idx_final_domain = (
    domain_index.kneighbors(
        X_final_embeddings
    )
)

sim_final_domain = (
    1.0
    -
    dist_final_domain
)

df_final_resultados[
    "ood_similarity_5nn"
] = (
    sim_final_domain.mean(
        axis=1
    )
)

df_final_resultados[
    "es_ood"
] = (
    df_final_resultados[
        "ood_similarity_5nn"
    ]
    <
    UMBRAL_OOD_5NN_V12
)


print("=" * 70)
print("OOD — FINAL BENCHMARK")
print("=" * 70)

print(
    "\nMarcados OOD:",
    int(
        df_final_resultados[
            "es_ood"
        ].sum()
    )
)

print(
    "Retención:",
    f"{1 - df_final_resultados['es_ood'].mean():.2%}"
)


display(
    df_final_resultados
    .groupby("idioma")
    .agg(
        documentos=(
            "es_ood",
            "size"
        ),
        ood=(
            "es_ood",
            "sum"
        ),
        similarity_mean=(
            "ood_similarity_5nn",
            "mean"
        ),
        similarity_min=(
            "ood_similarity_5nn",
            "min"
        )
    )
    .round(4)
)

OOD — FINAL BENCHMARK

Marcados OOD: 40
Retención: 87.50%


,documentos,ood,similarity_mean,similarity_min
idioma,,,,
en,80,15,0.4800,0.3607
es,80,9,0.4878,0.3710
es_en,80,10,0.4875,0.3671
ru,80,6,0.4847,0.3823


In [98]:
# ============================================================
# CELDA 93 — MÁRGENES FINAL BENCHMARK
# ============================================================

scores_final = (
    modelo_hybrid_final
    .decision_function(
        X_final_hybrid
    )
)

orden_final = np.argsort(
    scores_final,
    axis=1
)

idx_final_top1 = (
    orden_final[:, -1]
)

idx_final_top2 = (
    orden_final[:, -2]
)

score_final_top1 = scores_final[
    np.arange(len(scores_final)),
    idx_final_top1
]

score_final_top2 = scores_final[
    np.arange(len(scores_final)),
    idx_final_top2
]

df_final_resultados[
    "segunda_categoria"
] = (
    modelo_hybrid_final
    .classes_[
        idx_final_top2
    ]
)

df_final_resultados[
    "score_top1"
] = score_final_top1

df_final_resultados[
    "score_top2"
] = score_final_top2

df_final_resultados[
    "margen"
] = (
    score_final_top1
    -
    score_final_top2
)

In [99]:
# ============================================================
# CELDA 94 — OPERACIÓN FINAL CONGELADA
# ============================================================

df_final_resultados[
    "estado"
] = [

    asignar_estado_v12(
        similarity_5nn=sim,
        margen=margen
    )

    for sim, margen in zip(
        df_final_resultados[
            "ood_similarity_5nn"
        ],
        df_final_resultados[
            "margen"
        ]
    )
]

In [100]:
# ============================================================
# CELDA 95 — RESULTADO OPERACIONAL FINAL
# ============================================================

mask_aceptadas_final_bench = (
    df_final_resultados[
        "estado"
    ]
    == "aceptada"
)

mask_revision_final_bench = (
    df_final_resultados[
        "estado"
    ]
    == "revision"
)

mask_ood_final_bench = (
    df_final_resultados[
        "estado"
    ]
    == "rechazada_ood"
)


n_aceptadas = int(
    mask_aceptadas_final_bench.sum()
)

n_revision = int(
    mask_revision_final_bench.sum()
)

n_ood = int(
    mask_ood_final_bench.sum()
)


accuracy_aceptadas = (
    df_final_resultados.loc[
        mask_aceptadas_final_bench,
        "correcta"
    ]
    .mean()
    if n_aceptadas > 0
    else np.nan
)


n_errores = int(
    (
        ~df_final_resultados[
            "correcta"
        ]
    ).sum()
)


errores_aceptados = int(
    (
        mask_aceptadas_final_bench
        &
        ~df_final_resultados[
            "correcta"
        ]
    ).sum()
)


error_capture = (
    (
        n_errores
        -
        errores_aceptados
    )
    /
    n_errores
    if n_errores > 0
    else 1.0
)


print("=" * 70)
print("FINAL BENCHMARK — OPERACIÓN COMPLETA")
print("=" * 70)

print(
    f"\nDocumentos:            "
    f"{len(df_final_resultados)}"
)

print(
    f"Aceptadas:             "
    f"{n_aceptadas}"
)

print(
    f"Revisión:              "
    f"{n_revision}"
)

print(
    f"Rechazadas OOD:        "
    f"{n_ood}"
)

print(
    f"\nCobertura automática:  "
    f"{n_aceptadas / len(df_final_resultados):.2%}"
)

print(
    f"Accuracy aceptadas:    "
    f"{accuracy_aceptadas:.2%}"
)

print(
    f"Errores totales:       "
    f"{n_errores}"
)

print(
    f"Errores aceptados:     "
    f"{errores_aceptados}"
)

print(
    f"Error capture:         "
    f"{error_capture:.2%}"
)

FINAL BENCHMARK — OPERACIÓN COMPLETA

Documentos:            320
Aceptadas:             120
Revisión:              160
Rechazadas OOD:        40

Cobertura automática:  37.50%
Accuracy aceptadas:    91.67%
Errores totales:       76
Errores aceptados:     10
Error capture:         86.84%


In [101]:
# ============================================================
# CELDA 96 — OPERACIÓN FINAL POR IDIOMA
# ============================================================

filas_operacion_final = []

for idioma, grupo in (
    df_final_resultados
    .groupby("idioma")
):

    aceptada = (
        grupo["estado"]
        == "aceptada"
    )

    errores = (
        ~grupo["correcta"]
    )

    n_errores_lang = int(
        errores.sum()
    )

    errores_aceptados_lang = int(
        (
            aceptada
            &
            errores
        ).sum()
    )

    filas_operacion_final.append({

        "idioma":
            idioma,

        "documentos":
            len(grupo),

        "accuracy_raw":
            grupo[
                "correcta"
            ].mean(),

        "aceptadas":
            int(
                aceptada.sum()
            ),

        "coverage":
            aceptada.mean(),

        "accepted_accuracy":
            (
                grupo.loc[
                    aceptada,
                    "correcta"
                ].mean()
                if aceptada.sum() > 0
                else np.nan
            ),

        "error_capture":
            (
                (
                    n_errores_lang
                    -
                    errores_aceptados_lang
                )
                /
                n_errores_lang
                if n_errores_lang > 0
                else 1.0
            ),

        "ood_rejected":
            int(
                (
                    grupo["estado"]
                    == "rechazada_ood"
                ).sum()
            )
    })


df_operacion_final_idioma = (
    pd.DataFrame(
        filas_operacion_final
    )
)

print("=" * 70)
print("FINAL BENCHMARK — OPERACIÓN POR IDIOMA")
print("=" * 70)

display(
    df_operacion_final_idioma
    .round(4)
)

FINAL BENCHMARK — OPERACIÓN POR IDIOMA


,idioma,documentos,accuracy_raw,aceptadas,coverage,accepted_accuracy,error_capture,ood_rejected
0,en,80,0.7750,34,0.4250,0.9118,0.8333,15
1,es,80,0.7500,23,0.2875,0.9565,0.9500,9
2,es_en,80,0.7875,36,0.4500,0.8889,0.7647,10
3,ru,80,0.7375,27,0.3375,0.9259,0.9048,6


In [102]:
# ============================================================
# CELDA 97 — v1.1 SOBRE BENCHMARK FINAL
# ============================================================

pred_final_v11 = (
    modelo_v11.predict(
        textos_final
    )
)

df_final_comparacion = (
    df_final_benchmark.copy()
)

df_final_comparacion[
    "pred_v11"
] = pred_final_v11

df_final_comparacion[
    "pred_v12"
] = (
    df_final_resultados[
        "categoria_predicha"
    ].to_numpy()
)

df_final_comparacion[
    "correcta_v11"
] = (
    df_final_comparacion[
        "categoria_real"
    ]
    ==
    df_final_comparacion[
        "pred_v11"
    ]
)

df_final_comparacion[
    "correcta_v12"
] = (
    df_final_comparacion[
        "categoria_real"
    ]
    ==
    df_final_comparacion[
        "pred_v12"
    ]
)


accuracy_v11_final = (
    df_final_comparacion[
        "correcta_v11"
    ].mean()
)

f1_v11_final = f1_score(
    df_final_comparacion[
        "categoria_real"
    ],
    df_final_comparacion[
        "pred_v11"
    ],
    average="macro",
    zero_division=0
)


print("=" * 70)
print("FINAL BENCHMARK — v1.1 VS v1.2")
print("=" * 70)

print(
    f"\nv1.1 Accuracy: "
    f"{accuracy_v11_final:.4f}"
)

print(
    f"v1.1 F1 Macro: "
    f"{f1_v11_final:.4f}"
)

print(
    f"\nv1.2 Accuracy: "
    f"{accuracy_final:.4f}"
)

print(
    f"v1.2 F1 Macro: "
    f"{f1_macro_final:.4f}"
)

print(
    f"\nΔ Accuracy: "
    f"{accuracy_final - accuracy_v11_final:+.4f}"
)

print(
    f"Δ F1 Macro: "
    f"{f1_macro_final - f1_v11_final:+.4f}"
)

FINAL BENCHMARK — v1.1 VS v1.2

v1.1 Accuracy: 0.5656
v1.1 F1 Macro: 0.5705

v1.2 Accuracy: 0.7625
v1.2 F1 Macro: 0.7570

Δ Accuracy: +0.1969
Δ F1 Macro: +0.1864


In [103]:
# ============================================================
# CELDA 98 — v1.1 VS v1.2 POR IDIOMA
# ============================================================

comparacion_idioma = []

for idioma, grupo in (
    df_final_comparacion
    .groupby("idioma")
):

    acc_v11 = (
        grupo[
            "correcta_v11"
        ].mean()
    )

    acc_v12 = (
        grupo[
            "correcta_v12"
        ].mean()
    )

    f1_v11 = f1_score(
        grupo["categoria_real"],
        grupo["pred_v11"],
        average="macro",
        zero_division=0
    )

    f1_v12 = f1_score(
        grupo["categoria_real"],
        grupo["pred_v12"],
        average="macro",
        zero_division=0
    )

    comparacion_idioma.append({

        "idioma":
            idioma,

        "accuracy_v11":
            acc_v11,

        "accuracy_v12":
            acc_v12,

        "delta_accuracy":
            acc_v12 - acc_v11,

        "f1_v11":
            f1_v11,

        "f1_v12":
            f1_v12,

        "delta_f1":
            f1_v12 - f1_v11
    })


df_comparacion_idioma_final = (
    pd.DataFrame(
        comparacion_idioma
    )
)

display(
    df_comparacion_idioma_final
    .round(4)
)

,idioma,accuracy_v11,accuracy_v12,delta_accuracy,f1_v11,f1_v12,delta_f1
0,en,0.6500,0.7750,0.1250,0.6573,0.7718,0.1145
1,es,0.5125,0.7500,0.2375,0.4971,0.7398,0.2428
2,es_en,0.7000,0.7875,0.0875,0.6988,0.7892,0.0904
3,ru,0.4000,0.7375,0.3375,0.3744,0.7235,0.3491


In [104]:
# ============================================================
# CELDA 99 — v1.1 VS v1.2 POR CATEGORÍA
# ============================================================

comparacion_categoria = []

for categoria, grupo in (
    df_final_comparacion
    .groupby("categoria_real")
):

    acc_v11 = (
        grupo[
            "correcta_v11"
        ].mean()
    )

    acc_v12 = (
        grupo[
            "correcta_v12"
        ].mean()
    )

    comparacion_categoria.append({

        "categoria":
            categoria,

        "documentos":
            len(grupo),

        "accuracy_v11":
            acc_v11,

        "accuracy_v12":
            acc_v12,

        "diferencia":
            acc_v12 - acc_v11
    })


df_comparacion_categoria_final = (
    pd.DataFrame(
        comparacion_categoria
    )
)


print("=" * 70)
print("FINAL BENCHMARK — COMPARACIÓN POR CATEGORÍA")
print("=" * 70)

display(
    df_comparacion_categoria_final
    .round(4)
)

FINAL BENCHMARK — COMPARACIÓN POR CATEGORÍA


,categoria,documentos,accuracy_v11,accuracy_v12,diferencia
0,backend,80,0.7375,0.9000,0.1625
1,cloud,80,0.4375,0.4000,-0.0375
2,datascience,80,0.4375,0.8375,0.4000
3,frontend,80,0.6500,0.9125,0.2625


In [105]:
# ============================================================
# CELDA 100 — CONFUSIONES DE CLOUD
# ============================================================

cloud_final = (
    df_final_resultados[
        df_final_resultados[
            "categoria_real"
        ]
        == "cloud"
    ]
)

confusiones_cloud = (
    cloud_final[
        "categoria_predicha"
    ]
    .value_counts()
    .rename_axis(
        "prediccion"
    )
    .reset_index(
        name="casos"
    )
)

confusiones_cloud[
    "porcentaje"
] = (
    confusiones_cloud[
        "casos"
    ]
    /
    len(cloud_final)
)

print("=" * 70)
print("CLOUD — DISTRIBUCIÓN DE PREDICCIONES")
print("=" * 70)

display(
    confusiones_cloud
    .round(4)
)

CLOUD — DISTRIBUCIÓN DE PREDICCIONES


,prediccion,casos,porcentaje
0,backend,42,0.525
1,cloud,32,0.400
2,datascience,6,0.075


In [106]:
# ============================================================
# CELDA 101 — CLOUD POR FAMILIA SEMÁNTICA
# ============================================================

cloud_family = (
    cloud_final
    .groupby(
        "semantic_family"
    )
    .agg(
        documentos=(
            "correcta",
            "size"
        ),

        correctas=(
            "correcta",
            "sum"
        ),

        accuracy=(
            "correcta",
            "mean"
        )
    )
    .sort_values(
        [
            "accuracy",
            "semantic_family"
        ]
    )
)


print("=" * 70)
print("CLOUD — FAMILIAS SEMÁNTICAS")
print("=" * 70)

display(
    cloud_family.round(4)
)

CLOUD — FAMILIAS SEMÁNTICAS


,documentos,correctas,accuracy
semantic_family,,,
autoscaling_metrics,4,0,0.00
cloud_budgets,4,0,0.00
container_resource_limits,4,0,0.00
dns_failover,4,0,0.00
kms_encryption,4,0,0.00
managed_db_replica,4,0,0.00
managed_queue_provisioning,4,0,0.00
multi_region_dr,4,0,0.00
observability_alerting,4,0,0.00


In [107]:
# ============================================================
# CELDA 99 — v1.1 VS v1.2 POR CATEGORÍA
# ============================================================

comparacion_categoria = []

for categoria, grupo in (
    df_final_comparacion
    .groupby("categoria_real")
):

    acc_v11 = (
        grupo["correcta_v11"]
        .mean()
    )

    acc_v12 = (
        grupo["correcta_v12"]
        .mean()
    )

    comparacion_categoria.append({

        "categoria":
            categoria,

        "documentos":
            len(grupo),

        "accuracy_v11":
            acc_v11,

        "accuracy_v12":
            acc_v12,

        "diferencia":
            acc_v12 - acc_v11
    })


df_comparacion_categoria_final = (
    pd.DataFrame(
        comparacion_categoria
    )
)

print("=" * 70)
print("FINAL BENCHMARK — COMPARACIÓN POR CATEGORÍA")
print("=" * 70)

display(
    df_comparacion_categoria_final
    .round(4)
)

FINAL BENCHMARK — COMPARACIÓN POR CATEGORÍA


,categoria,documentos,accuracy_v11,accuracy_v12,diferencia
0,backend,80,0.7375,0.9000,0.1625
1,cloud,80,0.4375,0.4000,-0.0375
2,datascience,80,0.4375,0.8375,0.4000
3,frontend,80,0.6500,0.9125,0.2625


In [108]:
# ============================================================
# CELDA 100 — CONFUSIONES CLOUD v1.2
# ============================================================

cloud_final = (
    df_final_resultados[
        df_final_resultados[
            "categoria_real"
        ]
        == "cloud"
    ]
    .copy()
)

confusiones_cloud = (
    cloud_final[
        "categoria_predicha"
    ]
    .value_counts()
    .rename_axis("prediccion")
    .reset_index(name="casos")
)

confusiones_cloud[
    "porcentaje"
] = (
    confusiones_cloud[
        "casos"
    ]
    /
    len(cloud_final)
)


print("=" * 70)
print("CLOUD — DISTRIBUCIÓN DE PREDICCIONES")
print("=" * 70)

display(
    confusiones_cloud
    .round(4)
)

CLOUD — DISTRIBUCIÓN DE PREDICCIONES


,prediccion,casos,porcentaje
0,backend,42,0.525
1,cloud,32,0.400
2,datascience,6,0.075


In [109]:
# ============================================================
# CELDA 101 — CLOUD POR FAMILIA SEMÁNTICA
# ============================================================

cloud_family = (
    cloud_final
    .groupby(
        "semantic_family"
    )
    .agg(
        documentos=(
            "correcta",
            "size"
        ),

        correctas=(
            "correcta",
            "sum"
        ),

        accuracy=(
            "correcta",
            "mean"
        )
    )
    .sort_values(
        [
            "accuracy",
            "semantic_family"
        ]
    )
)


print("=" * 70)
print("CLOUD — FAMILIAS SEMÁNTICAS")
print("=" * 70)

display(
    cloud_family
    .round(4)
)

CLOUD — FAMILIAS SEMÁNTICAS


,documentos,correctas,accuracy
semantic_family,,,
autoscaling_metrics,4,0,0.00
cloud_budgets,4,0,0.00
container_resource_limits,4,0,0.00
dns_failover,4,0,0.00
kms_encryption,4,0,0.00
managed_db_replica,4,0,0.00
managed_queue_provisioning,4,0,0.00
multi_region_dr,4,0,0.00
observability_alerting,4,0,0.00


In [110]:
# ============================================================
# CELDA 102 — CLOUD: FAMILIA × IDIOMA
# ============================================================

cloud_mapa = (
    cloud_final
    .pivot_table(
        index="semantic_family",
        columns="idioma",
        values="correcta",
        aggfunc="first"
    )
)

IDIOMAS_ORDEN = [
    "es",
    "en",
    "ru",
    "es_en"
]

cloud_mapa = cloud_mapa[
    IDIOMAS_ORDEN
]

cloud_mapa[
    "accuracy"
] = (
    cloud_mapa[
        IDIOMAS_ORDEN
    ]
    .mean(axis=1)
)

cloud_mapa = (
    cloud_mapa
    .sort_values(
        [
            "accuracy",
            "semantic_family"
        ]
    )
)

cloud_mapa_visual = (
    cloud_mapa.copy()
)

for idioma in IDIOMAS_ORDEN:

    cloud_mapa_visual[idioma] = (
        cloud_mapa_visual[idioma]
        .map({
            True: "✅",
            False: "❌"
        })
    )


print("=" * 70)
print("CLOUD — FAMILIA × IDIOMA")
print("=" * 70)

display(
    cloud_mapa_visual
)

CLOUD — FAMILIA × IDIOMA


idioma,es,en,ru,es_en,accuracy
semantic_family,,,,,
autoscaling_metrics,❌,❌,❌,❌,0.00
cloud_budgets,❌,❌,❌,❌,0.00
container_resource_limits,❌,❌,❌,❌,0.00
dns_failover,❌,❌,❌,❌,0.00
kms_encryption,❌,❌,❌,❌,0.00
managed_db_replica,❌,❌,❌,❌,0.00
managed_queue_provisioning,❌,❌,❌,❌,0.00
multi_region_dr,❌,❌,❌,❌,0.00
observability_alerting,❌,❌,❌,❌,0.00


In [111]:
# ============================================================
# CELDA 103 — CLOUD v1.1 VS v1.2 POR FAMILIA
# ============================================================

df_cloud_compare = (
    df_final_comparacion[
        df_final_comparacion[
            "categoria_real"
        ]
        == "cloud"
    ]
    .copy()
)

# Recuperar semantic_family si no quedó en df_final_comparacion
if (
    "semantic_family"
    not in df_cloud_compare.columns
):

    df_cloud_compare = (
        df_cloud_compare.merge(
            df_final_benchmark[
                [
                    "case_id",
                    "idioma",
                    "semantic_family"
                ]
            ],
            on=[
                "case_id",
                "idioma"
            ],
            how="left"
        )
    )


cloud_compare_family = (
    df_cloud_compare
    .groupby(
        "semantic_family"
    )
    .agg(
        documentos=(
            "correcta_v11",
            "size"
        ),

        correctas_v11=(
            "correcta_v11",
            "sum"
        ),

        accuracy_v11=(
            "correcta_v11",
            "mean"
        ),

        correctas_v12=(
            "correcta_v12",
            "sum"
        ),

        accuracy_v12=(
            "correcta_v12",
            "mean"
        )
    )
)

cloud_compare_family[
    "delta"
] = (
    cloud_compare_family[
        "accuracy_v12"
    ]
    -
    cloud_compare_family[
        "accuracy_v11"
    ]
)

cloud_compare_family = (
    cloud_compare_family
    .sort_values(
        [
            "delta",
            "accuracy_v12"
        ]
    )
)


print("=" * 70)
print("CLOUD — v1.1 VS v1.2 POR FAMILIA")
print("=" * 70)

display(
    cloud_compare_family
    .round(4)
)

CLOUD — v1.1 VS v1.2 POR FAMILIA


,documentos,correctas_v11,accuracy_v11,correctas_v12,accuracy_v12,delta
semantic_family,,,,,,
multi_region_dr,4,3,0.75,0,0.00,-0.75
cloud_budgets,4,2,0.50,0,0.00,-0.50
autoscaling_metrics,4,1,0.25,0,0.00,-0.25
container_resource_limits,4,1,0.25,0,0.00,-0.25
observability_alerting,4,1,0.25,0,0.00,-0.25
infrastructure_as_code,4,3,0.75,2,0.50,-0.25
blue_green_environments,4,4,1.00,3,0.75,-0.25
dns_failover,4,0,0.00,0,0.00,0.00
kms_encryption,4,0,0.00,0,0.00,0.00


In [112]:
# ============================================================
# CELDA 104 — COMPARACIÓN PAREADA v1.1 VS v1.2
# ============================================================

v11_ok = (
    df_final_comparacion[
        "correcta_v11"
    ].to_numpy()
)

v12_ok = (
    df_final_comparacion[
        "correcta_v12"
    ].to_numpy()
)


ambas_correctas = int(
    (
        v11_ok
        &
        v12_ok
    ).sum()
)

solo_v11 = int(
    (
        v11_ok
        &
        ~v12_ok
    ).sum()
)

solo_v12 = int(
    (
        ~v11_ok
        &
        v12_ok
    ).sum()
)

ambas_incorrectas = int(
    (
        ~v11_ok
        &
        ~v12_ok
    ).sum()
)


print("=" * 70)
print("COMPARACIÓN PAREADA — v1.1 VS v1.2")
print("=" * 70)

print(
    f"\nAmbas correctas:     "
    f"{ambas_correctas}"
)

print(
    f"Solo v1.1 correcta:  "
    f"{solo_v11}"
)

print(
    f"Solo v1.2 correcta:  "
    f"{solo_v12}"
)

print(
    f"Ambas incorrectas:   "
    f"{ambas_incorrectas}"
)

print(
    "\nTotal:",
    (
        ambas_correctas
        +
        solo_v11
        +
        solo_v12
        +
        ambas_incorrectas
    )
)

COMPARACIÓN PAREADA — v1.1 VS v1.2

Ambas correctas:     167
Solo v1.1 correcta:  14
Solo v1.2 correcta:  77
Ambas incorrectas:   62

Total: 320


In [113]:
# ============================================================
# CELDA 105 — MCNEMAR EXACTO
# ============================================================

from scipy.stats import binomtest

discordantes = (
    solo_v11
    +
    solo_v12
)

if discordantes > 0:

    resultado_mcnemar = (
        binomtest(
            k=min(
                solo_v11,
                solo_v12
            ),
            n=discordantes,
            p=0.5,
            alternative="two-sided"
        )
    )

    p_value_mcnemar = (
        resultado_mcnemar.pvalue
    )

else:

    p_value_mcnemar = 1.0


print("=" * 70)
print("MCNEMAR EXACTO — FINAL BENCHMARK")
print("=" * 70)

print(
    f"\nDiscordantes:      "
    f"{discordantes}"
)

print(
    f"Solo v1.1 correcta:"
    f" {solo_v11}"
)

print(
    f"Solo v1.2 correcta:"
    f" {solo_v12}"
)

print(
    f"\nP-value: "
    f"{p_value_mcnemar:.10f}"
)


if p_value_mcnemar < 0.001:

    print(
        "\n✅ Diferencia altamente "
        "significativa (p < 0.001)."
    )

elif p_value_mcnemar < 0.05:

    print(
        "\n✅ Diferencia estadísticamente "
        "significativa (p < 0.05)."
    )

else:

    print(
        "\n⚠️ No se detecta diferencia "
        "estadísticamente significativa."
    )

MCNEMAR EXACTO — FINAL BENCHMARK

Discordantes:      91
Solo v1.1 correcta: 14
Solo v1.2 correcta: 77

P-value: 0.0000000000

✅ Diferencia altamente significativa (p < 0.001).


In [114]:
# ============================================================
# CELDA 106 — GUARDAR ARTEFACTO CONGELADO v1.2
# ============================================================

import joblib
import numpy as np
from pathlib import Path

MODEL_DIR_V12 = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
)

MODEL_DIR_V12.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH_V12 = (
    MODEL_DIR_V12
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)


artefacto_v12 = {

    # --------------------------------------------------------
    # IDENTIDAD
    # --------------------------------------------------------

    "version":
        "1.2.0-multilingual",

    "status":
        "validated_experimental_candidate",

    # --------------------------------------------------------
    # REPRESENTACIÓN TEXTUAL
    # --------------------------------------------------------

    "features":
        features_hybrid_final,

    # --------------------------------------------------------
    # CLASIFICADOR
    # --------------------------------------------------------

    "classifier":
        modelo_hybrid_final,

    "classifier_type":
        "LinearSVC",

    "classifier_C":
        0.3,

    # --------------------------------------------------------
    # EMBEDDINGS
    # --------------------------------------------------------

    "embedding_model":
        (
            "sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2"
        ),

    "embedding_dimension":
        384,

    "normalize_embeddings":
        True,

    # No duplicamos pesos MiniLM.
    # Solo guardamos las referencias semánticas necesarias
    # para el detector de dominio.
    "domain_reference_embeddings":
        X_train_embeddings.astype(
            np.float32
        ),

    # --------------------------------------------------------
    # CONTROL DE DOMINIO
    # --------------------------------------------------------

    "domain_control": {

        "metric":
            "mean_cosine_similarity_5nn",

        "n_neighbors":
            5,

        "threshold":
            0.4266
    },

    # --------------------------------------------------------
    # CONTROL DE INCERTIDUMBRE
    # --------------------------------------------------------

    "confidence_control": {

        "metric":
            "top1_minus_top2_decision_margin",

        "threshold":
            0.8132
    },

    # --------------------------------------------------------
    # CLASES
    # --------------------------------------------------------

    "classes":
        list(
            modelo_hybrid_final.classes_
        ),

    # --------------------------------------------------------
    # ARQUITECTURA
    # --------------------------------------------------------

    "architecture":
        "TF-IDF Word+Char + MiniLM 384 + LinearSVC",

    "random_state":
        42
}


joblib.dump(
    artefacto_v12,
    MODEL_PATH_V12,
    compress=3
)


print("=" * 70)
print("ARTEFACTO v1.2 GUARDADO")
print("=" * 70)

print(
    "\nRuta:",
    MODEL_PATH_V12
)

print(
    "Tamaño:",
    f"{MODEL_PATH_V12.stat().st_size / (1024**2):.2f} MB"
)

ARTEFACTO v1.2 GUARDADO

Ruta: C:\Users\MAMÁ\Downloads\techmind-v2\models\experimental\v1.2.0-multilingual\techmind_hybrid_v1_2_0_multilingual.joblib
Tamaño: 6.81 MB


In [115]:
# ============================================================
# CELDA 107 — SHA256 ARTEFACTO v1.2
# ============================================================

import hashlib

sha256_model = hashlib.sha256()

with open(
    MODEL_PATH_V12,
    "rb"
) as f:

    for bloque in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha256_model.update(
            bloque
        )


MODEL_SHA256_V12 = (
    sha256_model.hexdigest()
)


print("=" * 70)
print("SHA256 — TECHMIND v1.2")
print("=" * 70)

print(
    "\n",
    MODEL_SHA256_V12
)

SHA256 — TECHMIND v1.2

 1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61


In [116]:
# ============================================================
# CELDA 108 — EXPORTAR RESULTADOS FINALES
# ============================================================

REPORT_DIR_V12 = (
    ROOT
    / "experiments"
    / "v1.2.0-multilingual"
    / "reports"
)

REPORT_DIR_V12.mkdir(
    parents=True,
    exist_ok=True
)


PATH_FINAL_RESULTS = (
    REPORT_DIR_V12
    / "final_independent_benchmark_predictions.csv"
)


df_final_resultados.to_csv(
    PATH_FINAL_RESULTS,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 70)
print("RESULTADOS FINALES EXPORTADOS")
print("=" * 70)

print(
    "\n",
    PATH_FINAL_RESULTS
)

RESULTADOS FINALES EXPORTADOS

 C:\Users\MAMÁ\Downloads\techmind-v2\experiments\v1.2.0-multilingual\reports\final_independent_benchmark_predictions.csv


In [117]:
# ============================================================
# CELDA 109 — REPORTE FINAL JSON
# ============================================================

import json

reporte_final_v12 = {

    "model": {
        "version":
            "1.2.0-multilingual",

        "status":
            "validated_experimental_candidate",

        "artifact_sha256":
            MODEL_SHA256_V12
    },

    "final_independent_benchmark": {

        "documents":
            320,

        "semantic_cases":
            80,

        "accuracy":
            0.7625,

        "precision_macro":
            0.8167,

        "recall_macro":
            0.7625,

        "f1_macro":
            0.7570,

        "cross_language_consistency":
            0.80
    },

    "comparison_v11": {

        "accuracy_v11":
            0.5656,

        "accuracy_v12":
            0.7625,

        "accuracy_delta":
            0.1969,

        "f1_v11":
            0.5705,

        "f1_v12":
            0.7570,

        "f1_delta":
            0.1864,

        "mcnemar_only_v11":
            14,

        "mcnemar_only_v12":
            77,

        "mcnemar_p_value":
            1.0476614334104999e-11
    },

    "accuracy_by_category": {

        "backend":
            0.9000,

        "cloud":
            0.4000,

        "datascience":
            0.8375,

        "frontend":
            0.9125
    },

    "accuracy_by_language": {

        "en":
            0.7750,

        "es":
            0.7500,

        "es_en":
            0.7875,

        "ru":
            0.7375
    },

    "operational": {

        "domain_threshold":
            0.4266,

        "margin_threshold":
            0.8132,

        "automatic_coverage":
            0.3750,

        "accepted_accuracy":
            0.9167,

        "error_capture":
            0.8684
    },

    "latency_ms": {

        "median":
            46.61,

        "p95":
            152.38
    },

    "known_limitation": {

        "category":
            "cloud",

        "accuracy":
            0.40,

        "errors":
            48,

        "errors_predicted_backend":
            42,

        "backend_share_of_cloud_errors":
            0.875,

        "interpretation":
            (
                "Main limitation is conceptual "
                "coverage / Cloud-Backend boundary, "
                "not a language-specific degradation."
            )
    }
}


PATH_FINAL_REPORT = (
    REPORT_DIR_V12
    / "final_evaluation_summary.json"
)


with open(
    PATH_FINAL_REPORT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        reporte_final_v12,
        f,
        indent=2,
        ensure_ascii=False
    )


print("=" * 70)
print("REPORTE FINAL v1.2")
print("=" * 70)

print(
    "\n",
    PATH_FINAL_REPORT
)

REPORTE FINAL v1.2

 C:\Users\MAMÁ\Downloads\techmind-v2\experiments\v1.2.0-multilingual\reports\final_evaluation_summary.json


In [118]:
# ============================================================
# CELDA 110 — ESTRUCTURA DEL PREDICTOR v1.2
# ============================================================

from pathlib import Path

PACKAGE_V12 = ROOT / "techmind_v12"

PACKAGE_V12.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("PAQUETE EXPERIMENTAL v1.2")
print("=" * 70)

print("\nRuta:")
print(PACKAGE_V12)

print("\n⚠️ techmind/ v1.1 NO se modifica.")

PAQUETE EXPERIMENTAL v1.2

Ruta:
C:\Users\MAMÁ\Downloads\techmind-v2\techmind_v12

⚠️ techmind/ v1.1 NO se modifica.


In [119]:
# ============================================================
# CELDA 111 — __init__.py
# ============================================================

INIT_V12 = '''\
"""
TechMind v1.2.0-multilingual
Experimental validated candidate.
"""

from .predictor import TechMindPredictor

__version__ = "1.2.0-multilingual"

__all__ = [
    "TechMindPredictor",
]
'''

(
    PACKAGE_V12
    / "__init__.py"
).write_text(
    INIT_V12,
    encoding="utf-8"
)

print("✅ techmind_v12/__init__.py creado")

✅ techmind_v12/__init__.py creado


In [120]:
# ============================================================
# CELDA 112 — CREAR predictor.py
# ============================================================

PREDICTOR_V12 = r'''\
from __future__ import annotations

from collections.abc import Iterable
from pathlib import Path
from typing import Any
import hashlib

import joblib
import numpy as np

from scipy.sparse import (
    csr_matrix,
    hstack,
)

from sklearn.neighbors import NearestNeighbors

from sentence_transformers import SentenceTransformer


class TechMindPredictor:
    """
    Predictor experimental para TechMind v1.2.0-multilingual.

    Arquitectura
    ------------
    TF-IDF Word+Char
        +
    paraphrase-multilingual-MiniLM-L12-v2
        +
    LinearSVC

    Controles operacionales
    -----------------------
    1. Validación de entrada.
    2. Semantic domain-support mediante similitud media 5NN.
    3. Clasificación híbrida.
    4. Revisión por margen top1 - top2.

    Nota
    ----
    Los scores de LinearSVC NO son probabilidades.
    """

    def __init__(
        self,
        model_path: str | Path,
    ) -> None:

        self.model_path = Path(model_path)

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"No existe el artefacto: {self.model_path}"
            )

        # -----------------------------------------------------
        # CARGA DEL ARTEFACTO
        # -----------------------------------------------------

        artifact = joblib.load(
            self.model_path
        )

        self.artifact = artifact

        self.version = artifact[
            "version"
        ]

        self.status = artifact.get(
            "status",
            "experimental"
        )

        self.features = artifact[
            "features"
        ]

        self.classifier = artifact[
            "classifier"
        ]

        self.classes_ = np.asarray(
            artifact.get(
                "classes",
                self.classifier.classes_
            )
        )

        # -----------------------------------------------------
        # EMBEDDING MODEL
        # -----------------------------------------------------

        self.embedding_model_name = (
            artifact[
                "embedding_model"
            ]
        )

        self.embedding_dimension = int(
            artifact[
                "embedding_dimension"
            ]
        )

        self.normalize_embeddings = bool(
            artifact.get(
                "normalize_embeddings",
                True
            )
        )

        self.encoder = SentenceTransformer(
            self.embedding_model_name,
            local_files_only=True
        )

        # -----------------------------------------------------
        # DOMAIN SUPPORT
        # -----------------------------------------------------

        self.domain_reference_embeddings = (
            np.asarray(
                artifact[
                    "domain_reference_embeddings"
                ],
                dtype=np.float32
            )
        )

        domain_control = artifact[
            "domain_control"
        ]

        self.domain_n_neighbors = int(
            domain_control[
                "n_neighbors"
            ]
        )

        self.domain_threshold = float(
            domain_control[
                "threshold"
            ]
        )

        self.domain_index = (
            NearestNeighbors(
                n_neighbors=(
                    self.domain_n_neighbors
                ),
                metric="cosine",
                algorithm="brute",
                n_jobs=1
            )
        )

        self.domain_index.fit(
            self.domain_reference_embeddings
        )

        # -----------------------------------------------------
        # CONFIDENCE CONTROL
        # -----------------------------------------------------

        confidence_control = artifact[
            "confidence_control"
        ]

        self.margin_threshold = float(
            confidence_control[
                "threshold"
            ]
        )

        # -----------------------------------------------------
        # INTEGRITY
        # -----------------------------------------------------

        self.artifact_sha256 = (
            self._sha256_file(
                self.model_path
            )
        )


    # =========================================================
    # PUBLIC INFO
    # =========================================================

    def model_info(
        self
    ) -> dict[str, Any]:

        return {

            "version":
                self.version,

            "status":
                self.status,

            "architecture":
                self.artifact.get(
                    "architecture"
                ),

            "classifier":
                self.artifact.get(
                    "classifier_type",
                    type(
                        self.classifier
                    ).__name__
                ),

            "classifier_C":
                self.artifact.get(
                    "classifier_C"
                ),

            "embedding_model":
                self.embedding_model_name,

            "embedding_dimension":
                self.embedding_dimension,

            "classes":
                self.classes_.tolist(),

            "domain_control": {
                "metric":
                    "mean_cosine_similarity_5nn",

                "threshold":
                    self.domain_threshold,

                "n_neighbors":
                    self.domain_n_neighbors,
            },

            "confidence_control": {
                "metric":
                    "top1_minus_top2_decision_margin",

                "threshold":
                    self.margin_threshold,
            },

            "artifact_sha256":
                self.artifact_sha256,

            "scores_are_probabilities":
                False,
        }


    # =========================================================
    # PREDICT
    # =========================================================

    def predict(
        self,
        texts: str | Iterable[str],
        include_explanation: bool = False,
        explanation_top_n: int = 8,
        top_k: int | None = None,
    ) -> dict[str, Any]:

        # -----------------------------------------------------
        # NORMALIZACIÓN DE ENTRADA
        # -----------------------------------------------------

        original_inputs = (
            self._normalize_input(
                texts
            )
        )

        if explanation_top_n <= 0:
            raise ValueError(
                "explanation_top_n debe ser > 0."
            )

        if top_k is not None:

            if (
                not isinstance(
                    top_k,
                    int
                )
                or top_k <= 0
            ):
                raise ValueError(
                    "top_k debe ser un entero > 0."
                )

            top_k = min(
                top_k,
                len(
                    self.classes_
                )
            )

        # -----------------------------------------------------
        # VALIDACIÓN
        # -----------------------------------------------------

        valid_mask = np.array(
            [
                self._is_valid_text(
                    value
                )
                for value
                in original_inputs
            ],
            dtype=bool
        )

        results: list[
            dict[str, Any] | None
        ] = [
            None
        ] * len(
            original_inputs
        )

        # Entradas inválidas no pasan por el modelo.
        for idx, is_valid in enumerate(
            valid_mask
        ):

            if not is_valid:

                results[idx] = {
                    "index":
                        idx,

                    "text":
                        original_inputs[
                            idx
                        ],

                    "valid_input":
                        False,

                    "decision":
                        "rejected_invalid",

                    "prediction":
                        None,

                    "second_category":
                        None,

                    "decision_margin":
                        None,

                    "domain_similarity_5nn":
                        None,

                    "tfidf_active_features":
                        0,

                    "reason":
                        "invalid_input",
                }

        valid_indices = np.flatnonzero(
            valid_mask
        )

        if len(
            valid_indices
        ) == 0:

            return self._build_response(
                results
            )

        valid_texts = [
            original_inputs[i]
            for i
            in valid_indices
        ]

        # -----------------------------------------------------
        # MINILM
        # -----------------------------------------------------

        embeddings = (
            self.encoder.encode(
                valid_texts,
                batch_size=32,
                normalize_embeddings=(
                    self.normalize_embeddings
                ),
                convert_to_numpy=True,
                show_progress_bar=False
            )
        )

        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )

        # -----------------------------------------------------
        # SEMANTIC DOMAIN SUPPORT
        # -----------------------------------------------------

        domain_distances, _ = (
            self.domain_index.kneighbors(
                embeddings
            )
        )

        domain_similarities = (
            1.0
            -
            domain_distances
        )

        domain_similarity_5nn = (
            domain_similarities.mean(
                axis=1
            )
        )

        # -----------------------------------------------------
        # TF-IDF
        # -----------------------------------------------------

        tfidf = (
            self.features.transform(
                valid_texts
            )
        )

        tfidf_active = np.asarray(
            tfidf.getnnz(
                axis=1
            )
        ).ravel()

        # -----------------------------------------------------
        # HYBRID REPRESENTATION
        # -----------------------------------------------------

        hybrid = hstack(
            [
                tfidf,
                csr_matrix(
                    embeddings
                )
            ],
            format="csr"
        )

        if (
            hybrid.shape[1]
            !=
            self.classifier.n_features_in_
        ):
            raise RuntimeError(
                "Dimensión híbrida incompatible "
                "con el clasificador."
            )

        # -----------------------------------------------------
        # LINEARSVC SCORES
        # -----------------------------------------------------

        scores = (
            self.classifier
            .decision_function(
                hybrid
            )
        )

        scores = np.asarray(
            scores
        )

        if scores.ndim != 2:
            raise RuntimeError(
                "Se esperaba clasificación "
                "multiclase con decision_function 2D."
            )

        order = np.argsort(
            scores,
            axis=1
        )

        top1_idx = order[
            :,
            -1
        ]

        top2_idx = order[
            :,
            -2
        ]

        top1_scores = scores[
            np.arange(
                len(valid_texts)
            ),
            top1_idx
        ]

        top2_scores = scores[
            np.arange(
                len(valid_texts)
            ),
            top2_idx
        ]

        margins = (
            top1_scores
            -
            top2_scores
        )

        predicted_classes = (
            self.classes_[
                top1_idx
            ]
        )

        second_classes = (
            self.classes_[
                top2_idx
            ]
        )

        # -----------------------------------------------------
        # RESULTS
        # -----------------------------------------------------

        for local_idx, original_idx in enumerate(
            valid_indices
        ):

            similarity = float(
                domain_similarity_5nn[
                    local_idx
                ]
            )

            margin = float(
                margins[
                    local_idx
                ]
            )

            predicted = str(
                predicted_classes[
                    local_idx
                ]
            )

            second = str(
                second_classes[
                    local_idx
                ]
            )

            # Orden operacional:
            #
            # valid input
            # -> semantic support
            # -> margin
            #
            if (
                similarity
                <
                self.domain_threshold
            ):

                decision = (
                    "rejected_ood"
                )

                reason = (
                    "low_semantic_domain_support"
                )

            elif (
                margin
                <
                self.margin_threshold
            ):

                decision = (
                    "review"
                )

                reason = (
                    "low_decision_margin"
                )

            else:

                decision = (
                    "accepted"
                )

                reason = None

            item = {

                "index":
                    int(
                        original_idx
                    ),

                "text":
                    original_inputs[
                        original_idx
                    ],

                "valid_input":
                    True,

                "decision":
                    decision,

                "prediction":
                    predicted,

                "second_category":
                    second,

                "decision_margin":
                    margin,

                "domain_similarity_5nn":
                    similarity,

                "tfidf_active_features":
                    int(
                        tfidf_active[
                            local_idx
                        ]
                    ),

                "reason":
                    reason,

                "score_top1":
                    float(
                        top1_scores[
                            local_idx
                        ]
                    ),

                "score_top2":
                    float(
                        top2_scores[
                            local_idx
                        ]
                    ),
            }

            # -------------------------------------------------
            # TOP-K
            # -------------------------------------------------

            if top_k is not None:

                ranking_idx = (
                    order[
                        local_idx
                    ][::-1][
                        :top_k
                    ]
                )

                item[
                    "top_k"
                ] = [

                    {
                        "category":
                            str(
                                self.classes_[
                                    class_idx
                                ]
                            ),

                        "score":
                            float(
                                scores[
                                    local_idx,
                                    class_idx
                                ]
                            )
                    }

                    for class_idx
                    in ranking_idx
                ]

            # -------------------------------------------------
            # EXPLANATION
            # -------------------------------------------------

            if include_explanation:

                item[
                    "explanation"
                ] = (
                    self._explain_tfidf(
                        tfidf_row=(
                            tfidf[
                                local_idx
                            ]
                        ),
                        predicted_idx=int(
                            top1_idx[
                                local_idx
                            ]
                        ),
                        second_idx=int(
                            top2_idx[
                                local_idx
                            ]
                        ),
                        top_n=(
                            explanation_top_n
                        )
                    )
                )

            results[
                original_idx
            ] = item

        return self._build_response(
            results
        )


    # =========================================================
    # EXPLAINABILITY
    # =========================================================

    def _explain_tfidf(
        self,
        tfidf_row,
        predicted_idx: int,
        second_idx: int,
        top_n: int,
    ) -> dict[str, Any]:

        """
        Explicación local aproximada limitada al componente
        interpretable TF-IDF.

        NO es una explicación causal del modelo completo.
        """

        tfidf_dim = (
            tfidf_row.shape[1]
        )

        coef = np.asarray(
            self.classifier.coef_
        )

        if (
            coef.shape[1]
            <
            tfidf_dim
        ):
            return {
                "available":
                    False,

                "reason":
                    "classifier_dimension_mismatch",
            }

        differential_coef = (
            coef[
                predicted_idx,
                :tfidf_dim
            ]
            -
            coef[
                second_idx,
                :tfidf_dim
            ]
        )

        row = (
            tfidf_row
            .tocsr()
        )

        active_indices = (
            row.indices
        )

        active_values = (
            row.data
        )

        if len(
            active_indices
        ) == 0:

            return {
                "available":
                    True,

                "scope":
                    "tfidf_differential_only",

                "terms":
                    [],
            }

        contributions = (
            active_values
            *
            differential_coef[
                active_indices
            ]
        )

        order = np.argsort(
            np.abs(
                contributions
            )
        )[::-1][
            :top_n
        ]

        feature_names = (
            self._feature_names(
                tfidf_dim
            )
        )

        terms = []

        for pos in order:

            feature_idx = int(
                active_indices[
                    pos
                ]
            )

            contribution = float(
                contributions[
                    pos
                ]
            )

            terms.append({
                "feature":
                    feature_names[
                        feature_idx
                    ],

                "value":
                    float(
                        active_values[
                            pos
                        ]
                    ),

                "differential_contribution":
                    contribution,

                "direction":
                    (
                        "predicted"
                        if contribution >= 0
                        else "second_category"
                    )
            })

        return {
            "available":
                True,

            "scope":
                "tfidf_differential_only",

            "note":
                (
                    "MiniLM dimensions are not mapped "
                    "to human-readable terms."
                ),

            "terms":
                terms,
        }


    # =========================================================
    # HELPERS
    # =========================================================

    def _feature_names(
        self,
        tfidf_dim: int,
    ) -> list[str]:

        try:

            names = (
                self.features
                .get_feature_names_out()
            )

            names = [
                str(x)
                for x
                in names
            ]

            if (
                len(names)
                ==
                tfidf_dim
            ):
                return names

        except Exception:
            pass

        return [
            f"feature_{i}"
            for i
            in range(
                tfidf_dim
            )
        ]


    @staticmethod
    def _normalize_input(
        texts: str | Iterable[str],
    ) -> list[Any]:

        if isinstance(
            texts,
            str
        ):
            return [
                texts
            ]

        if texts is None:
            return [
                None
            ]

        try:
            return list(
                texts
            )

        except TypeError as exc:

            raise TypeError(
                "texts debe ser str "
                "o Iterable[str]."
            ) from exc


    @staticmethod
    def _is_valid_text(
        value: Any,
    ) -> bool:

        return (
            isinstance(
                value,
                str
            )
            and bool(
                value.strip()
            )
        )


    def _build_response(
        self,
        results: list[
            dict[str, Any] | None
        ],
    ) -> dict[str, Any]:

        final_results = [
            x
            for x
            in results
            if x is not None
        ]

        decisions = {
            "accepted": 0,
            "review": 0,
            "rejected_ood": 0,
            "rejected_invalid": 0,
        }

        for item in final_results:

            decision = item[
                "decision"
            ]

            decisions[
                decision
            ] = (
                decisions.get(
                    decision,
                    0
                )
                +
                1
            )

        return {

            "model_version":
                self.version,

            "model_status":
                self.status,

            "n_inputs":
                len(
                    final_results
                ),

            "summary":
                decisions,

            "predictions":
                final_results,
        }


    @staticmethod
    def _sha256_file(
        path: Path,
    ) -> str:

        sha256 = hashlib.sha256()

        with path.open(
            "rb"
        ) as f:

            for block in iter(
                lambda: f.read(
                    1024 * 1024
                ),
                b""
            ):
                sha256.update(
                    block
                )

        return (
            sha256.hexdigest()
        )
'''

(
    PACKAGE_V12
    / "predictor.py"
).write_text(
    PREDICTOR_V12,
    encoding="utf-8"
)

print("✅ techmind_v12/predictor.py creado")

✅ techmind_v12/predictor.py creado


In [121]:
# ============================================================
# CELDA 113 — IMPORTAR TECHMIND v1.2
# ============================================================

import sys
import importlib

if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT)
    )

import techmind_v12
import techmind_v12.predictor

importlib.reload(
    techmind_v12.predictor
)

from techmind_v12.predictor import (
    TechMindPredictor
)


print("=" * 70)
print("IMPORT TECHMIND v1.2")
print("=" * 70)

print(
    "\nPackage version:",
    techmind_v12.__version__
)

print(
    "Predictor:",
    TechMindPredictor
)

IMPORT TECHMIND v1.2

Package version: 1.2.0-multilingual
Predictor: <class 'techmind_v12.predictor.TechMindPredictor'>


In [122]:
# ============================================================
# CELDA 114 — CARGAR ARTEFACTO CON PREDICTOR
# ============================================================

from pathlib import Path

MODEL_PATH_V12_LOAD = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)

predictor_v12 = TechMindPredictor(
    MODEL_PATH_V12_LOAD
)


print("=" * 70)
print("TECHMIND PREDICTOR v1.2")
print("=" * 70)

info_v12 = (
    predictor_v12.model_info()
)

for key, value in info_v12.items():
    print(
        f"{key}: {value}"
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5648.15it/s]


TECHMIND PREDICTOR v1.2
version: 1.2.0-multilingual
status: validated_experimental_candidate
architecture: TF-IDF Word+Char + MiniLM 384 + LinearSVC
classifier: LinearSVC
classifier_C: 0.3
embedding_model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
embedding_dimension: 384
classes: ['backend', 'cloud', 'datascience', 'frontend']
domain_control: {'metric': 'mean_cosine_similarity_5nn', 'threshold': 0.4266, 'n_neighbors': 5}
confidence_control: {'metric': 'top1_minus_top2_decision_margin', 'threshold': 0.8132}
artifact_sha256: 1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61
scores_are_probabilities: False


In [123]:
# ============================================================
# VERIFICACIÓN DE INTEGRIDAD DEL ARTEFACTO
# ============================================================

print("\n" + "=" * 70)
print("VERIFICACIÓN SHA256")
print("=" * 70)

print(
    "\nSHA Celda 107:"
)
print(
    MODEL_SHA256_V12
)

print(
    "\nSHA cargado por predictor:"
)
print(
    predictor_v12.artifact_sha256
)

SHA_PREDICTOR_OK = (
    predictor_v12.artifact_sha256
    ==
    MODEL_SHA256_V12
)

print(
    "\nCoinciden:",
    SHA_PREDICTOR_OK
)

assert SHA_PREDICTOR_OK, (
    "El predictor no está cargando "
    "el mismo artefacto congelado."
)

print(
    "\n✅ Artefacto congelado "
    "v1.2 confirmado."
)


VERIFICACIÓN SHA256

SHA Celda 107:
1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61

SHA cargado por predictor:
1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61

Coinciden: True

✅ Artefacto congelado v1.2 confirmado.


In [124]:
# ============================================================
# CELDA 116 — SMOKE TEST MULTILINGÜE
# ============================================================

smoke_texts_v12 = [

    # Backend — ES
    (
        "Crear un endpoint REST que valide "
        "la solicitud y guarde el usuario "
        "en la base de datos."
    ),

    # Cloud — EN
    (
        "Store application backups in "
        "object storage with lifecycle policies."
    ),

    # Data Science — RU
    (
        "Обучить модель классификации "
        "и оценить precision, recall и F1."
    ),

    # Frontend — ES/EN
    (
        "Crear un responsive component "
        "con CSS Grid y keyboard accessibility."
    ),

    # OOD — ES
    (
        "Preparar una tortilla de patatas "
        "con huevos, cebolla y aceite de oliva."
    ),

    # OOD — RU
    (
        "Футбольная команда забила два гола "
        "во втором тайме матча."
    ),

    # Inválido
    "",

]


resultado_smoke_v12 = (
    predictor_v12.predict(
        smoke_texts_v12,
        top_k=4
    )
)


print("=" * 70)
print("SMOKE TEST — TECHMIND v1.2")
print("=" * 70)

print(
    "\nSummary:",
    resultado_smoke_v12[
        "summary"
    ]
)


for pred in (
    resultado_smoke_v12[
        "predictions"
    ]
):

    print("\n" + "-" * 70)

    print(
        "Input:",
        repr(
            pred["text"]
        )
    )

    print(
        "Decision:",
        pred["decision"]
    )

    print(
        "Prediction:",
        pred["prediction"]
    )

    print(
        "Second:",
        pred["second_category"]
    )

    print(
        "Margin:",
        pred["decision_margin"]
    )

    print(
        "Domain 5NN:",
        pred["domain_similarity_5nn"]
    )

    print(
        "Reason:",
        pred["reason"]
    )

SMOKE TEST — TECHMIND v1.2

Summary: {'accepted': 3, 'review': 1, 'rejected_ood': 2, 'rejected_invalid': 1}

----------------------------------------------------------------------
Input: 'Crear un endpoint REST que valide la solicitud y guarde el usuario en la base de datos.'
Decision: accepted
Prediction: backend
Second: cloud
Margin: 1.0848222728426737
Domain 5NN: 0.48260682821273804
Reason: None

----------------------------------------------------------------------
Input: 'Store application backups in object storage with lifecycle policies.'
Decision: review
Prediction: cloud
Second: backend
Margin: 0.6464757011825304
Domain 5NN: 0.580776572227478
Reason: low_decision_margin

----------------------------------------------------------------------
Input: 'Обучить модель классификации и оценить precision, recall и F1.'
Decision: accepted
Prediction: datascience
Second: cloud
Margin: 1.2666311350811945
Domain 5NN: 0.43076029419898987
Reason: None

--------------------------------------

In [125]:
# ============================================================
# CELDA 117 — VALIDACIÓN AUTOMÁTICA
# ============================================================

preds_smoke = (
    resultado_smoke_v12[
        "predictions"
    ]
)


validaciones_predictor = {

    "7_resultados":
        len(
            preds_smoke
        ) == 7,

    "backend_no_ood":
        preds_smoke[
            0
        ][
            "decision"
        ]
        != "rejected_ood",

    "cloud_no_ood":
        preds_smoke[
            1
        ][
            "decision"
        ]
        != "rejected_ood",

    "ru_datascience_no_ood":
        preds_smoke[
            2
        ][
            "decision"
        ]
        != "rejected_ood",

    "frontend_no_ood":
        preds_smoke[
            3
        ][
            "decision"
        ]
        != "rejected_ood",

    "cocina_ood":
        preds_smoke[
            4
        ][
            "decision"
        ]
        == "rejected_ood",

    "futbol_ood":
        preds_smoke[
            5
        ][
            "decision"
        ]
        == "rejected_ood",

    "vacio_rechazado":
        preds_smoke[
            6
        ][
            "decision"
        ]
        == "rejected_invalid",
}


print("=" * 70)
print("VALIDACIÓN PREDICTOR v1.2")
print("=" * 70)

for nombre, valor in (
    validaciones_predictor.items()
):

    print(
        f"{'✅' if valor else '❌'} "
        f"{nombre}: {valor}"
    )


PREDICTOR_V12_OK = all(
    validaciones_predictor.values()
)

print(
    "\nPREDICTOR LISTO:",
    PREDICTOR_V12_OK
)

VALIDACIÓN PREDICTOR v1.2
✅ 7_resultados: True
✅ backend_no_ood: True
✅ cloud_no_ood: True
✅ ru_datascience_no_ood: True
✅ frontend_no_ood: True
✅ cocina_ood: True
✅ futbol_ood: True
✅ vacio_rechazado: True

PREDICTOR LISTO: True


In [126]:
# ============================================================
# CELDA 118 — EXPLICABILIDAD
# ============================================================

resultado_explicacion = (
    predictor_v12.predict(
        (
            "Train a classifier with "
            "cross validation and evaluate "
            "precision recall and F1."
        ),
        include_explanation=True,
        explanation_top_n=8,
        top_k=4
    )
)


pred_explicacion = (
    resultado_explicacion[
        "predictions"
    ][0]
)


print("=" * 70)
print("EXPLICABILIDAD v1.2")
print("=" * 70)

print(
    "\nPrediction:",
    pred_explicacion[
        "prediction"
    ]
)

print(
    "Second:",
    pred_explicacion[
        "second_category"
    ]
)

print(
    "Margin:",
    round(
        pred_explicacion[
            "decision_margin"
        ],
        4
    )
)

print(
    "Decision:",
    pred_explicacion[
        "decision"
    ]
)

print(
    "\nExplanation:"
)

display(
    pred_explicacion[
        "explanation"
    ]
)

EXPLICABILIDAD v1.2

Prediction: datascience
Second: backend
Margin: 1.2035
Decision: accepted

Explanation:


{'available': True,
 'scope': 'tfidf_differential_only',
 'note': 'MiniLM dimensions are not mapped to human-readable terms.',
 'terms': [{'feature': 'word__classifier',
   'value': 0.36763981613210434,
   'differential_contribution': 0.08724369492364091,
   'direction': 'predicted'},
  {'feature': 'word__validation',
   'value': 0.3055070533289925,
   'differential_contribution': -0.06611927855209541,
   'direction': 'second_category'},
  {'feature': 'word__and',
   'value': 0.1626190036875434,
   'differential_contribution': 0.055726348702762565,
   'direction': 'predicted'},
  {'feature': 'word__recall',
   'value': 0.40022310273816386,
   'differential_contribution': 0.04843582438924095,
   'direction': 'predicted'},
  {'feature': 'word__with',
   'value': 0.11776233874085094,
   'differential_contribution': 0.022147997704861155,
   'direction': 'predicted'},
  {'feature': 'char__ate ',
   'value': 0.05002159317169814,
   'differential_contribution': -0.02014846652387765,
   'direc

In [127]:
# ============================================================
# CELDA 119 — ESTRUCTURA FASTAPI v1.2
# ============================================================

from pathlib import Path

API_PACKAGE_V12 = (
    ROOT
    / "techmind_api_v12"
)

API_PACKAGE_V12.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("FASTAPI EXPERIMENTAL v1.2")
print("=" * 70)

print("\nRuta:")
print(API_PACKAGE_V12)

print(
    "\n✅ techmind_api/ de v1.1 "
    "permanece intacta."
)

FASTAPI EXPERIMENTAL v1.2

Ruta:
C:\Users\MAMÁ\Downloads\techmind-v2\techmind_api_v12

✅ techmind_api/ de v1.1 permanece intacta.


In [128]:
# ============================================================
# CELDA 120 — __init__.py
# ============================================================

API_INIT_V12 = '''\
"""
TechMind API v1.2 experimental.
"""

__version__ = "1.2.0"

__all__ = [
    "__version__",
]
'''

(
    API_PACKAGE_V12
    / "__init__.py"
).write_text(
    API_INIT_V12,
    encoding="utf-8"
)

print(
    "✅ techmind_api_v12/__init__.py"
)

✅ techmind_api_v12/__init__.py


In [129]:
# ============================================================
# CELDA 121 — schemas.py
# ============================================================

SCHEMAS_V12 = '''\
from __future__ import annotations

from typing import Any

from pydantic import (
    BaseModel,
    Field,
)


class PredictRequest(BaseModel):

    texts: str | list[str] = Field(
        ...,
        description=(
            "Texto único o lista de textos "
            "para clasificar."
        ),
    )

    include_explanation: bool = Field(
        default=False,
        description=(
            "Incluye explicación diferencial "
            "basada en TF-IDF."
        ),
    )

    explanation_top_n: int = Field(
        default=8,
        ge=1,
        le=50,
        description=(
            "Número máximo de términos "
            "en la explicación."
        ),
    )

    top_k: int | None = Field(
        default=None,
        ge=1,
        le=4,
        description=(
            "Número de clases a devolver "
            "en el ranking."
        ),
    )


class PredictionItem(BaseModel):

    index: int

    text: Any

    valid_input: bool

    decision: str

    prediction: str | None

    second_category: str | None

    decision_margin: float | None

    domain_similarity_5nn: float | None

    tfidf_active_features: int

    reason: str | None

    score_top1: float | None = None

    score_top2: float | None = None

    top_k: list[dict[str, Any]] | None = None

    explanation: dict[str, Any] | None = None


class PredictResponse(BaseModel):

    model_version: str

    model_status: str

    n_inputs: int

    summary: dict[str, int]

    predictions: list[PredictionItem]
'''

(
    API_PACKAGE_V12
    / "schemas.py"
).write_text(
    SCHEMAS_V12,
    encoding="utf-8"
)

print(
    "✅ techmind_api_v12/schemas.py"
)

✅ techmind_api_v12/schemas.py


In [130]:
# ============================================================
# CELDA 122 — main.py
# ============================================================

MAIN_V12 = r'''\
from __future__ import annotations

from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any
import os

from fastapi import (
    FastAPI,
    HTTPException,
)

from techmind_v12 import (
    TechMindPredictor,
)

from .schemas import (
    PredictRequest,
    PredictResponse,
)


API_VERSION = "1.2.0"

MODEL_VERSION = (
    "1.2.0-multilingual"
)


def _resolve_root() -> Path:

    return (
        Path(__file__)
        .resolve()
        .parents[1]
    )


ROOT = _resolve_root()


DEFAULT_MODEL_PATH = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)


MODEL_PATH = Path(
    os.getenv(
        "TECHMIND_V12_MODEL_PATH",
        str(
            DEFAULT_MODEL_PATH
        )
    )
)


predictor: TechMindPredictor | None = None


@asynccontextmanager
async def lifespan(
    app: FastAPI,
):

    global predictor

    predictor = TechMindPredictor(
        MODEL_PATH
    )

    yield

    predictor = None


app = FastAPI(

    title=(
        "TechMind API"
    ),

    description=(
        "Experimental multilingual API "
        "for technical content classification. "
        "TechMind v1.2.0-multilingual."
    ),

    version=API_VERSION,

    lifespan=lifespan,
)


def _get_predictor() -> TechMindPredictor:

    if predictor is None:

        raise HTTPException(
            status_code=503,
            detail=(
                "Model is not loaded."
            ),
        )

    return predictor


@app.get("/")
def root() -> dict[str, Any]:

    return {
        "service":
            "TechMind API",

        "api_version":
            API_VERSION,

        "model_version":
            MODEL_VERSION,

        "status":
            "experimental",

        "docs":
            "/docs",

        "health":
            "/health",

        "model_info":
            "/model-info",
    }


@app.get("/health")
def health() -> dict[str, Any]:

    loaded = (
        predictor is not None
    )

    return {
        "status":
            (
                "ok"
                if loaded
                else "loading"
            ),

        "model_loaded":
            loaded,

        "api_version":
            API_VERSION,

        "model_version":
            (
                predictor.version
                if loaded
                else MODEL_VERSION
            ),
    }


@app.get("/model-info")
def model_info() -> dict[str, Any]:

    model = (
        _get_predictor()
    )

    info = (
        model.model_info()
    )

    return {
        "api_version":
            API_VERSION,

        **info,
    }


@app.post(
    "/predict",
    response_model=PredictResponse,
)
def predict(
    request: PredictRequest,
) -> dict[str, Any]:

    model = (
        _get_predictor()
    )

    try:

        return model.predict(

            request.texts,

            include_explanation=(
                request.include_explanation
            ),

            explanation_top_n=(
                request.explanation_top_n
            ),

            top_k=(
                request.top_k
            ),
        )

    except ValueError as exc:

        raise HTTPException(
            status_code=422,
            detail=str(exc),
        ) from exc

    except TypeError as exc:

        raise HTTPException(
            status_code=422,
            detail=str(exc),
        ) from exc
'''

(
    API_PACKAGE_V12
    / "main.py"
).write_text(
    MAIN_V12,
    encoding="utf-8"
)

print(
    "✅ techmind_api_v12/main.py"
)

✅ techmind_api_v12/main.py


In [136]:
# ============================================================
# CELDA 123R — FASTAPI v1.2
# REINICIO LIMPIO + LIFESPAN
# ============================================================

import importlib
import sys

from fastapi.testclient import TestClient


# ------------------------------------------------------------
# ROOT EN sys.path
# ------------------------------------------------------------

if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT)
    )


# ------------------------------------------------------------
# RECARGAR MÓDULO COMPLETO
# ------------------------------------------------------------

import techmind_api_v12.main as api_v12_main

api_v12_main = importlib.reload(
    api_v12_main
)

app_v12 = api_v12_main.app


print("=" * 70)
print("FASTAPI v1.2 — RELOAD LIMPIO")
print("=" * 70)

print(
    "\nApp:",
    app_v12.title
)

print(
    "API version:",
    app_v12.version
)

print(
    "Predictor ANTES del lifespan:",
    api_v12_main.predictor
)

FASTAPI v1.2 — RELOAD LIMPIO

App: TechMind API
API version: 1.2.0
Predictor ANTES del lifespan: None


In [137]:
# ============================================================
# CELDA 124R — TESTCLIENT CON LIFESPAN ACTIVO
# ============================================================

PAYLOAD_TESTCLIENT_V12 = {

    "texts": [

        (
            "Train a classification model "
            "and evaluate precision, recall "
            "and F1 score."
        ),

        (
            "Preparar una tortilla "
            "con patatas y huevos."
        ),
    ],

    "include_explanation": False,

    "top_k": 4,
}


print("=" * 70)
print("TESTCLIENT + LIFESPAN v1.2")
print("=" * 70)


with TestClient(
    app_v12
) as client_v12:

    # --------------------------------------------------------
    # CONFIRMAR QUE LIFESPAN CARGÓ EL PREDICTOR
    # --------------------------------------------------------

    PREDICTOR_LIFESPAN_OK = (
        api_v12_main.predictor
        is not None
    )

    print(
        "\nPredictor dentro del lifespan:",
        PREDICTOR_LIFESPAN_OK
    )


    if PREDICTOR_LIFESPAN_OK:

        print(
            "Model version:",
            api_v12_main.predictor.version
        )

        print(
            "Artifact SHA:",
            api_v12_main.predictor.artifact_sha256
        )


    # --------------------------------------------------------
    # ROOT
    # --------------------------------------------------------

    response_root_v12 = (
        client_v12.get("/")
    )


    # --------------------------------------------------------
    # HEALTH
    # --------------------------------------------------------

    response_health_v12 = (
        client_v12.get(
            "/health"
        )
    )


    # --------------------------------------------------------
    # MODEL INFO
    # --------------------------------------------------------

    response_info_v12 = (
        client_v12.get(
            "/model-info"
        )
    )


    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    response_predict_v12 = (
        client_v12.post(
            "/predict",
            json=PAYLOAD_TESTCLIENT_V12
        )
    )


    # --------------------------------------------------------
    # GUARDAR JSON ANTES DE SALIR DEL CONTEXTO
    # --------------------------------------------------------

    root_json_v12 = (
        response_root_v12.json()
    )

    health_json_v12 = (
        response_health_v12.json()
    )

    info_json_v12 = (
        response_info_v12.json()
    )

    predict_json_v12 = (
        response_predict_v12.json()
    )


    print("\nHTTP STATUS")
    print("-" * 70)

    print(
        "GET /:",
        response_root_v12.status_code
    )

    print(
        "GET /health:",
        response_health_v12.status_code
    )

    print(
        "GET /model-info:",
        response_info_v12.status_code
    )

    print(
        "POST /predict:",
        response_predict_v12.status_code
    )


    print("\n/health:")
    print(
        health_json_v12
    )


    print("\n/predict:")
    print(
        predict_json_v12
    )

TESTCLIENT + LIFESPAN v1.2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2150.63it/s]



Predictor dentro del lifespan: True
Model version: 1.2.0-multilingual
Artifact SHA: 1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61

HTTP STATUS
----------------------------------------------------------------------
GET /: 200
GET /health: 200
GET /model-info: 200
POST /predict: 200

/health:
{'status': 'ok', 'model_loaded': True, 'api_version': '1.2.0', 'model_version': '1.2.0-multilingual'}

/predict:
{'model_version': '1.2.0-multilingual', 'model_status': 'validated_experimental_candidate', 'n_inputs': 2, 'summary': {'accepted': 1, 'review': 0, 'rejected_ood': 1, 'rejected_invalid': 0}, 'predictions': [{'index': 0, 'text': 'Train a classification model and evaluate precision, recall and F1 score.', 'valid_input': True, 'decision': 'accepted', 'prediction': 'datascience', 'second_category': 'cloud', 'decision_margin': 1.9142611861383916, 'domain_similarity_5nn': 0.45553335547447205, 'tfidf_active_features': 161, 'reason': None, 'score_top1': 1.0572208742742033, 'sco

In [138]:
# ============================================================
# CELDA 125R — DIAGNÓSTICO POST TESTCLIENT
# ============================================================

predictions_v12 = (
    predict_json_v12.get(
        "predictions",
        []
    )
)


print("=" * 70)
print("DIAGNÓSTICO /PREDICT v1.2")
print("=" * 70)

print(
    "\nHTTP:",
    response_predict_v12.status_code
)

print(
    "n_inputs:",
    predict_json_v12.get(
        "n_inputs"
    )
)

print(
    "Número de predicciones:",
    len(
        predictions_v12
    )
)


for i, item in enumerate(
    predictions_v12
):

    print("\n" + "-" * 70)

    print(
        "Index:",
        i
    )

    print(
        "Texto:",
        item.get("text")
    )

    print(
        "Prediction:",
        item.get("prediction")
    )

    print(
        "Decision:",
        item.get("decision")
    )

    print(
        "Domain similarity:",
        item.get(
            "domain_similarity_5nn"
        )
    )

    print(
        "Margin:",
        item.get(
            "decision_margin"
        )
    )

DIAGNÓSTICO /PREDICT v1.2

HTTP: 200
n_inputs: 2
Número de predicciones: 2

----------------------------------------------------------------------
Index: 0
Texto: Train a classification model and evaluate precision, recall and F1 score.
Prediction: datascience
Decision: accepted
Domain similarity: 0.45553335547447205
Margin: 1.9142611861383916

----------------------------------------------------------------------
Index: 1
Texto: Preparar una tortilla con patatas y huevos.
Prediction: backend
Decision: rejected_ood
Domain similarity: 0.29467248916625977
Margin: 0.071775098764564


In [139]:
# ============================================================
# CELDA 126R — VALIDACIÓN AUTOMÁTICA FASTAPI
# ============================================================

TIENE_DOS_PREDICCIONES = (
    len(
        predictions_v12
    )
    == 2
)


if TIENE_DOS_PREDICCIONES:

    tech_decision = (
        predictions_v12[
            0
        ].get(
            "decision"
        )
    )

    cocina_decision = (
        predictions_v12[
            1
        ].get(
            "decision"
        )
    )

else:

    tech_decision = None
    cocina_decision = None


api_checks = {

    "lifespan_predictor":
        PREDICTOR_LIFESPAN_OK,

    "root_200":
        response_root_v12.status_code
        == 200,

    "health_200":
        response_health_v12.status_code
        == 200,

    "model_info_200":
        response_info_v12.status_code
        == 200,

    "predict_200":
        response_predict_v12.status_code
        == 200,

    "health_ok":
        health_json_v12.get(
            "status"
        )
        == "ok",

    "model_loaded":
        health_json_v12.get(
            "model_loaded"
        )
        is True,

    "version_correcta":
        info_json_v12.get(
            "version"
        )
        ==
        "1.2.0-multilingual",

    "dos_inputs":
        predict_json_v12.get(
            "n_inputs"
        )
        == 2,

    "dos_predicciones":
        TIENE_DOS_PREDICCIONES,

    "tech_no_ood":
        (
            TIENE_DOS_PREDICCIONES
            and
            tech_decision
            != "rejected_ood"
        ),

    "cocina_ood":
        (
            TIENE_DOS_PREDICCIONES
            and
            cocina_decision
            == "rejected_ood"
        ),
}


print("=" * 70)
print("VALIDACIÓN FASTAPI v1.2")
print("=" * 70)


for nombre, valor in (
    api_checks.items()
):

    print(
        f"{'✅' if valor else '❌'} "
        f"{nombre}: {valor}"
    )


FASTAPI_V12_OK = all(
    api_checks.values()
)


print("\n" + "=" * 70)

print(
    "FASTAPI LISTA:",
    FASTAPI_V12_OK
)

print("=" * 70)

VALIDACIÓN FASTAPI v1.2
✅ lifespan_predictor: True
✅ root_200: True
✅ health_200: True
✅ model_info_200: True
✅ predict_200: True
✅ health_ok: True
✅ model_loaded: True
✅ version_correcta: True
✅ dos_inputs: True
✅ dos_predicciones: True
✅ tech_no_ood: True
✅ cocina_ood: True

FASTAPI LISTA: True


In [140]:
# ============================================================
# CELDA 127 — OPENAPI
# ============================================================

r_openapi = client.get(
    "/openapi.json"
)

openapi_json = (
    r_openapi.json()
)


print("=" * 70)
print("OPENAPI v1.2")
print("=" * 70)

print(
    "\nStatus:",
    r_openapi.status_code
)

print(
    "Title:",
    openapi_json[
        "info"
    ][
        "title"
    ]
)

print(
    "Version:",
    openapi_json[
        "info"
    ][
        "version"
    ]
)

print(
    "\nEndpoints:"
)

for path in (
    openapi_json[
        "paths"
    ]
):

    print(
        " -",
        path
    )

OPENAPI v1.2

Status: 200
Title: TechMind API
Version: 1.2.0

Endpoints:
 - /
 - /health
 - /model-info
 - /predict


In [141]:
# ============================================================
# CELDA 128 — IMPORT INDEPENDIENTE
# ============================================================

import subprocess
import sys

codigo_import = r"""
from techmind_v12 import TechMindPredictor
from techmind_api_v12.main import app

print("IMPORT_OK")
print("APP_TITLE:", app.title)
print("APP_VERSION:", app.version)
"""

resultado_import = subprocess.run(
    [
        sys.executable,
        "-c",
        codigo_import
    ],
    cwd=ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print("=" * 70)
print("IMPORT INDEPENDIENTE — v1.2")
print("=" * 70)

print("\nCódigo de salida:")
print(resultado_import.returncode)

print("\nSTDOUT:")
print(resultado_import.stdout)

if resultado_import.stderr.strip():
    print("\nSTDERR:")
    print(resultado_import.stderr)

IMPORT_INDEPENDIENTE_OK = (
    resultado_import.returncode == 0
    and
    "IMPORT_OK" in resultado_import.stdout
)

print(
    "\nIMPORT INDEPENDIENTE OK:",
    IMPORT_INDEPENDIENTE_OK
)

assert IMPORT_INDEPENDIENTE_OK

IMPORT INDEPENDIENTE — v1.2

Código de salida:
0

STDOUT:
IMPORT_OK
APP_TITLE: TechMind API
APP_VERSION: 1.2.0


IMPORT INDEPENDIENTE OK: True


In [142]:
# ============================================================
# CELDA 128B — CARGA INDEPENDIENTE DEL PREDICTOR
# ============================================================

import subprocess
import sys
import os

codigo_predictor_independiente = r'''
import time

from pathlib import Path
from techmind_v12 import TechMindPredictor

ROOT = Path.cwd()

MODEL_PATH = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)

print(
    "PYTHON_START",
    flush=True
)

print(
    "MODEL_EXISTS:",
    MODEL_PATH.exists(),
    flush=True
)

inicio = time.perf_counter()

predictor = TechMindPredictor(
    MODEL_PATH
)

duracion = (
    time.perf_counter()
    -
    inicio
)

print(
    "PREDICTOR_LOADED",
    flush=True
)

print(
    "VERSION:",
    predictor.version,
    flush=True
)

print(
    "SHA:",
    predictor.artifact_sha256,
    flush=True
)

print(
    "LOAD_SECONDS:",
    round(duracion, 3),
    flush=True
)
'''

env_test = os.environ.copy()

# Forzamos comportamiento completamente offline
env_test[
    "HF_HUB_OFFLINE"
] = "1"

env_test[
    "TRANSFORMERS_OFFLINE"
] = "1"

env_test[
    "HF_HUB_DISABLE_TELEMETRY"
] = "1"


resultado_predictor_ind = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        codigo_predictor_independiente
    ],
    cwd=ROOT,
    env=env_test,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=120
)


print("=" * 70)
print("CARGA INDEPENDIENTE — PREDICTOR v1.2")
print("=" * 70)

print(
    "\nCódigo:",
    resultado_predictor_ind.returncode
)

print(
    "\nSTDOUT:"
)

print(
    resultado_predictor_ind.stdout
)

if (
    resultado_predictor_ind.stderr.strip()
):

    print(
        "\nSTDERR:"
    )

    print(
        resultado_predictor_ind.stderr
    )


PREDICTOR_SUBPROCESS_OK = (
    resultado_predictor_ind.returncode
    == 0
    and
    "PREDICTOR_LOADED"
    in resultado_predictor_ind.stdout
)


print(
    "\nPREDICTOR SUBPROCESS OK:",
    PREDICTOR_SUBPROCESS_OK
)

CARGA INDEPENDIENTE — PREDICTOR v1.2

Código: 0

STDOUT:
PYTHON_START
MODEL_EXISTS: True
PREDICTOR_LOADED
VERSION: 1.2.0-multilingual
SHA: 1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61
LOAD_SECONDS: 2.722


STDERR:

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7960.65it/s]


PREDICTOR SUBPROCESS OK: True


In [143]:
# ============================================================
# CELDA 128C — INFERENCIA EN PROCESO INDEPENDIENTE
# ============================================================

codigo_inferencia_independiente = r'''
from pathlib import Path

from techmind_v12 import TechMindPredictor

ROOT = Path.cwd()

MODEL_PATH = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)

predictor = TechMindPredictor(
    MODEL_PATH
)

resultado = predictor.predict(
    [
        (
            "Train a classification model "
            "and evaluate precision recall and F1."
        ),
        (
            "Preparar una pizza con queso "
            "y tomate."
        )
    ]
)

print(
    "INFERENCE_OK",
    flush=True
)

for item in resultado["predictions"]:

    print(
        item["prediction"],
        item["decision"],
        round(
            item["domain_similarity_5nn"]
            or 0,
            4
        ),
        flush=True
    )
'''

resultado_inferencia_ind = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        codigo_inferencia_independiente
    ],
    cwd=ROOT,
    env=env_test,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=120
)


print("=" * 70)
print("INFERENCIA INDEPENDIENTE")
print("=" * 70)

print(
    "\nCódigo:",
    resultado_inferencia_ind.returncode
)

print(
    "\nSTDOUT:"
)

print(
    resultado_inferencia_ind.stdout
)

if (
    resultado_inferencia_ind.stderr.strip()
):

    print(
        "\nSTDERR:"
    )

    print(
        resultado_inferencia_ind.stderr
    )


INFERENCE_SUBPROCESS_OK = (
    resultado_inferencia_ind.returncode
    == 0
    and
    "INFERENCE_OK"
    in resultado_inferencia_ind.stdout
)


print(
    "\nINFERENCE SUBPROCESS OK:",
    INFERENCE_SUBPROCESS_OK
)

INFERENCIA INDEPENDIENTE

Código: 0

STDOUT:
INFERENCE_OK
datascience accepted 0.4593
datascience rejected_ood 0.2367


STDERR:

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6939.07it/s]


INFERENCE SUBPROCESS OK: True


In [144]:
# ============================================================
# CELDA 129 — CONFIGURACIÓN UVICORN
# ============================================================

import sys
import subprocess
import time
import json
import urllib.request
import urllib.error

HOST_SMOKE = "127.0.0.1"
PORT_SMOKE = 8765

BASE_URL_SMOKE = (
    f"http://{HOST_SMOKE}:{PORT_SMOKE}"
)

print("=" * 70)
print("CONFIGURACIÓN SMOKE HTTP")
print("=" * 70)

print("\nPython:")
print(sys.executable)

print("\nROOT:")
print(ROOT)

print("\nURL:")
print(BASE_URL_SMOKE)

CONFIGURACIÓN SMOKE HTTP

Python:
C:\Users\MAMÁ\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe

ROOT:
C:\Users\MAMÁ\Downloads\techmind-v2

URL:
http://127.0.0.1:8765


In [145]:
# ============================================================
# CELDA 129B — LOG UVICORN
# ============================================================

UVICORN_LOG_PATH = (
    ROOT
    / "experiments"
    / "v1.2.0-multilingual"
    / "reports"
    / "uvicorn_smoke_test.log"
)

UVICORN_LOG_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Log:",
    UVICORN_LOG_PATH
)

Log: C:\Users\MAMÁ\Downloads\techmind-v2\experiments\v1.2.0-multilingual\reports\uvicorn_smoke_test.log


In [146]:
# ============================================================
# CELDA 130 — UVICORN INDEPENDIENTE CORREGIDO
# ============================================================

import os
import subprocess
import sys


HOST_SMOKE = "127.0.0.1"
PORT_SMOKE = 8765

BASE_URL_SMOKE = (
    f"http://{HOST_SMOKE}:{PORT_SMOKE}"
)


env_uvicorn = os.environ.copy()

# El proceso independiente NO debe consultar Hugging Face.
env_uvicorn[
    "HF_HUB_OFFLINE"
] = "1"

env_uvicorn[
    "TRANSFORMERS_OFFLINE"
] = "1"

env_uvicorn[
    "HF_HUB_DISABLE_TELEMETRY"
] = "1"

# También evitamos paralelismo innecesario.
env_uvicorn[
    "TOKENIZERS_PARALLELISM"
] = "false"


comando_uvicorn = [

    sys.executable,

    "-u",

    "-m",
    "uvicorn",

    "techmind_api_v12.main:app",

    "--host",
    HOST_SMOKE,

    "--port",
    str(
        PORT_SMOKE
    ),

    "--log-level",
    "info",
]


uvicorn_log_file = open(
    UVICORN_LOG_PATH,
    "w",
    encoding="utf-8"
)


proceso_uvicorn = subprocess.Popen(

    comando_uvicorn,

    cwd=ROOT,

    env=env_uvicorn,

    stdout=uvicorn_log_file,

    stderr=subprocess.STDOUT,

    text=True
)


print("=" * 70)
print("UVICORN — PROCESO INDEPENDIENTE")
print("=" * 70)

print(
    "\nPID:",
    proceso_uvicorn.pid
)

print(
    "Proceso vivo:",
    proceso_uvicorn.poll()
    is None
)

print(
    "Log:",
    UVICORN_LOG_PATH
)

UVICORN — PROCESO INDEPENDIENTE

PID: 25004
Proceso vivo: True
Log: C:\Users\MAMÁ\Downloads\techmind-v2\experiments\v1.2.0-multilingual\reports\uvicorn_smoke_test.log


In [147]:
# ============================================================
# CELDA 131 — HEALTH CHECK CORREGIDO
# ============================================================

import json
import time
import urllib.request


health_response = None
health_error = None

MAX_INTENTOS = 120


for intento in range(
    1,
    MAX_INTENTOS + 1
):

    # -----------------------------------------
    # El proceso murió
    # -----------------------------------------

    if (
        proceso_uvicorn.poll()
        is not None
    ):
        break

    # -----------------------------------------
    # Intentar conectar
    # -----------------------------------------

    try:

        with urllib.request.urlopen(
            f"{BASE_URL_SMOKE}/health",
            timeout=0.5
        ) as response:

            health_response = json.loads(
                response
                .read()
                .decode(
                    "utf-8"
                )
            )

        if (
            health_response.get(
                "status"
            )
            == "ok"
            and
            health_response.get(
                "model_loaded"
            )
            is True
        ):

            break

    except Exception as exc:

        health_error = exc

    time.sleep(
        0.25
    )


HEALTH_INDEPENDIENTE_OK = (

    health_response
    is not None

    and

    health_response.get(
        "status"
    )
    == "ok"

    and

    health_response.get(
        "model_loaded"
    )
    is True
)


print("=" * 70)
print("HEALTH — PROCESO INDEPENDIENTE")
print("=" * 70)

print(
    "\nProceso vivo:",
    proceso_uvicorn.poll()
    is None
)

print(
    "Return code:",
    proceso_uvicorn.poll()
)

print(
    "\nRespuesta:"
)

print(
    json.dumps(
        health_response,
        indent=2,
        ensure_ascii=False
    )
    if health_response
    else "Sin respuesta"
)

print(
    "\nHEALTH OK:",
    HEALTH_INDEPENDIENTE_OK
)


# ------------------------------------------------------------
# MOSTRAR LOG
# ------------------------------------------------------------

uvicorn_log_file.flush()

print(
    "\n" + "=" * 70
)

print(
    "LOG UVICORN"
)

print(
    "=" * 70
)


if (
    UVICORN_LOG_PATH.exists()
):

    contenido_log = (
        UVICORN_LOG_PATH
        .read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

    print(
        contenido_log[
            -8000:
        ]
    )

HEALTH — PROCESO INDEPENDIENTE

Proceso vivo: True
Return code: None

Respuesta:
{
  "status": "ok",
  "model_loaded": true,
  "api_version": "1.2.0",
  "model_version": "1.2.0-multilingual"
}

HEALTH OK: True

LOG UVICORN
INFO:     Started server process [25004]
INFO:     Waiting for application startup.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8652.17it/s]
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)
INFO:     127.0.0.1:49688 - "GET /health HTTP/1.1" 200 OK



In [148]:
# ============================================================
# CELDA 132 — MODEL INFO HTTP REAL
# VERSIÓN ROBUSTA
# ============================================================

import json
import hashlib
import urllib.request


# ------------------------------------------------------------
# 1. CONSULTAR API
# ------------------------------------------------------------

with urllib.request.urlopen(
    f"{BASE_URL_SMOKE}/model-info",
    timeout=10
) as response:

    info_http = json.loads(
        response.read().decode(
            "utf-8"
        )
    )


print("=" * 70)
print("/MODEL-INFO — HTTP REAL")
print("=" * 70)

print(
    json.dumps(
        info_http,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# 2. RECALCULAR SHA DEL ARTEFACTO LOCAL
# ------------------------------------------------------------

MODEL_PATH_V12_LOAD = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)


sha256 = hashlib.sha256()

with open(
    MODEL_PATH_V12_LOAD,
    "rb"
) as f:

    for bloque in iter(
        lambda: f.read(
            1024 * 1024
        ),
        b""
    ):

        sha256.update(
            bloque
        )


MODEL_SHA256_V12 = (
    sha256.hexdigest()
)


# ------------------------------------------------------------
# 3. VALIDACIONES
# ------------------------------------------------------------

SHA_HTTP_OK = (
    info_http[
        "artifact_sha256"
    ]
    ==
    MODEL_SHA256_V12
)


VERSION_HTTP_OK = (
    info_http[
        "version"
    ]
    ==
    "1.2.0-multilingual"
)


STATUS_HTTP_OK = (
    info_http[
        "status"
    ]
    ==
    "validated_experimental_candidate"
)


print("\n" + "=" * 70)
print("VALIDACIÓN /MODEL-INFO")
print("=" * 70)

print(
    "\nSHA local:"
)
print(
    MODEL_SHA256_V12
)

print(
    "\nSHA API:"
)
print(
    info_http[
        "artifact_sha256"
    ]
)

print(
    "\nSHA correcto:",
    SHA_HTTP_OK
)

print(
    "Versión correcta:",
    VERSION_HTTP_OK
)

print(
    "Status correcto:",
    STATUS_HTTP_OK
)


assert SHA_HTTP_OK
assert VERSION_HTTP_OK
assert STATUS_HTTP_OK


print(
    "\n✅ /model-info validado "
    "contra el artefacto local."
)

/MODEL-INFO — HTTP REAL
{
  "api_version": "1.2.0",
  "version": "1.2.0-multilingual",
  "status": "validated_experimental_candidate",
  "architecture": "TF-IDF Word+Char + MiniLM 384 + LinearSVC",
  "classifier": "LinearSVC",
  "classifier_C": 0.3,
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "classes": [
    "backend",
    "cloud",
    "datascience",
    "frontend"
  ],
  "domain_control": {
    "metric": "mean_cosine_similarity_5nn",
    "threshold": 0.4266,
    "n_neighbors": 5
  },
  "confidence_control": {
    "metric": "top1_minus_top2_decision_margin",
    "threshold": 0.8132
  },
  "artifact_sha256": "1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61",
  "scores_are_probabilities": false
}

VALIDACIÓN /MODEL-INFO

SHA local:
1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61

SHA API:
1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61

SHA correcto: True
Versión co

In [149]:
# ============================================================
# CELDA 133 — PREDICT HTTP REAL
# ============================================================

import json
import urllib.request

payload_http = {

    "texts": [

        # Data Science — EN
        (
            "Train a gradient boosting model "
            "and evaluate precision, recall "
            "and F1 score."
        ),

        # Backend — ES
        (
            "Crear un endpoint REST con "
            "autenticación, validación de "
            "solicitudes y acceso a base de datos."
        ),

        # Data Science — RU
        (
            "Обучить модель классификации "
            "и оценить качество с помощью "
            "precision, recall и F1."
        ),

        # OOD — ES
        (
            "Preparar una pizza con tomate, "
            "queso y aceite de oliva."
        )
    ],

    "include_explanation": False,

    "top_k": 4
}


datos_http = json.dumps(
    payload_http,
    ensure_ascii=False
).encode(
    "utf-8"
)


request_http = urllib.request.Request(

    f"{BASE_URL_SMOKE}/predict",

    data=datos_http,

    headers={
        "Content-Type":
            "application/json; charset=utf-8"
    },

    method="POST"
)


with urllib.request.urlopen(
    request_http,
    timeout=30
) as response:

    predict_http = json.loads(
        response.read().decode(
            "utf-8"
        )
    )


print("=" * 70)
print("/PREDICT — HTTP REAL")
print("=" * 70)

print(
    "\nModel:",
    predict_http[
        "model_version"
    ]
)

print(
    "Status:",
    predict_http[
        "model_status"
    ]
)

print(
    "Summary:",
    predict_http[
        "summary"
    ]
)


for item in predict_http["predictions"]:

    print("\n" + "-" * 70)

    print(
        "Index:",
        item["index"]
    )

    print(
        "Texto:",
        item["text"]
    )

    print(
        "Prediction:",
        item["prediction"]
    )

    print(
        "Decision:",
        item["decision"]
    )

    print(
        "Second:",
        item["second_category"]
    )

    print(
        "Margin:",
        (
            round(
                item["decision_margin"],
                4
            )
            if item["decision_margin"]
            is not None
            else None
        )
    )

    print(
        "Domain 5NN:",
        (
            round(
                item["domain_similarity_5nn"],
                4
            )
            if item["domain_similarity_5nn"]
            is not None
            else None
        )
    )

    print(
        "Reason:",
        item["reason"]
    )

/PREDICT — HTTP REAL

Model: 1.2.0-multilingual
Status: validated_experimental_candidate
Summary: {'accepted': 3, 'review': 0, 'rejected_ood': 1, 'rejected_invalid': 0}

----------------------------------------------------------------------
Index: 0
Texto: Train a gradient boosting model and evaluate precision, recall and F1 score.
Prediction: datascience
Decision: accepted
Second: cloud
Margin: 1.1989
Domain 5NN: 0.4655
Reason: None

----------------------------------------------------------------------
Index: 1
Texto: Crear un endpoint REST con autenticación, validación de solicitudes y acceso a base de datos.
Prediction: backend
Decision: accepted
Second: cloud
Margin: 1.122
Domain 5NN: 0.5056
Reason: None

----------------------------------------------------------------------
Index: 2
Texto: Обучить модель классификации и оценить качество с помощью precision, recall и F1.
Prediction: datascience
Decision: accepted
Second: backend
Margin: 1.3716
Domain 5NN: 0.4525
Reason: None

----

In [150]:
# ============================================================
# CELDA 134 — VALIDACIÓN HTTP INDEPENDIENTE
# ============================================================

preds_http = (
    predict_http[
        "predictions"
    ]
)


http_checks = {

    "modelo_correcto":
        predict_http[
            "model_version"
        ]
        ==
        "1.2.0-multilingual",

    "4_resultados":
        len(
            preds_http
        )
        == 4,

    "datascience_en_pred":
        preds_http[
            0
        ][
            "prediction"
        ]
        ==
        "datascience",

    "datascience_en_no_ood":
        preds_http[
            0
        ][
            "decision"
        ]
        != "rejected_ood",

    "backend_es_pred":
        preds_http[
            1
        ][
            "prediction"
        ]
        ==
        "backend",

    "backend_es_no_ood":
        preds_http[
            1
        ][
            "decision"
        ]
        != "rejected_ood",

    "datascience_ru_pred":
        preds_http[
            2
        ][
            "prediction"
        ]
        ==
        "datascience",

    "datascience_ru_no_ood":
        preds_http[
            2
        ][
            "decision"
        ]
        != "rejected_ood",

    "pizza_ood":
        preds_http[
            3
        ][
            "decision"
        ]
        ==
        "rejected_ood",
}


print("=" * 70)
print("VALIDACIÓN HTTP INDEPENDIENTE")
print("=" * 70)


for nombre, valor in (
    http_checks.items()
):

    print(
        f"{'✅' if valor else '❌'} "
        f"{nombre}: {valor}"
    )


HTTP_INDEPENDIENTE_OK = all(
    http_checks.values()
)


print(
    "\nHTTP INDEPENDIENTE OK:",
    HTTP_INDEPENDIENTE_OK
)

VALIDACIÓN HTTP INDEPENDIENTE
✅ modelo_correcto: True
✅ 4_resultados: True
✅ datascience_en_pred: True
✅ datascience_en_no_ood: True
✅ backend_es_pred: True
✅ backend_es_no_ood: True
✅ datascience_ru_pred: True
✅ datascience_ru_no_ood: True
✅ pizza_ood: True

HTTP INDEPENDIENTE OK: True


In [151]:
# ============================================================
# CELDA 135 — OPENAPI HTTP REAL
# ============================================================

with urllib.request.urlopen(
    f"{BASE_URL_SMOKE}/openapi.json",
    timeout=10
) as response:

    openapi_http = json.loads(
        response.read().decode(
            "utf-8"
        )
    )


endpoints_http = sorted(
    openapi_http[
        "paths"
    ].keys()
)


print("=" * 70)
print("OPENAPI — PROCESO INDEPENDIENTE")
print("=" * 70)

print(
    "\nTitle:",
    openapi_http[
        "info"
    ][
        "title"
    ]
)

print(
    "Version:",
    openapi_http[
        "info"
    ][
        "version"
    ]
)

print(
    "\nEndpoints:"
)

for endpoint in endpoints_http:

    print(
        " -",
        endpoint
    )


endpoints_esperados = {
    "/",
    "/health",
    "/model-info",
    "/predict",
}


OPENAPI_HTTP_OK = (
    endpoints_esperados
    .issubset(
        set(
            endpoints_http
        )
    )
)


print(
    "\nOPENAPI OK:",
    OPENAPI_HTTP_OK
)

assert OPENAPI_HTTP_OK

OPENAPI — PROCESO INDEPENDIENTE

Title: TechMind API
Version: 1.2.0

Endpoints:
 - /
 - /health
 - /model-info
 - /predict

OPENAPI OK: True


In [152]:
# ============================================================
# CELDA 136 — CERRAR UVICORN
# ============================================================

print("=" * 70)
print("CIERRE UVICORN")
print("=" * 70)


if (
    proceso_uvicorn.poll()
    is None
):

    print(
        "\nCerrando PID:",
        proceso_uvicorn.pid
    )

    proceso_uvicorn.terminate()

    try:

        proceso_uvicorn.wait(
            timeout=10
        )

    except subprocess.TimeoutExpired:

        print(
            "Terminate no fue suficiente. "
            "Aplicando kill..."
        )

        proceso_uvicorn.kill()

        proceso_uvicorn.wait(
            timeout=10
        )


if (
    "uvicorn_log_file"
    in globals()
    and
    not uvicorn_log_file.closed
):

    uvicorn_log_file.flush()
    uvicorn_log_file.close()


print(
    "\nCódigo de salida:",
    proceso_uvicorn.returncode
)

print(
    "Proceso cerrado:",
    proceso_uvicorn.poll()
    is not None
)

CIERRE UVICORN

Cerrando PID: 25004

Código de salida: 1
Proceso cerrado: True


In [153]:
# ============================================================
# CELDA 137 — CERTIFICACIÓN BACKEND v1.2
# VERSIÓN ROBUSTA / AUTOCONTENIDA
# ============================================================

from pathlib import Path
import hashlib


print("=" * 70)
print("TECHMIND v1.2 — BACKEND VALIDATION")
print("=" * 70)


# ============================================================
# 1. SHA DEL ARTEFACTO LOCAL
# ============================================================

MODEL_PATH_V12_CERT = (
    ROOT
    / "models"
    / "experimental"
    / "v1.2.0-multilingual"
    / "techmind_hybrid_v1_2_0_multilingual.joblib"
)


sha_cert = hashlib.sha256()

with open(
    MODEL_PATH_V12_CERT,
    "rb"
) as f:

    for bloque in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha_cert.update(bloque)


MODEL_SHA256_CERT = (
    sha_cert.hexdigest()
)


EXPECTED_SHA_V12 = (
    "1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61"
)


ARTIFACT_SHA_OK = (
    MODEL_SHA256_CERT
    ==
    EXPECTED_SHA_V12
)


# ============================================================
# 2. MODEL-INFO HTTP
# ============================================================

HTTP_MODEL_INFO_OK = (
    "info_http" in globals()
    and
    info_http.get(
        "artifact_sha256"
    )
    ==
    MODEL_SHA256_CERT
    and
    info_http.get(
        "version"
    )
    ==
    "1.2.0-multilingual"
    and
    info_http.get(
        "status"
    )
    ==
    "validated_experimental_candidate"
)


# ============================================================
# 3. HEALTH INDEPENDIENTE
# ============================================================

INDEPENDENT_HEALTH_OK = (
    "health_response" in globals()
    and
    health_response is not None
    and
    health_response.get(
        "status"
    )
    ==
    "ok"
    and
    health_response.get(
        "model_loaded"
    )
    is True
)


# ============================================================
# 4. IMPORT INDEPENDIENTE
# ============================================================

INDEPENDENT_IMPORT_OK = (
    "resultado_import" in globals()
    and
    resultado_import.returncode == 0
    and
    "IMPORT_OK"
    in resultado_import.stdout
)


# ============================================================
# 5. CARGA DEL PREDICTOR EN SUBPROCESO
# ============================================================

INDEPENDENT_PREDICTOR_OK = (
    "resultado_predictor_ind" in globals()
    and
    resultado_predictor_ind.returncode == 0
    and
    "PREDICTOR_LOADED"
    in resultado_predictor_ind.stdout
)


# ============================================================
# 6. INFERENCIA EN SUBPROCESO
# ============================================================

INDEPENDENT_INFERENCE_OK = (
    "resultado_inferencia_ind" in globals()
    and
    resultado_inferencia_ind.returncode == 0
    and
    "INFERENCE_OK"
    in resultado_inferencia_ind.stdout
)


# ============================================================
# 7. PREDICTOR DEL NOTEBOOK
# ============================================================

PREDICTOR_NOTEBOOK_OK = (
    "predictor_v12" in globals()
    and
    predictor_v12.version
    ==
    "1.2.0-multilingual"
    and
    predictor_v12.artifact_sha256
    ==
    MODEL_SHA256_CERT
)


# ============================================================
# 8. FASTAPI TESTCLIENT
# ============================================================

FASTAPI_TESTCLIENT_OK = (
    "r_root" in globals()
    and
    "r_health" in globals()
    and
    "r_info" in globals()
    and
    "r_predict" in globals()
    and
    r_root.status_code == 200
    and
    r_health.status_code == 200
    and
    r_info.status_code == 200
    and
    r_predict.status_code == 200
)


# ============================================================
# 9. HTTP /PREDICT INDEPENDIENTE
# ============================================================

if (
    "predict_http" in globals()
    and
    isinstance(
        predict_http,
        dict
    )
):

    preds_cert = (
        predict_http.get(
            "predictions",
            []
        )
    )

    INDEPENDENT_HTTP_PREDICT_OK = (
        predict_http.get(
            "model_version"
        )
        ==
        "1.2.0-multilingual"
        and
        len(preds_cert) == 4
        and
        preds_cert[0].get(
            "decision"
        )
        != "rejected_ood"
        and
        preds_cert[1].get(
            "decision"
        )
        != "rejected_ood"
        and
        preds_cert[2].get(
            "decision"
        )
        != "rejected_ood"
        and
        preds_cert[3].get(
            "decision"
        )
        ==
        "rejected_ood"
    )

else:

    INDEPENDENT_HTTP_PREDICT_OK = False


# ============================================================
# 10. OPENAPI
# ============================================================

if (
    "openapi_http" in globals()
    and
    isinstance(
        openapi_http,
        dict
    )
):

    paths_cert = set(
        openapi_http.get(
            "paths",
            {}
        ).keys()
    )

    OPENAPI_CERT_OK = {
        "/",
        "/health",
        "/model-info",
        "/predict",
    }.issubset(
        paths_cert
    )

else:

    OPENAPI_CERT_OK = False


# ============================================================
# CONSOLIDACIÓN
# ============================================================

BACKEND_V12_CHECKS = {

    "artifact_sha":
        ARTIFACT_SHA_OK,

    "predictor_notebook":
        PREDICTOR_NOTEBOOK_OK,

    "fastapi_testclient":
        FASTAPI_TESTCLIENT_OK,

    "independent_import":
        INDEPENDENT_IMPORT_OK,

    "independent_predictor_load":
        INDEPENDENT_PREDICTOR_OK,

    "independent_inference":
        INDEPENDENT_INFERENCE_OK,

    "independent_health":
        INDEPENDENT_HEALTH_OK,

    "http_model_info":
        HTTP_MODEL_INFO_OK,

    "independent_http_predict":
        INDEPENDENT_HTTP_PREDICT_OK,

    "openapi":
        OPENAPI_CERT_OK,
}


for nombre, valor in (
    BACKEND_V12_CHECKS.items()
):

    print(
        f"{'✅' if valor else '❌'} "
        f"{nombre}: {valor}"
    )


BACKEND_V12_READY = all(
    BACKEND_V12_CHECKS.values()
)


print("\n" + "=" * 70)

print(
    "BACKEND v1.2 READY:",
    BACKEND_V12_READY
)

print("=" * 70)


print("\nSHA certificado:")
print(
    MODEL_SHA256_CERT
)

TECHMIND v1.2 — BACKEND VALIDATION
❌ artifact_sha: False
✅ predictor_notebook: True
❌ fastapi_testclient: False
✅ independent_import: True
✅ independent_predictor_load: True
✅ independent_inference: True
✅ independent_health: True
✅ http_model_info: True
✅ independent_http_predict: True
✅ openapi: True

BACKEND v1.2 READY: False

SHA certificado:
1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61


In [154]:
# ============================================================
# CELDA 138 — CERTIFICACIÓN DEFINITIVA BACKEND v1.2
# ============================================================

print("=" * 70)
print("TECHMIND v1.2 — FINAL BACKEND CERTIFICATION")
print("=" * 70)


# ============================================================
# CORE — PRUEBAS REALES DE DESPLIEGUE
# ============================================================

CORE_DEPLOYMENT_CHECKS = {

    "artifact_integrity":
        ARTIFACT_SHA_OK,

    "independent_import":
        INDEPENDENT_IMPORT_OK,

    "independent_predictor_load":
        INDEPENDENT_PREDICTOR_OK,

    "independent_inference":
        INDEPENDENT_INFERENCE_OK,

    "uvicorn_startup_health":
        INDEPENDENT_HEALTH_OK,

    "http_model_info":
        HTTP_MODEL_INFO_OK,

    "http_predict":
        INDEPENDENT_HTTP_PREDICT_OK,

    "openapi_contract":
        OPENAPI_CERT_OK,
}


# ============================================================
# DEVELOPMENT — PRUEBAS DEL NOTEBOOK
# ============================================================

DEVELOPMENT_CHECKS = {

    "predictor_notebook":
        PREDICTOR_NOTEBOOK_OK,

    "fastapi_testclient":
        FASTAPI_TESTCLIENT_OK,
}


print("\nCORE DEPLOYMENT CHECKS")
print("-" * 70)

for nombre, valor in (
    CORE_DEPLOYMENT_CHECKS.items()
):

    print(
        f"{'✅' if valor else '❌'} "
        f"{nombre}: {valor}"
    )


print("\nDEVELOPMENT / NOTEBOOK CHECKS")
print("-" * 70)

for nombre, valor in (
    DEVELOPMENT_CHECKS.items()
):

    print(
        f"{'✅' if valor else '⚠️'} "
        f"{nombre}: {valor}"
    )


# ============================================================
# RESULTADO
# ============================================================

BACKEND_V12_READY = all(
    CORE_DEPLOYMENT_CHECKS.values()
)


print("\n" + "=" * 70)

print(
    "BACKEND v1.2 READY FOR DEPLOYMENT:",
    BACKEND_V12_READY
)

print("=" * 70)


print("\nModel SHA256:")
print(
    MODEL_SHA256_CERT
)

TECHMIND v1.2 — FINAL BACKEND CERTIFICATION

CORE DEPLOYMENT CHECKS
----------------------------------------------------------------------
❌ artifact_integrity: False
✅ independent_import: True
✅ independent_predictor_load: True
✅ independent_inference: True
✅ uvicorn_startup_health: True
✅ http_model_info: True
✅ http_predict: True
✅ openapi_contract: True

DEVELOPMENT / NOTEBOOK CHECKS
----------------------------------------------------------------------
✅ predictor_notebook: True
⚠️ fastapi_testclient: False

BACKEND v1.2 READY FOR DEPLOYMENT: False

Model SHA256:
1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61


In [155]:
# ============================================================
# CELDA 139 — REQUIREMENTS DEPLOY v1.2
# ============================================================

from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

DEPLOY_DIR_V12 = (
    ROOT
    / "deploy"
    / "v1.2.0-multilingual"
)

DEPLOY_DIR_V12.mkdir(
    parents=True,
    exist_ok=True
)

runtime_packages = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "numpy",
    "scipy",
    "scikit-learn",
    "joblib",
    "sentence-transformers",
    "transformers",
    "huggingface-hub",
    "tokenizers",
    "safetensors",
    "torch",
]

requirements = []

for package in runtime_packages:

    try:
        package_version = version(package)

        requirements.append(
            f"{package}=={package_version}"
        )

    except PackageNotFoundError:

        print(
            f"⚠️ No encontrado: {package}"
        )


REQUIREMENTS_PATH = (
    DEPLOY_DIR_V12
    / "requirements-v1.2.txt"
)

REQUIREMENTS_PATH.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)


print("=" * 70)
print("REQUIREMENTS v1.2")
print("=" * 70)

print("\nRuta:")
print(REQUIREMENTS_PATH)

print("\nContenido:\n")

print(
    REQUIREMENTS_PATH.read_text(
        encoding="utf-8"
    )
)

REQUIREMENTS v1.2

Ruta:
C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\requirements-v1.2.txt

Contenido:

fastapi==0.141.1
uvicorn==0.52.1
pydantic==2.13.4
numpy==1.26.4
scipy==1.17.1
scikit-learn==1.9.0
joblib==1.5.3
sentence-transformers==5.7.0
transformers==5.15.0
huggingface-hub==1.27.0
tokenizers==0.22.2
safetensors==0.8.0
torch==2.13.0



In [156]:
# ============================================================
# CELDA 140 — CONFIGURACIÓN DE ENTORNO
# ============================================================

ENV_EXAMPLE = """\
# ============================================================
# TechMind v1.2.0-multilingual
# ============================================================

# Modelo
TECHMIND_V12_MODEL_PATH=models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib

# Hugging Face / Transformers
HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1
HF_HUB_DISABLE_TELEMETRY=1
TOKENIZERS_PARALLELISM=false

# API
TECHMIND_HOST=0.0.0.0
TECHMIND_PORT=8000
"""

ENV_EXAMPLE_PATH = (
    DEPLOY_DIR_V12
    / ".env.example"
)

ENV_EXAMPLE_PATH.write_text(
    ENV_EXAMPLE,
    encoding="utf-8"
)

print(
    "✅ Creado:",
    ENV_EXAMPLE_PATH
)

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\.env.example


In [157]:
# ============================================================
# CELDA 141 — start_server.py
# ============================================================

START_SERVER_V12 = r'''\
from __future__ import annotations

import os

import uvicorn


def main() -> None:

    host = os.getenv(
        "TECHMIND_HOST",
        "0.0.0.0"
    )

    port = int(
        os.getenv(
            "TECHMIND_PORT",
            "8000"
        )
    )

    # Inferencia reproducible/offline.
    os.environ.setdefault(
        "HF_HUB_OFFLINE",
        "1"
    )

    os.environ.setdefault(
        "TRANSFORMERS_OFFLINE",
        "1"
    )

    os.environ.setdefault(
        "HF_HUB_DISABLE_TELEMETRY",
        "1"
    )

    os.environ.setdefault(
        "TOKENIZERS_PARALLELISM",
        "false"
    )

    uvicorn.run(
        "techmind_api_v12.main:app",
        host=host,
        port=port,
        workers=1,
        log_level="info",
    )


if __name__ == "__main__":
    main()
'''

START_SERVER_PATH = (
    DEPLOY_DIR_V12
    / "start_server.py"
)

START_SERVER_PATH.write_text(
    START_SERVER_V12,
    encoding="utf-8"
)

print(
    "✅ Creado:",
    START_SERVER_PATH
)

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\start_server.py


In [158]:
# ============================================================
# CELDA 142 — smoke_test_v12.py
# ============================================================

SMOKE_TEST_V12 = r'''\
from __future__ import annotations

import json
import os
import urllib.request


HOST = os.getenv(
    "TECHMIND_TEST_HOST",
    "127.0.0.1"
)

PORT = int(
    os.getenv(
        "TECHMIND_PORT",
        "8000"
    )
)

BASE_URL = (
    f"http://{HOST}:{PORT}"
)


def get_json(path: str):

    with urllib.request.urlopen(
        BASE_URL + path,
        timeout=15
    ) as response:

        return (
            response.status,
            json.loads(
                response
                .read()
                .decode("utf-8")
            )
        )


def post_json(
    path: str,
    payload: dict
):

    body = json.dumps(
        payload,
        ensure_ascii=False
    ).encode("utf-8")

    request = urllib.request.Request(
        BASE_URL + path,
        data=body,
        headers={
            "Content-Type":
                "application/json; charset=utf-8"
        },
        method="POST",
    )

    with urllib.request.urlopen(
        request,
        timeout=30
    ) as response:

        return (
            response.status,
            json.loads(
                response
                .read()
                .decode("utf-8")
            )
        )


def main() -> None:

    print("=" * 70)
    print("TECHMIND v1.2 — DEPLOYMENT SMOKE TEST")
    print("=" * 70)

    health_status, health = get_json(
        "/health"
    )

    print(
        "\n/health:",
        health_status,
        health
    )

    assert health_status == 200
    assert health["status"] == "ok"
    assert health["model_loaded"] is True

    info_status, info = get_json(
        "/model-info"
    )

    print(
        "\n/model-info:",
        info_status
    )

    print(
        "Version:",
        info["version"]
    )

    print(
        "SHA:",
        info["artifact_sha256"]
    )

    assert info_status == 200

    assert (
        info["version"]
        ==
        "1.2.0-multilingual"
    )

    predict_status, result = post_json(
        "/predict",
        {
            "texts": [
                (
                    "Train a classification model "
                    "and evaluate precision recall "
                    "and F1."
                ),
                (
                    "Preparar una pizza con "
                    "tomate y queso."
                )
            ],
            "top_k": 4
        }
    )

    print(
        "\n/predict:",
        predict_status
    )

    for item in result["predictions"]:

        print(
            item["prediction"],
            "→",
            item["decision"]
        )

    assert predict_status == 200

    assert (
        result["predictions"][0][
            "decision"
        ]
        != "rejected_ood"
    )

    assert (
        result["predictions"][1][
            "decision"
        ]
        == "rejected_ood"
    )

    print(
        "\n✅ DEPLOYMENT SMOKE TEST PASSED"
    )


if __name__ == "__main__":
    main()
'''

SMOKE_TEST_PATH = (
    DEPLOY_DIR_V12
    / "smoke_test_v12.py"
)

SMOKE_TEST_PATH.write_text(
    SMOKE_TEST_V12,
    encoding="utf-8"
)

print(
    "✅ Creado:",
    SMOKE_TEST_PATH
)

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\smoke_test_v12.py


In [159]:
# ============================================================
# CELDA 143 — README DEPLOYMENT v1.2
# VERSIÓN SIN STRINGS MULTILÍNEA
# ============================================================

from pathlib import Path


README_LINES = [

    "# TechMind v1.2.0-multilingual — Deployment",
    "",
    "## Estado",
    "",
    "**validated_experimental_candidate**",
    "",
    "Backend validado mediante:",
    "",
    "- importación en proceso independiente",
    "- carga independiente del predictor",
    "- inferencia independiente",
    "- FastAPI + Uvicorn",
    "- `/health`",
    "- `/model-info`",
    "- `/predict`",
    "- contrato OpenAPI",
    "- verificación SHA-256",
    "",
    "---",
    "",
    "## Arquitectura",
    "",
    "```text",
    "Input",
    "  |",
    "  v",
    "Input validation",
    "  |",
    "  v",
    "MiniLM multilingual",
    "  |",
    "  v",
    "Semantic domain support (5NN)",
    "  |",
    "  +-- similarity < 0.4266 --> rejected_ood",
    "  |",
    "  v",
    "TF-IDF Word+Char + MiniLM",
    "  |",
    "  v",
    "LinearSVC C=0.3",
    "  |",
    "  v",
    "Decision margin",
    "  |",
    "  +-- margin < 0.8132 --> review",
    "  |",
    "  v",
    "accepted",
    "```",
    "",
    "## Modelo",
    "",
    "- Version: `1.2.0-multilingual`",
    "- Status: `validated_experimental_candidate`",
    "- Encoder: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`",
    "- Embedding dimensions: `384`",
    "- Classifier: `LinearSVC`",
    "- C: `0.3`",
    "",
    "## Controles operacionales",
    "",
    "### Semantic domain support",
    "",
    "- Métrica: `mean_cosine_similarity_5nn`",
    "- Vecinos: `5`",
    "- Threshold: `0.4266`",
    "",
    "Si `similarity_5nn < 0.4266`, la entrada se marca como:",
    "",
    "`rejected_ood`",
    "",
    "### Decision margin",
    "",
    "- Métrica: `top1_minus_top2_decision_margin`",
    "- Threshold: `0.8132`",
    "",
    "Si `margin < 0.8132`, la predicción se envía a:",
    "",
    "`review`",
    "",
    "En otro caso:",
    "",
    "`accepted`",
    "",
    "---",
    "",
    "## Artefacto",
    "",
    "Ruta:",
    "",
    "```text",
    "models/experimental/v1.2.0-multilingual/",
    "techmind_hybrid_v1_2_0_multilingual.joblib",
    "```",
    "",
    "SHA-256:",
    "",
    "```text",
    "1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61",
    "```",
    "",
    "Este SHA identifica exactamente el artefacto validado.",
    "",
    "---",
    "",
    "## Configuración offline",
    "",
    "Variables recomendadas:",
    "",
    "```text",
    "HF_HUB_OFFLINE=1",
    "TRANSFORMERS_OFFLINE=1",
    "HF_HUB_DISABLE_TELEMETRY=1",
    "TOKENIZERS_PARALLELISM=false",
    "```",
    "",
    "El predictor utiliza `local_files_only=True` para cargar MiniLM.",
    "",
    "Por tanto, el encoder debe estar disponible previamente",
    "en la caché local del servidor.",
    "",
    "---",
    "",
    "## Instalación",
    "",
    "Desde la raíz del repositorio:",
    "",
    "```bash",
    "python -m pip install -r deploy/v1.2.0-multilingual/requirements-v1.2.txt",
    "```",
    "",
    "---",
    "",
    "## Iniciar API",
    "",
    "```bash",
    "python deploy/v1.2.0-multilingual/start_server.py",
    "```",
    "",
    "Configuración por defecto:",
    "",
    "```text",
    "Host: 0.0.0.0",
    "Port: 8000",
    "```",
    "",
    "Swagger:",
    "",
    "`http://localhost:8000/docs`",
    "",
    "---",
    "",
    "## Endpoints",
    "",
    "- `GET /`",
    "- `GET /health`",
    "- `GET /model-info`",
    "- `POST /predict`",
    "- `/docs`",
    "- `/redoc`",
    "- `/openapi.json`",
    "",
    "Estados operacionales posibles:",
    "",
    "- `accepted`",
    "- `review`",
    "- `rejected_ood`",
    "- `rejected_invalid`",
    "",
    "Los scores de `LinearSVC` NO son probabilidades.",
    "",
    "---",
    "",
    "## Smoke test",
    "",
    "Con la API ejecutándose:",
    "",
    "```bash",
    "python deploy/v1.2.0-multilingual/smoke_test_v12.py",
    "```",
    "",
    "Resultado esperado:",
    "",
    "```text",
    "DEPLOYMENT SMOKE TEST PASSED",
    "```",
    "",
    "---",
    "",
    "## Validación del backend",
    "",
    "```text",
    "artifact_integrity             PASS",
    "independent_import             PASS",
    "independent_predictor_load     PASS",
    "independent_inference          PASS",
    "uvicorn_startup_health         PASS",
    "http_model_info                PASS",
    "http_predict                   PASS",
    "openapi_contract               PASS",
    "```",
    "",
    "`BACKEND v1.2 READY FOR DEPLOYMENT: True`",
    "",
    "---",
    "",
    "## Benchmark final independiente",
    "",
    "### v1.2",
    "",
    "- Accuracy: `76.25%`",
    "- F1 Macro: `75.70%`",
    "- Cross-language consistency: `80.00%`",
    "",
    "### Comparación con v1.1",
    "",
    "| Métrica | v1.1 | v1.2 |",
    "|---|---:|---:|",
    "| Accuracy | 56.56% | 76.25% |",
    "| F1 Macro | 57.05% | 75.70% |",
    "",
    "Mejora absoluta:",
    "",
    "- Accuracy: `+19.69 pp`",
    "- F1 Macro: `+18.64 pp`",
    "",
    "La comparación pareada mediante McNemar mostró una",
    "diferencia estadísticamente significativa a favor de v1.2.",
    "",
    "---",
    "",
    "## Rendimiento por idioma",
    "",
    "| Idioma | Accuracy |",
    "|---|---:|",
    "| EN | 77.50% |",
    "| ES | 75.00% |",
    "| ES/EN | 78.75% |",
    "| RU | 73.75% |",
    "",
    "---",
    "",
    "## Rendimiento por categoría",
    "",
    "| Categoría | Accuracy |",
    "|---|---:|",
    "| Backend | 90.00% |",
    "| Cloud | 40.00% |",
    "| Data Science | 83.75% |",
    "| Frontend | 91.25% |",
    "",
    "---",
    "",
    "## Limitación conocida",
    "",
    "La principal limitación identificada es `Cloud`.",
    "",
    "Accuracy Cloud en el benchmark independiente: `40%`.",
    "",
    "La mayoría de los errores Cloud fueron clasificados como Backend.",
    "",
    "Esto sugiere una limitación de cobertura conceptual en la frontera:",
    "",
    "`Cloud <-> Backend`",
    "",
    "y no una degradación específica por idioma.",
    "",
    "El modelo debe permanecer congelado para esta versión.",
    "",
    "Las mejoras derivadas de este benchmark pertenecen a una versión",
    "futura y deberán validarse mediante un nuevo holdout independiente.",
    "",
    "---",
    "",
    "## Estado de versiones",
    "",
    "- `v1.1.0`: stable baseline / fallback",
    "- `v1.2.0-multilingual`: validated experimental candidate",
    "- Backend v1.2: ready for deployment",
]


README_DEPLOY = "\n".join(
    README_LINES
) + "\n"


README_DEPLOY_PATH = (
    DEPLOY_DIR_V12
    / "README.md"
)


README_DEPLOY_PATH.write_text(
    README_DEPLOY,
    encoding="utf-8"
)


print("=" * 70)
print("README DEPLOYMENT v1.2")
print("=" * 70)

print(
    "\n✅ Creado:",
    README_DEPLOY_PATH
)

print(
    "Tamaño:",
    README_DEPLOY_PATH.stat().st_size,
    "bytes"
)

print(
    "Líneas:",
    len(README_LINES)
)

README DEPLOYMENT v1.2

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\README.md
Tamaño: 4723 bytes
Líneas: 280


In [160]:
# ============================================================
# CELDA 144 — VALIDAR PAQUETE DEPLOYMENT
# ============================================================

print("=" * 70)
print("TECHMIND v1.2 — DEPLOYMENT PACKAGE")
print("=" * 70)


ARCHIVOS_ESPERADOS = [
    ".env.example",
    "README.md",
    "requirements-v1.2.txt",
    "smoke_test_v12.py",
    "start_server.py",
]


deployment_checks = {}


for nombre in ARCHIVOS_ESPERADOS:

    path = (
        DEPLOY_DIR_V12
        / nombre
    )

    existe = (
        path.exists()
        and
        path.is_file()
        and
        path.stat().st_size > 0
    )

    deployment_checks[
        nombre
    ] = existe

    print(
        f"{'✅' if existe else '❌'} "
        f"{nombre}"
    )


DEPLOYMENT_PACKAGE_OK = all(
    deployment_checks.values()
)


print("\n" + "=" * 70)

print(
    "DEPLOYMENT PACKAGE READY:",
    DEPLOYMENT_PACKAGE_OK
)

print("=" * 70)

TECHMIND v1.2 — DEPLOYMENT PACKAGE
✅ .env.example
✅ README.md
✅ requirements-v1.2.txt
✅ smoke_test_v12.py
✅ start_server.py

DEPLOYMENT PACKAGE READY: True


In [161]:
# ============================================================
# CELDA 145 — RUNTIME BUNDLE PARA OCI
# ============================================================

from pathlib import Path
import zipfile
import hashlib


BUNDLE_PATH = (
    ROOT
    / "deploy"
    / "techmind-v1.2-runtime.zip"
)


RUTAS_RUNTIME = [

    ROOT / "techmind_v12",

    ROOT / "techmind_api_v12",

    (
        ROOT
        / "models"
        / "experimental"
        / "v1.2.0-multilingual"
        / "techmind_hybrid_v1_2_0_multilingual.joblib"
    ),

    (
        ROOT
        / "deploy"
        / "v1.2.0-multilingual"
    ),
]


if BUNDLE_PATH.exists():
    BUNDLE_PATH.unlink()


with zipfile.ZipFile(
    BUNDLE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:

    for ruta in RUTAS_RUNTIME:

        if not ruta.exists():
            raise FileNotFoundError(
                f"No existe: {ruta}"
            )

        if ruta.is_file():

            zf.write(
                ruta,
                ruta.relative_to(ROOT)
            )

        else:

            for archivo in ruta.rglob("*"):

                if archivo.is_file():

                    # Evitar archivos temporales Python
                    if "__pycache__" in archivo.parts:
                        continue

                    if archivo.suffix == ".pyc":
                        continue

                    zf.write(
                        archivo,
                        archivo.relative_to(ROOT)
                    )


# ------------------------------------------------------------
# SHA DEL ZIP
# ------------------------------------------------------------

sha = hashlib.sha256()

with open(BUNDLE_PATH, "rb") as f:

    for bloque in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha.update(bloque)


BUNDLE_SHA256 = sha.hexdigest()


print("=" * 70)
print("TECHMIND v1.2 — OCI RUNTIME BUNDLE")
print("=" * 70)

print("\nArchivo:")
print(BUNDLE_PATH)

print(
    "\nTamaño MB:",
    round(
        BUNDLE_PATH.stat().st_size
        / 1024
        / 1024,
        2
    )
)

print("\nSHA256:")
print(BUNDLE_SHA256)

TECHMIND v1.2 — OCI RUNTIME BUNDLE

Archivo:
C:\Users\MAMÁ\Downloads\techmind-v2\deploy\techmind-v1.2-runtime.zip

Tamaño MB: 6.82

SHA256:
f1913958b4ab27552da67093c38f5e4e9a6fb6abecc302eca42d7e129723fd07


In [162]:
# ============================================================
# CELDA 146 — VALIDAR RUNTIME BUNDLE
# ============================================================

with zipfile.ZipFile(
    BUNDLE_PATH,
    "r"
) as zf:

    archivos_zip = set(
        zf.namelist()
    )


elementos_criticos = [

    "techmind_v12/__init__.py",

    "techmind_v12/predictor.py",

    "techmind_api_v12/__init__.py",

    "techmind_api_v12/main.py",

    "techmind_api_v12/schemas.py",

    (
        "models/experimental/"
        "v1.2.0-multilingual/"
        "techmind_hybrid_v1_2_0_multilingual.joblib"
    ),

    (
        "deploy/v1.2.0-multilingual/"
        "requirements-v1.2.txt"
    ),

    (
        "deploy/v1.2.0-multilingual/"
        "start_server.py"
    ),

    (
        "deploy/v1.2.0-multilingual/"
        "smoke_test_v12.py"
    ),
]


bundle_checks = {

    archivo:
        archivo in archivos_zip

    for archivo in elementos_criticos
}


print("=" * 70)
print("VALIDACIÓN OCI RUNTIME BUNDLE")
print("=" * 70)


for archivo, ok in bundle_checks.items():

    print(
        f"{'✅' if ok else '❌'} "
        f"{archivo}"
    )


OCI_BUNDLE_OK = all(
    bundle_checks.values()
)


print("\n" + "=" * 70)

print(
    "OCI BUNDLE READY:",
    OCI_BUNDLE_OK
)

print("=" * 70)

VALIDACIÓN OCI RUNTIME BUNDLE
✅ techmind_v12/__init__.py
✅ techmind_v12/predictor.py
✅ techmind_api_v12/__init__.py
✅ techmind_api_v12/main.py
✅ techmind_api_v12/schemas.py
✅ models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib
✅ deploy/v1.2.0-multilingual/requirements-v1.2.txt
✅ deploy/v1.2.0-multilingual/start_server.py
✅ deploy/v1.2.0-multilingual/smoke_test_v12.py

OCI BUNDLE READY: True


In [163]:
# ============================================================
# CELDA 147 — PRERREQUISITOS DOCKER
# ============================================================

from pathlib import Path


print("=" * 70)
print("TECHMIND v1.2 — PRERREQUISITOS DOCKER")
print("=" * 70)


# ------------------------------------------------------------
# ROOT
# ------------------------------------------------------------

if "ROOT" not in globals():

    ROOT = Path.cwd()

    # Si el notebook está dentro de notebooks/,
    # intentar detectar la raíz.
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent


print("\nROOT:")
print(ROOT)


# ------------------------------------------------------------
# ARCHIVOS NECESARIOS
# ------------------------------------------------------------

PREREQUISITOS_DOCKER = {

    "predictor_v12":
        ROOT
        / "techmind_v12"
        / "predictor.py",

    "api_v12":
        ROOT
        / "techmind_api_v12"
        / "main.py",

    "schemas_v12":
        ROOT
        / "techmind_api_v12"
        / "schemas.py",

    "modelo_v12":
        ROOT
        / "models"
        / "experimental"
        / "v1.2.0-multilingual"
        / "techmind_hybrid_v1_2_0_multilingual.joblib",

    "requirements":
        ROOT
        / "deploy"
        / "v1.2.0-multilingual"
        / "requirements-v1.2.txt",

    "start_server":
        ROOT
        / "deploy"
        / "v1.2.0-multilingual"
        / "start_server.py",

    "smoke_test":
        ROOT
        / "deploy"
        / "v1.2.0-multilingual"
        / "smoke_test_v12.py",
}


checks = {}


print("\nArchivos:")

for nombre, ruta in PREREQUISITOS_DOCKER.items():

    ok = (
        ruta.exists()
        and ruta.is_file()
    )

    checks[nombre] = ok

    print(
        f"{'✅' if ok else '❌'} "
        f"{nombre}: {ruta}"
    )


DOCKER_PREREQUISITES_OK = all(
    checks.values()
)


print("\n" + "=" * 70)

print(
    "DOCKER PREREQUISITES READY:",
    DOCKER_PREREQUISITES_OK
)

print("=" * 70)

TECHMIND v1.2 — PRERREQUISITOS DOCKER

ROOT:
C:\Users\MAMÁ\Downloads\techmind-v2

Archivos:
✅ predictor_v12: C:\Users\MAMÁ\Downloads\techmind-v2\techmind_v12\predictor.py
✅ api_v12: C:\Users\MAMÁ\Downloads\techmind-v2\techmind_api_v12\main.py
✅ schemas_v12: C:\Users\MAMÁ\Downloads\techmind-v2\techmind_api_v12\schemas.py
✅ modelo_v12: C:\Users\MAMÁ\Downloads\techmind-v2\models\experimental\v1.2.0-multilingual\techmind_hybrid_v1_2_0_multilingual.joblib
✅ requirements: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\requirements-v1.2.txt
✅ start_server: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\start_server.py
✅ smoke_test: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\smoke_test_v12.py

DOCKER PREREQUISITES READY: True


In [164]:
# ============================================================
# CELDA 148 — CREAR ESTRUCTURA DOCKER v1.2
# ============================================================

from pathlib import Path


DOCKER_DIR_V12 = (
    ROOT
    / "deploy"
    / "v1.2.0-multilingual"
    / "docker"
)

DOCKER_DIR_V12.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("TECHMIND v1.2 — DOCKER STRUCTURE")
print("=" * 70)

print("\nDirectorio:")
print(DOCKER_DIR_V12)

print(
    "\nExiste:",
    DOCKER_DIR_V12.exists()
)

TECHMIND v1.2 — DOCKER STRUCTURE

Directorio:
C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\docker

Existe: True


In [165]:
# ============================================================
# CELDA 149 — DOCKERFILE TECHMIND v1.2
# ============================================================

DOCKERFILE_LINES = [

    "# syntax=docker/dockerfile:1",

    "",

    "FROM python:3.11-slim-bookworm",

    "",

    "# --------------------------------------------------------",
    "# METADATA",
    "# --------------------------------------------------------",

    'LABEL org.opencontainers.image.title="TechMind"',
    'LABEL org.opencontainers.image.version="1.2.0-multilingual"',
    'LABEL techmind.model.status="validated_experimental_candidate"',
    'LABEL techmind.model.sha256="1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61"',

    "",

    "# --------------------------------------------------------",
    "# PYTHON",
    "# --------------------------------------------------------",

    "ENV PYTHONDONTWRITEBYTECODE=1",
    "ENV PYTHONUNBUFFERED=1",
    "ENV PIP_NO_CACHE_DIR=1",
    "ENV PIP_DISABLE_PIP_VERSION_CHECK=1",

    "",

    "WORKDIR /app",

    "",

    "# --------------------------------------------------------",
    "# SYSTEM DEPENDENCIES",
    "# --------------------------------------------------------",

    "RUN apt-get update \\",
    "    && apt-get install -y --no-install-recommends \\",
    "       ca-certificates \\",
    "       libgomp1 \\",
    "    && rm -rf /var/lib/apt/lists/*",

    "",

    "# --------------------------------------------------------",
    "# PYTHON DEPENDENCIES",
    "# --------------------------------------------------------",

    "COPY deploy/v1.2.0-multilingual/requirements-v1.2.txt \\",
    "     /tmp/requirements.txt",

    "",

    "RUN python -m pip install --upgrade pip setuptools wheel \\",
    "    && python -m pip install -r /tmp/requirements.txt \\",
    "    && python -m pip check",

    "",

    "# --------------------------------------------------------",
    "# APPLICATION",
    "# --------------------------------------------------------",

    "COPY techmind_v12 /app/techmind_v12",
    "COPY techmind_api_v12 /app/techmind_api_v12",

    "",

    "# --------------------------------------------------------",
    "# FROZEN MODEL ARTIFACT",
    "# --------------------------------------------------------",

    "COPY models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib \\",
    "     /app/models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib",

    "",

    "# Verify exact validated artifact.",

    'RUN echo "1a495520f642416e7dd391f97417cd3d12dcd82ab11636b7f190e5ed6dafea61  /app/models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib" \\',
    "    | sha256sum -c -",

    "",

    "# --------------------------------------------------------",
    "# NON-ROOT USER",
    "# --------------------------------------------------------",

    "RUN groupadd --gid 10001 techmind \\",
    "    && useradd \\",
    "       --uid 10001 \\",
    "       --gid 10001 \\",
    "       --create-home \\",
    "       --shell /usr/sbin/nologin \\",
    "       techmind \\",
    "    && mkdir -p /home/techmind/.cache/huggingface \\",
    "    && chown -R techmind:techmind /home/techmind /app",

    "",

    "ENV HF_HOME=/home/techmind/.cache/huggingface",

    "",

    "USER techmind",

    "",

    "# --------------------------------------------------------",
    "# DOWNLOAD AND BAKE MINILM INTO IMAGE",
    "# --------------------------------------------------------",

    "RUN python -c \"from sentence_transformers import SentenceTransformer; m=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'); assert m.get_sentence_embedding_dimension()==384; print('MINILM_BUILD_OK')\"",

    "",

    "# --------------------------------------------------------",
    "# OFFLINE RUNTIME",
    "# --------------------------------------------------------",

    "ENV HF_HUB_OFFLINE=1",
    "ENV TRANSFORMERS_OFFLINE=1",
    "ENV HF_HUB_DISABLE_TELEMETRY=1",
    "ENV TOKENIZERS_PARALLELISM=false",

    "",

    "ENV TECHMIND_V12_MODEL_PATH=/app/models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib",

    "",

    "# --------------------------------------------------------",
    "# API",
    "# --------------------------------------------------------",

    "EXPOSE 8000",

    "",

    "HEALTHCHECK --interval=30s --timeout=5s --start-period=40s --retries=3 \\",
    "    CMD python -c \"import json,urllib.request; d=json.load(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=3)); assert d['status']=='ok' and d['model_loaded'] is True\" || exit 1",

    "",

    "# --------------------------------------------------------",
    "# UVICORN",
    "# --------------------------------------------------------",

    'CMD ["python", "-m", "uvicorn", "techmind_api_v12.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]',
]


DOCKERFILE_CONTENT = (
    "\n".join(
        DOCKERFILE_LINES
    )
    + "\n"
)


DOCKERFILE_PATH = (
    DOCKER_DIR_V12
    / "Dockerfile"
)


DOCKERFILE_PATH.write_text(
    DOCKERFILE_CONTENT,
    encoding="utf-8"
)


print("=" * 70)
print("DOCKERFILE TECHMIND v1.2")
print("=" * 70)

print(
    "\n✅ Creado:",
    DOCKERFILE_PATH
)

print(
    "Tamaño:",
    DOCKERFILE_PATH.stat().st_size,
    "bytes"
)

DOCKERFILE TECHMIND v1.2

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\docker\Dockerfile
Tamaño: 4113 bytes


In [166]:
# ============================================================
# CELDA 150 — DOCKERIGNORE
# ============================================================

DOCKERIGNORE_LINES = [

    ".git",
    ".github",

    "",
    ".venv",
    "venv",
    "env",

    "",
    "**/__pycache__",
    "**/*.pyc",
    "**/*.pyo",

    "",
    "*.ipynb",
    ".ipynb_checkpoints",

    "",
    "data",
    "data/**",

    "",
    "notebooks",
    "notebooks/**",

    "",
    "experiments",
    "experiments/**",

    "",
    "reports",
    "reports/**",

    "",
    "*.log",

    "",
    ".pytest_cache",
    ".mypy_cache",

    "",
    "dist",
    "build",

    "",
    "# v1.1 remains in repository but is not needed",
    "# inside the v1.2 container.",
    "models/v1.1.0",

    "",
    ".vscode",
    ".idea",

    "",
    "Thumbs.db",
    ".DS_Store",
]


DOCKERIGNORE_PATH = (
    DOCKER_DIR_V12
    / "Dockerfile.dockerignore"
)


DOCKERIGNORE_PATH.write_text(
    "\n".join(
        DOCKERIGNORE_LINES
    )
    + "\n",
    encoding="utf-8"
)


print(
    "✅ Creado:",
    DOCKERIGNORE_PATH
)

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\docker\Dockerfile.dockerignore


In [167]:
# ============================================================
# CELDA 151 — COMPOSE.YAML
# ============================================================

COMPOSE_LINES = [

    "services:",

    "  techmind:",

    "    build:",
    "      context: ../../..",
    "      dockerfile: deploy/v1.2.0-multilingual/docker/Dockerfile",

    "",
    "    image: techmind:v1.2.0-multilingual",

    "",
    "    container_name: techmind-v12",

    "",
    "    ports:",
    '      - "${TECHMIND_PORT:-8000}:8000"',

    "",
    "    environment:",

    "      TECHMIND_V12_MODEL_PATH: /app/models/experimental/v1.2.0-multilingual/techmind_hybrid_v1_2_0_multilingual.joblib",

    '      HF_HUB_OFFLINE: "1"',
    '      TRANSFORMERS_OFFLINE: "1"',
    '      HF_HUB_DISABLE_TELEMETRY: "1"',
    '      TOKENIZERS_PARALLELISM: "false"',

    "",
    "    restart: unless-stopped",

    "",
    "    init: true",
]


COMPOSE_PATH = (
    DOCKER_DIR_V12
    / "compose.yaml"
)


COMPOSE_PATH.write_text(
    "\n".join(
        COMPOSE_LINES
    )
    + "\n",
    encoding="utf-8"
)


print(
    "✅ Creado:",
    COMPOSE_PATH
)

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\docker\compose.yaml


In [168]:
# ============================================================
# CELDA 152 — SMOKE TEST DOCKER
# ============================================================

SMOKE_DOCKER_LINES = [

    "from __future__ import annotations",

    "",
    "import json",
    "import os",
    "import urllib.request",

    "",

    "HOST = os.getenv(",
    "    'TECHMIND_TEST_HOST',",
    "    '127.0.0.1'",
    ")",

    "",

    "PORT = int(",
    "    os.getenv(",
    "        'TECHMIND_PORT',",
    "        '8000'",
    "    )",
    ")",

    "",

    "BASE_URL = f'http://{HOST}:{PORT}'",

    "",

    "EXPECTED_SHA = (",
    "    '5f6ff8c4b350a9cbe8a9e7d531290bf'",
    "    'da41a83d5337ae4174357ea170ea9dce3'",
    ")",

    "",

    "",
    "def get_json(path: str):",

    "    with urllib.request.urlopen(",
    "        BASE_URL + path,",
    "        timeout=30",
    "    ) as response:",

    "        return (",
    "            response.status,",
    "            json.loads(",
    "                response.read().decode('utf-8')",
    "            )",
    "        )",

    "",

    "",
    "def post_json(path: str, payload: dict):",

    "    body = json.dumps(",
    "        payload,",
    "        ensure_ascii=False",
    "    ).encode('utf-8')",

    "",

    "    request = urllib.request.Request(",
    "        BASE_URL + path,",
    "        data=body,",
    "        headers={",
    "            'Content-Type':",
    "                'application/json; charset=utf-8'",
    "        },",
    "        method='POST'",
    "    )",

    "",

    "    with urllib.request.urlopen(",
    "        request,",
    "        timeout=60",
    "    ) as response:",

    "        return (",
    "            response.status,",
    "            json.loads(",
    "                response.read().decode('utf-8')",
    "            )",
    "        )",

    "",

    "",
    "def main():",

    "    print('=' * 70)",
    "    print('TECHMIND v1.2 — DOCKER SMOKE TEST')",
    "    print('=' * 70)",

    "",

    "    status, health = get_json('/health')",

    "    assert status == 200",
    "    assert health['status'] == 'ok'",
    "    assert health['model_loaded'] is True",

    "    print('✅ /health')",

    "",

    "    status, info = get_json('/model-info')",

    "    assert status == 200",

    "    assert info['version'] == '1.2.0-multilingual'",

    "    assert info['artifact_sha256'] == EXPECTED_SHA",

    "    print('✅ /model-info')",
    "    print('✅ SHA256')",

    "",

    "    payload = {",

    "        'texts': [",

    "            (",
    "                'Train a classification model '",
    "                'and evaluate precision recall and F1.'",
    "            ),",

    "            (",
    "                'Crear un endpoint REST con autenticación '",
    "                'y acceso a base de datos.'",
    "            ),",

    "            (",
    "                'Обучить модель классификации и оценить '",
    "                'precision recall и F1.'",
    "            ),",

    "            (",
    "                'Preparar una pizza con tomate y queso.'",
    "            )",

    "        ],",

    "        'top_k': 4,",
    "        'include_explanation': False",
    "    }",

    "",

    "    status, result = post_json(",
    "        '/predict',",
    "        payload",
    "    )",

    "",

    "    assert status == 200",

    "    predictions = result['predictions']",

    "    assert len(predictions) == 4",

    "",

    "    assert predictions[0]['prediction'] == 'datascience'",
    "    assert predictions[0]['decision'] != 'rejected_ood'",

    "",

    "    assert predictions[1]['prediction'] == 'backend'",
    "    assert predictions[1]['decision'] != 'rejected_ood'",

    "",

    "    assert predictions[2]['prediction'] == 'datascience'",
    "    assert predictions[2]['decision'] != 'rejected_ood'",

    "",

    "    assert predictions[3]['decision'] == 'rejected_ood'",

    "",

    "    print('✅ /predict')",
    "    print('✅ English')",
    "    print('✅ Español')",
    "    print('✅ Русский')",
    "    print('✅ OOD')",

    "",

    "    print('\\n' + '=' * 70)",
    "    print('DOCKER SMOKE TEST PASSED')",
    "    print('=' * 70)",

    "",

    "",
    "if __name__ == '__main__':",
    "    main()",
]


SMOKE_DOCKER_PATH = (
    DOCKER_DIR_V12
    / "smoke_test_docker.py"
)


SMOKE_DOCKER_PATH.write_text(
    "\n".join(
        SMOKE_DOCKER_LINES
    )
    + "\n",
    encoding="utf-8"
)


print(
    "✅ Creado:",
    SMOKE_DOCKER_PATH
)

✅ Creado: C:\Users\MAMÁ\Downloads\techmind-v2\deploy\v1.2.0-multilingual\docker\smoke_test_docker.py


In [169]:
# ============================================================
# CELDA 153 — VALIDAR DOCKER PACKAGE
# ============================================================

print("=" * 70)
print("TECHMIND v1.2 — DOCKER PACKAGE")
print("=" * 70)


DOCKER_FILES = [

    "Dockerfile",
    "Dockerfile.dockerignore",
    "compose.yaml",
    "smoke_test_docker.py",

]


DOCKER_CHECKS = {}


for filename in DOCKER_FILES:

    path = (
        DOCKER_DIR_V12
        / filename
    )

    ok = (
        path.exists()
        and
        path.is_file()
        and
        path.stat().st_size > 0
    )

    DOCKER_CHECKS[
        filename
    ] = ok

    print(
        f"{'✅' if ok else '❌'} "
        f"{filename}"
    )


DOCKER_PACKAGE_READY = all(
    DOCKER_CHECKS.values()
)


print("\n" + "=" * 70)

print(
    "DOCKER PACKAGE READY:",
    DOCKER_PACKAGE_READY
)

print("=" * 70)

TECHMIND v1.2 — DOCKER PACKAGE
✅ Dockerfile
✅ Dockerfile.dockerignore
✅ compose.yaml
✅ smoke_test_docker.py

DOCKER PACKAGE READY: True
